# 03 - Hybrid QCNN, multiclass detection

## Purpose

**Question:** how stable are the results across multiple random seeds (mean, standard
deviation and confidence intervals), rather than a single run?


The paper's architecture is unchanged: 8-qubit, 4-layer circuit with angle-encoded data
re-uploading at every layer and the three-scale entangler (ring CNOT, skip-one CZ, cross-half
CNOT), embedded as a 96-parameter bottleneck between a classical encoder and decoder.

What is added is **1multi-seed training** with mean +/- std and confidence intervals, and the
complete set of curves: learning curves with a seed band, ROC, precision-recall, calibration,
confusion matrix and per-class bars. Every array behind every figure is saved.

Checkpoint selection is on the **validation** split only. Test data is touched once, after
training.

> Training runs; use `QUICK_MODE=1` for a fast pass, or run on Grid5000 for the full
> multiclass schedule.


---
### Persistence contract

Every result below is written to `outputs/` in five layers -- **metrics**, per-sample
**predictions**, **curves** (every value behind every plotted line, as CSV *and* NPZ),
**model** weights with fitted preprocessors, and **context** (config, seed, versions,
timings, OAR job id).

So a new table, a new curve, an extra statistic or a different figure style can all be
produced later **without re-running any of this code**. The final cell audits completeness;
run it before releasing the Grid5000 allocation.

In [2]:
from __future__ import annotations
# ==================================================================
# STANDALONE BOOTSTRAP -- every src/ module inlined.
# This notebook has NO src/ dependency: drop it on Grid5000 and run it.
# Edit the originals in src/, then re-run build_notebooks.py
# ==================================================================

import os, sys, time, json, platform, random, warnings
warnings.filterwarnings('ignore')
import matplotlib
if not os.environ.get('DISPLAY') and not os.environ.get('JPY_PARENT_PID'):
    matplotlib.use('Agg')
    print('[setup] no display -> Agg backend (figures saved, not shown)')
import matplotlib.pyplot as plt
import numpy as np

# ==================================================================
# === Inlined from src/config.py ===
# ==================================================================
"""
config.py -- single source of truth for the experimental configuration.

Standalone notebooks, QUICK_MODE, environment overrides and comprehensive
persistence are all driven from this file.

IMPORTANT -- experimental design
--------------------------------
The experimental design is the one in the paper (79.19% multiclass accuracy,
8-qubit circuit, SHA-256 audit log):

    classes             8 (7 attacks + benign)
    PCA components      16
    quantum layers      4
    sample budget       50,000
    baselines           RF/GB/SVM/KNN/MLP
    tasks               binary + multiclass

Every constant below reproduces the paper, verified bit-exact against the
archived Grid5000 weights.

Environment overrides
---------------------
QUICK_MODE=1    switch every slow knob to its iteration-friendly value
DATA_DIR=/p     override the data directory
OUT_DIR=/p      override the outputs directory
"""

import os
from pathlib import Path

# =============================================================================
# Paths
# =============================================================================
# ROOT defaults to the package root, NOT the cwd, so data/ and outputs/ resolve
# identically whether the code is launched from the package root, from
# notebooks/, or by an OAR job in $HOME.
#
# `__file__` is undefined once this module is INLINED into a standalone
# notebook, which is the normal way these run on Grid5000 -- so fall back to
# walking up from the working directory.
def _find_pkg_root() -> Path:
    try:
        return Path(__file__).resolve().parent.parent      # running from src/
    except NameError:
        pass
    here = Path.cwd().resolve()
    for cand in (here, *here.parents):
        if (cand / "src").is_dir() or (cand / "notebooks").is_dir():
            return cand
    return here


_PKG_ROOT = _find_pkg_root()
ROOT      = Path(os.environ.get("ROOT", _PKG_ROOT))
DATA_DIR  = Path(os.environ.get("DATA_DIR", ROOT / "data"))
OUT_DIR   = Path(os.environ.get("OUT_DIR",  ROOT / "outputs"))
LOG_DIR   = ROOT / "logs"
PAPER_DIR = OUT_DIR / "paper"

# Archived Grid5000 artefacts (trained weights + fitted preprocessing).
# These let every INFERENCE-ONLY notebook run with no retraining at all.
#
# Resolution order matters. The BUNDLED copy must win, so that a deployed
# package is genuinely self-contained: if the sibling development folder were
# preferred, a deploy folder would silently read the archive from outside
# itself and would then break the moment it is copied to the cluster.
def _find_archive() -> Path:
    env = os.environ.get("ARCHIVE_DIR")
    if env:
        return Path(env)
    candidates = [
        ROOT / "salvaged_g5k_artifacts",                     # bundled (deploy)
        ROOT.parent / "code_archive" / "salvaged_g5k_artifacts",
        ROOT.parent / "salvaged_g5k_artifacts",
    ]
    for c in candidates:
        if (c / "prepared_data.npz").exists():
            return c
    return candidates[0]


ARCHIVE_DIR = _find_archive()

# =============================================================================
# Run mode
# =============================================================================
QUICK_MODE = os.environ.get("QUICK_MODE", "0") == "1"

# =============================================================================
# Seeds -- multiple random seeds, reporting mean performance together with
# standard deviations or confidence intervals.
# Seed 42 first: it is the seed of the paper's original runs, so its row stays
# directly comparable with the published tables.
# =============================================================================
SEEDS_FULL  = [42, 1, 7, 13, 2024]
SEEDS_QUICK = [42, 1, 7]
SEEDS = SEEDS_QUICK if QUICK_MODE else SEEDS_FULL

# Expensive sweeps use a reduced seed set.
SEEDS_SWEEP = SEEDS[:3]

# =============================================================================
# Dataset -- CICIoMT2024 WiFi/MQTT partition
# =============================================================================
DATASET      = "CICIoMT2024-WiFi-MQTT"
N_CLASSES    = 8
CLASS_NAMES  = ["Benign", "ARP_Spoofing", "Reconnaissance", "MQTT_Malformed",
                "MQTT_DoS", "MQTT_DDoS", "DoS", "DDoS"]

POOL_SIZE    = 50_000 if not QUICK_MODE else 10_000
TRAIN_FRAC   = 0.70
VAL_FRAC     = 0.15
N_FEATURES   = 25
N_PCA        = 16

# --- enlarged test set -------------------------------------------------------
# The paper's per-class table reports F1 on n=4 (MQTT_Malformed) and n=15
# (ARP_Spoofing). Those estimates are not defensible. Draw a DISJOINT enlarged
# test set so every class has at least MIN_TEST_PER_CLASS samples. Inference is
# cheap, so this costs minutes and no retraining.
ENLARGED_TEST       = True
MIN_TEST_PER_CLASS  = 100
TEST_SIZE_TARGET    = 20_000

# --- evaluation protocols: random / session / device / chronological splits --
SPLIT_PROTOCOLS = ["random", "session", "device", "chrono"]
SPLIT_DEFAULT   = "random"   # headline; the others reported beside it

# =============================================================================
# Architecture -- section 3.3 of the paper
# =============================================================================
N_QUBITS  = 8
N_LAYERS  = 4
ENTANGLE_SCALES = (1, 2, 4)     # ring CNOT / skip-one CZ / cross-half CNOT
USE_REUPLOADING = True

# Encoding convention: R_Y(z_j), NO pi factor. Matches eq. (3) of the paper and
# the config that trained the archived weights. A pi-scaled variant exists in
# smote_correction_optimized_V2 and is WRONG: loading the archived weights into
# it yields 2.9% binary accuracy against the published 99.63%.
ANGLE_SCALE = 1.0

# =============================================================================
# Training -- Table 4 of the paper (verified against the archived Config)
# =============================================================================
EPOCHS_BINARY     = 50  if not QUICK_MODE else 3
EPOCHS_MULTICLASS = 120 if not QUICK_MODE else 5
LR_BINARY         = 5e-3
LR_MULTICLASS     = 2e-3
LR_MIN_BINARY     = 5e-5
LR_MIN_MULTICLASS = 2e-5
BATCH_SIZE        = 32          # Table 4 of the paper
WEIGHT_DECAY      = 1e-3
GRAD_CLIP         = 1.0
CLASS_WEIGHTING   = "sqrt_inverse"

EARLY_STOP_PATIENCE   = 15
EARLY_STOP_MIN_EPOCHS = 30

# =============================================================================
# Quantum backend -- MEASURED, not assumed (bench_backend.py, task P0.7)
#   default.qubit   + backprop    3.34 ms/sample  ->   3.9 h per multiclass run
#   lightning.qubit + adjoint    27.63 ms/sample  ->  32.2 h per multiclass run
# At 8 qubits the state vector is 256 amplitudes, so per-execution overhead
# dominates; default.qubit keeps the whole batch in one broadcast torch tensor.
# This is also what the paper reports in section 4.1.
# =============================================================================
QUANTUM_DEVICE = "default.qubit"
DIFF_METHOD    = "backprop"

# --- shots, circuit evaluation, ideal simulation -----------------------------
SHOT_SWEEP        = [128, 512, 1024, 4096, None]   # None == analytic/ideal
NOISE_DEPOL       = [0.0, 0.001, 0.005, 0.01, 0.02]
NOISE_AMP_DAMP    = [0.001, 0.005, 0.01]
SHOTS_TRAINING_SUPPORTED = False   # parameter-shift + broadcasting is unusable

# --- qubit / depth sweep (motivates the 8-qubit choice) ---------------------
# 7 configs, not a full grid. Sweeps use a reduced, IDENTICAL budget
# for every config so the comparison is fair across configurations.
SWEEP_QUBITS  = [4, 6, 8, 10, 12]     # at L = N_LAYERS
SWEEP_LAYERS  = [2, 4, 6]             # at n_q = N_QUBITS
SWEEP_EPOCHS  = 40
SWEEP_SAMPLES = 10_000

# =============================================================================
# Baselines -- Table 6/7 grids of the paper, plus matched controls
# =============================================================================
BASELINES = ["Random Forest", "Gradient Boosting", "MLP", "KNN", "SVM (RBF)"]

# Family A: identical encoder/decoder scaffolding, bottleneck swapped. This is
# what isolates the quantum contribution (comparison against parameter-matched
# compact classical neural networks).
MATCHED_BOTTLENECKS = ["pqc", "linear", "fourier", "mlp96", "identity"]
# Family B: highly optimised lightweight classical deployables.
LIGHTWEIGHT_BASELINES = ["int8_mlp", "pruned_mlp", "gbdt_shallow", "rf20"]

# --- ablation variants -------------------------------------------------------
# Every variant shares the IDENTICAL encoder and decoder; only the bottleneck
# changes. Enforced by assert_scaffolding_identical() before any training.
ABLATION_VARIANTS = ["full", "no_reupload", "no_entangle",
                     "ring_only", "ring_skip", "ring_cross",
                     "classical_matched", "simple_mlp"]

# =============================================================================
# Quantum and related-work comparators
#   Quantum baselines, and a comparison against a related method:
#   Al-Hawawreh & Hossain, Information Fusion 99:101889, 2023
# =============================================================================
QUANTUM_BASELINES = ["qsvm_angle", "qsvm_iqp", "vqc", "amp_qnn", "alhawawreh_qdl"]

# QSVM: the fidelity Gram matrix is O(N^2) in memory, so the training set is
# subsampled (stratified, seeded). The test set is NOT subsampled, so the row
# stays directly comparable with every other model in the paper. Both numbers
# are recorded in the result JSON and must be stated alongside the results.
QSVM_TRAIN_SUBSAMPLE = 3_000 if not QUICK_MODE else 500
QSVM_C_GRID          = [0.5, 1.0, 4.0, 16.0]
QSVM_EMBEDDINGS      = ["angle", "iqp"]

# VQC / amplitude-QNN: no classical decoder, so the head is the measurement
# itself. Depth matches the headline circuit for a like-for-like comparison.
VQC_LAYERS      = N_LAYERS
AMP_QNN_QUBITS  = 4            # 2^4 = 16 amplitudes = the PCA dimension
AMP_QNN_LAYERS  = N_LAYERS

# --- Al-Hawawreh & Hossain (2023) --------------------------------------------
# The source paper fuses heterogeneous sensor and network modalities before a
# quantum deep-learning classifier. CICIoMT2024 WiFi/MQTT carries no biometric
# channel, so the re-implementation fuses the three FEATURE GROUPS the flow
# features fall into. Every such decision is a documented deviation, written
# into the result JSON, and the row must be presented as a faithful
# re-implementation under stated assumptions -- never as the authors' own
# reported result.
# A DISJOINT modality partition of the 25 flow features -- every feature lands
# in exactly one modality, which is what a fusion architecture requires. The
# overlapping keyword sets in perturb.column_groups() are deliberately
# overlapping (a feature can be perturbed by several mechanisms) and are the
# wrong thing to fuse over; this partition is defined once, here, and asserted
# to be a partition in qbaselines.group_feature_matrix().
ALHAWAWREH_MODALITIES = {
    "timing":   ["duration", "rate", "iat", "interval"],
    "flags":    ["flag", "fin", "syn", "rst", "psh", "ack", "urg", "ece", "cwr"],
    "protocol": ["http", "https", "dns", "tcp", "udp", "arp", "icmp", "proto"],
    "size":     ["header", "length", "size", "tot", "sum", "min", "max",
                 "avg", "std", "variance", "number", "count"],
}
ALHAWAWREH_GROUPS      = list(ALHAWAWREH_MODALITIES)
ALHAWAWREH_GROUP_DIM   = 8     # per-group dense encoder width
ALHAWAWREH_FUSION_DIM  = 16    # fused representation before the quantum layer

# Reported numbers from the published papers are TRANSCRIBED, never inferred.
# `reported` stays None until a value is copied from the cited article; NB14
# prints the outstanding rows rather than emitting an empty table cell, so an
# untranscribed number cannot silently reach a results table.
LITERATURE_ROWS = [
    {"ref": "Al-Hawawreh & Hossain (2023), Information Fusion 99:101889",
     "method": "Data fusion + quantum deep learning",
     "dataset": "(as reported by the authors)", "reported": None},
]

# =============================================================================
# Multiclass gap -- multiclass accuracy is lower than binary (79.19%)
# -----------------------------------------------------------------------------
# 97.0% of the paper model's multiclass error is a single confusion:
# 1,163 DoS -> DDoS plus 351 DDoS -> DoS out of 1,561 errors. Merging the two
# volumetric classes lifts accuracy to 99.37% with no retraining, which places
# the gap in one binary decision rather than in the 8-class problem.
# =============================================================================
VOLUMETRIC_CLASSES = (6, 7)          # DoS, DDoS in CLASS_NAMES order
COARSE_NAMES = ["Benign", "ARP_Spoofing", "Reconnaissance", "MQTT_Malformed",
                "MQTT_DoS", "MQTT_DDoS", "Volumetric_Flood"]
GAP_APPROACHES = ["hierarchical", "feature_path"]
GAP_SEEDS      = SEEDS_SWEEP          # exploratory; 3 seeds
# Raw features routed around PCA straight into the decoder (feature_path).
FEATURE_PATH_K = 4                    # selected by mutual information, on TRAIN only
# Classical specialists used to measure the REPRESENTATION ceiling: if a tuned
# random forest on the raw 25 features cannot separate DoS from DDoS either,
# the gap is a property of the feature set, not of the quantum model.
GAP_PROBES = ["Random Forest", "Gradient Boosting", "MLP"]

# =============================================================================
# Robustness -- adversarial attacks and realistic perturbations.
# Applied in the ORIGINAL 25-feature space, BEFORE scaling and PCA.
# =============================================================================
PERTURBATIONS = ["missing", "jitter", "packet_loss", "protocol_var"]
PERTURB_SEVERITIES = [0.0, 0.01, 0.05, 0.10, 0.20]
ADV_ATTACKS   = ["FGSM", "PGD-10", "transfer", "constrained"]
ADV_EPSILONS  = [0.0, 0.01, 0.02, 0.05, 0.10]
# The PCA-space Gaussian sweep of the paper is kept only as a labelled baseline.
PCA_GAUSS_SIGMAS = [0.0, 0.01, 0.02, 0.05, 0.1, 0.2]

# =============================================================================
# Explainability -- benefit of combining SHAP and LIME; heuristic inverse
# mapping; explanation stability and fidelity
# =============================================================================
XAI_CONFIG = {
    "subset": 300, "nsamples": 100, "bg_size": 200,
    "deletion_samples": 30, "lime_samples": 20,
    "stability_repeats": 10,      # LIME is stochastic -> variance is the point
    "perturb_eps": 0.01,          # local Lipschitz probe
}

# =============================================================================
# Audit log -- experimental demonstration of its added value
# =============================================================================
AUDIT_CHAIN_LENGTHS = [1_000, 10_000, 100_000]
AUDIT_TRIALS        = {1_000: 100, 10_000: 50, 100_000: 10}
AUDIT_ATTACKS       = ["modify", "delete", "reorder", "truncate"]
AUDIT_FLOW_RATES    = [10, 100, 1_000, 10_000]   # flows/s, storage projection

# =============================================================================
# Cost / energy -- operational cost, energy and circuit evaluations
# =============================================================================
LATENCY_REPEATS  = 200
COST_BATCHES     = [1, 8, 32, 128]
KWOLLECT_ENABLED = os.environ.get("KWOLLECT", "0") == "1"   # Grid5000 wattmeter

# =============================================================================
# Optional second dataset (generalisation)
# =============================================================================
WUSTL_ENABLED = os.environ.get("WUSTL", "0") == "1"
WUSTL_PATH    = DATA_DIR / "wustl-ehms-2020.csv"


def summary() -> str:
    return "\n".join([
        "=== Experiment configuration ===",
        f"  MODE          : {'QUICK' if QUICK_MODE else 'FULL'}",
        f"  ROOT          : {ROOT}",
        f"  DATA_DIR      : {DATA_DIR}",
        f"  OUT_DIR       : {OUT_DIR}",
        f"  ARCHIVE_DIR   : {ARCHIVE_DIR}",
        f"  SEEDS         : {SEEDS}",
        f"  DATASET       : {DATASET}  ({N_CLASSES} classes)",
        f"  POOL / PCA    : {POOL_SIZE} samples, {N_FEATURES} -> {N_PCA}",
        f"  QCNN          : {N_QUBITS} qubits / {N_LAYERS} layers, "
        f"scales={ENTANGLE_SCALES}",
        f"  BACKEND       : {QUANTUM_DEVICE} / {DIFF_METHOD}",
        f"  EPOCHS        : binary {EPOCHS_BINARY}, multiclass {EPOCHS_MULTICLASS}",
        f"  ENLARGED TEST : {ENLARGED_TEST} (min {MIN_TEST_PER_CLASS}/class)",
    ])


# =============================================================================
# Compatibility shim
# -----------------------------------------------------------------------------
# circuits.py / models.py / train.py were verified BIT-EXACT against the
# archived Grid5000 weights (100% per-sample agreement on both tasks). They are
# ported here unchanged so that verification still holds; this shim exposes the
# structured accessors they expect on top of the flat constants above.
# =============================================================================
from dataclasses import dataclass, field, asdict as _asdict
from typing import Dict, Tuple


@dataclass(frozen=True)
class _Arch:
    N_QUBITS: int = N_QUBITS
    N_LAYERS: int = N_LAYERS
    ENC_BINARY: Tuple[int, ...] = (32,)
    DEC_BINARY: Tuple[int, ...] = (16,)
    ENC_MULTICLASS: Tuple[int, ...] = (64, 32)
    DEC_MULTICLASS: Tuple[int, ...] = (64, 32)
    DROPOUT_ENC: float = 0.20
    DROPOUT_DEC_BINARY: float = 0.20
    DROPOUT_DEC_MULTI: Tuple[float, float] = (0.15, 0.10)


@dataclass(frozen=True)
class _Train:
    EPOCHS_BINARY: int = EPOCHS_BINARY
    EPOCHS_MULTICLASS: int = EPOCHS_MULTICLASS
    LR_BINARY: float = LR_BINARY
    LR_MIN_BINARY: float = LR_MIN_BINARY
    LR_MULTICLASS: float = LR_MULTICLASS
    LR_MIN_MULTICLASS: float = LR_MIN_MULTICLASS
    BATCH_SIZE: int = BATCH_SIZE
    WEIGHT_DECAY: float = WEIGHT_DECAY
    GRAD_CLIP: float = GRAD_CLIP
    EARLY_STOP_PATIENCE: int = EARLY_STOP_PATIENCE
    EARLY_STOP_MIN_EPOCHS: int = EARLY_STOP_MIN_EPOCHS
    CHECKPOINT_EVERY_N_EPOCHS: int = 10
    SMOKE_EPOCHS: int = 2
    SMOKE_SAMPLES: int = 2000


@dataclass(frozen=True)
class _Quantum:
    BACKEND_PREFERENCE: Tuple[str, ...] = (QUANTUM_DEVICE,)
    BACKEND_FALLBACK: str = "lightning.qubit"
    DIFF_METHOD_PREFERRED: str = DIFF_METHOD
    DIFF_METHOD_FALLBACK: str = "adjoint"
    SHOTS_TRAINING_SUPPORTED: bool = SHOTS_TRAINING_SUPPORTED


@dataclass(frozen=True)
class _Data:
    PCA_COMPONENTS: int = N_PCA
    N_CLASSES: int = N_CLASSES


@dataclass(frozen=True)
class _CFG:
    arch: _Arch = field(default_factory=_Arch)
    train: _Train = field(default_factory=_Train)
    quantum: _Quantum = field(default_factory=_Quantum)
    data: _Data = field(default_factory=_Data)

    def as_dict(self) -> Dict:
        return {"arch": _asdict(self.arch), "train": _asdict(self.train),
                "quantum": _asdict(self.quantum), "data": _asdict(self.data),
                "seeds": list(SEEDS), "class_names": list(CLASS_NAMES),
                "dataset": DATASET, "angle_scale": ANGLE_SCALE}


CFG = _CFG()


def epochs_for(task: str, smoke: bool = False) -> int:
    if smoke:
        return CFG.train.SMOKE_EPOCHS
    return EPOCHS_BINARY if task == "binary" else EPOCHS_MULTICLASS


def lr_for(task: str):
    if task == "binary":
        return LR_BINARY, LR_MIN_BINARY
    return LR_MULTICLASS, LR_MIN_MULTICLASS


def n_classes_for(task: str) -> int:
    return 2 if task == "binary" else N_CLASSES


def set_global_seed(seed: int) -> None:
    """Seed every RNG the pipeline touches. Called once per run cell."""
    import random
    import numpy as _np
    import torch
    random.seed(seed)
    _np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


RESULTS_DIR = str(OUT_DIR)


# =============================================================================
# QUICK_MODE training-set cap
# -----------------------------------------------------------------------------
# QUICK_MODE reduces epochs and seeds, but the notebooks load their splits from
# the ARCHIVE (35,000 training rows), which POOL_SIZE does not affect. Without
# a cap, a "quick" ablation is 8 variants x 2 tasks x 3 seeds x 3 epochs at
# ~430 s/epoch -- about 17 hours, which defeats the purpose of a smoke test.
#
# subsample_for_mode() is applied to TRAINING data only. Validation and test
# splits stay whole, so the metrics remain interpretable.
# =============================================================================
QUICK_TRAIN_SAMPLES = 2_000


def subsample_for_mode(X, y, seed: int = 42):
    """Cap the training set in QUICK_MODE; return it untouched otherwise."""
    if not QUICK_MODE or len(X) <= QUICK_TRAIN_SAMPLES:
        return X, y
    import numpy as _np
    rng = _np.random.default_rng(seed)
    # Stratified so every class survives the cut -- an unstratified sample
    # would drop the rare classes and make the smoke test meaningless.
    idx = []
    for c in _np.unique(y):
        pos = _np.flatnonzero(y == c)
        take = max(1, int(round(len(pos) / len(y) * QUICK_TRAIN_SAMPLES)))
        idx.append(rng.choice(pos, min(take, len(pos)), replace=False))
    idx = _np.concatenate(idx)
    rng.shuffle(idx)
    print(f"[QUICK_MODE] training set capped {len(X)} -> {len(idx)} (stratified)")
    return X[idx], y[idx]

# ==================================================================
# === Inlined from src/persistence.py ===
# ==================================================================
"""
persistence.py -- comprehensive artefact persistence.

The design goal is:

    "store enough raw artefacts that any future statistical analysis
     (Friedman, McNemar, bootstrap, calibration, new SHAP samples, threshold
     sweeps, fairness metrics, etc.) can be added WITHOUT re-running any
     training."

Persisting every artefact avoids re-runs. Gaps in earlier Grid5000 runs that
would each have forced a re-run:

  * noise_results.json          kept F1 ONLY -> no accuracy/AUC/per-class, no
                                predictions, so no confidence intervals
  * sample_efficiency_results    F1 only
  * ablation_*_partial.json      7 scalars per variant; no weights, no curves,
                                no predictions
  * baselines                    best_params kept but NOT the fitted models and
                                NOT per-sample predictions -> the paired
                                McNemar tests were impossible
  * XAI                          PNG figures only; no SHAP/LIME values

Accordingly EVERY run persists five layers:

  1. scalars   metrics.json          -- the numbers that go in a table
  2. vectors   predictions.npz       -- y_true, y_pred, probs, test_idx
  3. curves    curves/*.csv + *.npz  -- every value behind every plotted line
  4. state     checkpoint.pt / model.joblib + preprocessors/
  5. context   manifest.json         -- config, seed, versions, timing, OAR id

A run is only considered COMPLETE when all applicable layers exist;
`check_completeness()` reports what is missing and is designed to be called
BEFORE the compute allocation is released.
"""

import json
import os
import platform
import subprocess
import sys
import time
from pathlib import Path
from typing import Any, Dict, Iterable, Optional, Sequence

import numpy as np
import pandas as pd


# =============================================================================
# Provenance (layer 5)
# =============================================================================
def _git_commit() -> str:
    try:
        return subprocess.check_output(["git", "rev-parse", "--short", "HEAD"],
                                       stderr=subprocess.DEVNULL).decode().strip()
    except Exception:
        return "nogit"


def _versions() -> Dict[str, str]:
    out = {"python": sys.version.split()[0]}
    for mod in ("numpy", "pandas", "sklearn", "torch", "pennylane",
                "shap", "lime", "statsmodels", "scipy"):
        try:
            out[mod] = getattr(__import__(mod), "__version__", "unknown")
        except Exception:
            out[mod] = "absent"
    return out


def build_context(seed: Optional[int] = None,
                  extra: Optional[Dict] = None) -> Dict:
    ctx = {
        "saved_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "hostname": platform.node(),
        "platform": platform.platform(),
        "cpu_count": os.cpu_count(),
        "git_commit": _git_commit(),
        "versions": _versions(),
        "seed": seed,
        "job": {k: os.environ.get(k) for k in
                ("OAR_JOB_ID", "OAR_ARRAY_INDEX", "SLURM_JOB_ID",
                 "OMP_NUM_THREADS", "MKL_NUM_THREADS")},
    }
    if extra:
        ctx.update(extra)
    return ctx


def _jsonable(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, (set, tuple)):
        return list(o)
    if isinstance(o, Path):
        return str(o)
    return str(o)


def write_json(path, obj) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, default=_jsonable), encoding="utf-8")
    return path


# =============================================================================
# Run directories
# =============================================================================
def run_dir(out_dir, experiment: str, variant: str = "main",
            task: str = "multiclass", split: str = "random",
            seed: int = 42) -> Path:
    """One directory per (experiment, variant, task, split, seed) cell.

    A walltime kill therefore costs one cell, never a whole stage, and
    re-submitting can skip completed cells (see `is_done`).
    """
    p = (Path(out_dir) / experiment / "runs" /
         f"{variant}__{task}__{split}__seed{seed}")
    p.mkdir(parents=True, exist_ok=True)
    return p


def is_done(out_dir, experiment: str, variant: str = "main",
            task: str = "multiclass", split: str = "random",
            seed: int = 42) -> bool:
    d = (Path(out_dir) / experiment / "runs" /
         f"{variant}__{task}__{split}__seed{seed}")
    return (d / "manifest.json").exists()


# =============================================================================
# Layer 3 -- curves
# =============================================================================
def save_curve(out_dir, experiment: str, curve_name: str, **series) -> Path:
    """Persist EVERY value behind a plotted line, as CSV and NPZ.

    Figures are not data. Saving the underlying values means that changing an
    axis, adding a confidence band, or re-plotting at a different DPI never
    requires recomputing the experiment.
    """
    d = Path(out_dir) / experiment / "curves"
    d.mkdir(parents=True, exist_ok=True)

    arrays = {k: np.asarray(v) for k, v in series.items()}
    np.savez_compressed(d / f"{curve_name}.npz", **arrays)

    n = max(len(v) for v in arrays.values())
    df = pd.DataFrame({k: pd.Series(list(v)) for k, v in arrays.items()},
                      index=range(n))
    csv = d / f"{curve_name}.csv"
    df.to_csv(csv, index=False)
    return csv


# =============================================================================
# Layer 1/2 -- sweeps in LONG format
# =============================================================================
def save_sweep(out_dir, experiment: str, sweep_name: str,
               records: Sequence[Dict]) -> Path:
    """Persist a sweep as a long-format table carrying EVERY metric.

    Every metric is kept (accuracy, precision, recall, F1 and AUC), so the
    analysis can be given confidence intervals without re-running it.
    """
    d = Path(out_dir) / experiment
    d.mkdir(parents=True, exist_ok=True)
    write_json(d / f"sweep_{sweep_name}.json",
               {"records": list(records), "context": build_context()})
    df = pd.DataFrame(list(records))
    csv = d / f"sweep_{sweep_name}.csv"
    df.to_csv(csv, index=False)
    return csv


# =============================================================================
# The main entry point
# =============================================================================
def save_run(out_dir, *, experiment: str, variant: str = "main",
             task: str = "multiclass", split: str = "random", seed: int = 42,
             y_true=None, y_pred=None, probs=None, test_idx=None,
             metrics: Optional[Dict] = None,
             history: Optional[Dict] = None,
             model=None, preprocessors: Optional[Dict] = None,
             params: Optional[Dict] = None,
             timings: Optional[Dict] = None,
             class_names: Optional[Sequence[str]] = None,
             extra: Optional[Dict] = None,
             save_checkpoint: bool = True) -> Path:
    """Persist all five artefact layers for one run cell."""
    d = run_dir(out_dir, experiment, variant, task, split, seed)
    files: Dict[str, str] = {}

    # -- layer 2: per-sample vectors (paired significance tests need these) --
    if y_true is not None and y_pred is not None:
        np.savez_compressed(
            d / "predictions.npz",
            y_true=np.asarray(y_true), y_pred=np.asarray(y_pred),
            probs=np.asarray(probs) if probs is not None else np.array([]),
            test_idx=np.asarray(test_idx) if test_idx is not None else np.array([]),
        )
        files["predictions"] = "predictions.npz"
        write_json(d / "per_class.json",
                   per_class_report(y_true, y_pred, class_names))
        files["per_class"] = "per_class.json"

    # -- layer 1: scalars ---------------------------------------------------
    if metrics:
        write_json(d / "metrics.json",
                   {k: v for k, v in metrics.items() if not k.startswith("_")})
        files["metrics"] = "metrics.json"

    # -- layer 3: curves ----------------------------------------------------
    if history:
        series = {k: v for k, v in history.items()
                  if isinstance(v, (list, tuple, np.ndarray))}
        if series:
            np.savez_compressed(d / "history.npz",
                                **{k: np.asarray(v) for k, v in series.items()})
            pd.DataFrame({k: pd.Series(list(v)) for k, v in series.items()}
                         ).to_csv(d / "history.csv", index=False)
            files["curves"] = "history.{csv,npz}"
        write_json(d / "history_scalars.json",
                   {k: v for k, v in history.items() if np.isscalar(v)})

    # -- layer 4: state -----------------------------------------------------
    if model is not None and save_checkpoint:
        try:
            import torch
            if hasattr(model, "state_dict"):
                torch.save(model.state_dict(), d / "checkpoint.pt")
                files["model"] = "checkpoint.pt"
                if params is None:
                    params = {"total": int(sum(p.numel() for p in model.parameters()
                                               if p.requires_grad))}
            else:
                raise AttributeError
        except Exception:
            try:
                import joblib
                joblib.dump(model, d / "model.joblib")
                files["model"] = "model.joblib"
            except Exception as e:      # never let persistence kill a run
                files["model_error"] = str(e)

    if preprocessors:
        # Same rule as the model branch above: persistence must never kill a
        # run. A long-path failure on Windows, a full disk, or an unpicklable
        # transformer costs the preprocessors, not the metrics and predictions
        # that took the hours to produce -- and the failure is recorded rather
        # than swallowed.
        import joblib
        pdir = d / "preprocessors"
        saved, failed = [], {}
        try:
            pdir.mkdir(parents=True, exist_ok=True)
        except Exception as e:
            failed["__mkdir__"] = str(e)
        for k, v in preprocessors.items():
            try:
                joblib.dump(v, pdir / f"{k}.joblib")
                saved.append(k)
            except Exception as e:
                failed[k] = f"{type(e).__name__}: {e}"
        if saved:
            files["preprocessors"] = ",".join(sorted(saved))
        if failed:
            files["preprocessors_error"] = json.dumps(failed)
            print(f"[persist] WARNING: preprocessors not saved: {failed}",
                  flush=True)

    # -- layer 5: context ---------------------------------------------------
    write_json(d / "manifest.json", {
        "experiment": experiment, "variant": variant, "task": task,
        "split": split, "seed": seed,
        "files": files, "params": params or {}, "timings": timings or {},
        "metrics": {k: v for k, v in (metrics or {}).items()
                    if not k.startswith("_")},
        "context": build_context(seed=seed, extra=extra),
    })

    _append_index(out_dir, experiment, variant, task, split, seed, metrics, params)
    print(f"[persist] {experiment}/{variant}__{task}__{split}__seed{seed}"
          f" -> {sorted(files)}", flush=True)
    return d


def per_class_report(y_true, y_pred, class_names=None) -> Dict:
    """Per-class precision/recall/F1/support + confusion matrix.

    Always saved, never only the weighted average -- the per-class table is a
    paper table in its own right (Table 10 of the paper).
    """
    from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
    labels = sorted(set(np.asarray(y_true).tolist()) |
                    set(np.asarray(y_pred).tolist()))
    p, r, f1, sup = precision_recall_fscore_support(
        y_true, y_pred, average=None, labels=labels, zero_division=0)
    return {
        "labels": [int(x) for x in labels],
        "class_names": list(class_names) if class_names else
                       [str(x) for x in labels],
        "precision": p.tolist(), "recall": r.tolist(),
        "f1": f1.tolist(), "support": sup.tolist(),
        "confusion_matrix": confusion_matrix(y_true, y_pred,
                                             labels=labels).tolist(),
    }


def _append_index(out_dir, experiment, variant, task, split, seed,
                  metrics, params) -> None:
    """Maintain runs_index.csv so aggregation is a one-line DataFrame read."""
    idx = Path(out_dir) / experiment / "runs_index.csv"
    row = {"experiment": experiment, "variant": variant, "task": task,
           "split": split, "seed": seed,
           "n_params": (params or {}).get("total")}
    for k, v in (metrics or {}).items():
        if not k.startswith("_") and np.isscalar(v):
            row[k] = v
    df = pd.DataFrame([row])
    if idx.exists():
        df = pd.concat([pd.read_csv(idx), df], ignore_index=True)
        df = df.drop_duplicates(subset=["experiment", "variant", "task",
                                        "split", "seed"], keep="last")
    idx.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(idx, index=False)


# =============================================================================
# Loading (for aggregation -- no experiment is ever re-run to make a table)
# =============================================================================
def load_index(out_dir, experiment: str) -> pd.DataFrame:
    p = Path(out_dir) / experiment / "runs_index.csv"
    return pd.read_csv(p) if p.exists() else pd.DataFrame()


def load_predictions(out_dir, experiment: str, variant="main",
                     task="multiclass", split="random", seed=42) -> Dict:
    d = (Path(out_dir) / experiment / "runs" /
         f"{variant}__{task}__{split}__seed{seed}")
    z = np.load(d / "predictions.npz")
    return {k: z[k] for k in z.files}


def load_all_runs(out_dir, experiment: str) -> list:
    root = Path(out_dir) / experiment / "runs"
    out = []
    if not root.exists():
        return out
    for d in sorted(root.iterdir()):
        m = d / "manifest.json"
        if m.exists():
            out.append(json.loads(m.read_text(encoding="utf-8")))
    return out


# =============================================================================
# Completeness gate -- call BEFORE releasing the Grid5000 allocation
# =============================================================================
REQUIRED_LAYERS = ("metrics", "predictions", "curves", "model")


def check_completeness(out_dir, experiment: Optional[str] = None,
                       required: Iterable[str] = REQUIRED_LAYERS,
                       strict: bool = False) -> pd.DataFrame:
    """Report which run cells are missing which artefact layer.

    Run this before the cluster allocation is released, so that any gap can be
    fixed while the cluster is still reserved.
    """
    out_dir = Path(out_dir)
    exps = ([experiment] if experiment
            else [p.name for p in out_dir.iterdir() if p.is_dir()])
    rows = []
    for exp in exps:
        for man in load_all_runs(out_dir, exp):
            present = set(man.get("files", {}))
            missing = [l for l in required if l not in present]
            rows.append({"experiment": exp, "variant": man["variant"],
                         "task": man["task"], "split": man["split"],
                         "seed": man["seed"],
                         "present": ",".join(sorted(present)),
                         "missing": ",".join(missing), "complete": not missing})
    df = pd.DataFrame(rows)
    if df.empty:
        print(f"[completeness] no runs found under {out_dir}")
        return df

    n_bad = int((~df["complete"]).sum())
    print(f"\n{'='*70}\nARTEFACT COMPLETENESS -- {len(df)} run cells\n{'='*70}")
    for _, r in df.iterrows():
        flag = "OK " if r["complete"] else "GAP"
        print(f"  [{flag}] {r['experiment']}/{r['variant']}__{r['task']}"
              f"__{r['split']}__seed{r['seed']}   missing: {r['missing'] or '-'}")
    if n_bad:
        msg = (f"\n  {n_bad} incomplete cell(s). FIX BEFORE RELEASING THE "
               f"ALLOCATION -- regenerating any of these means re-running the job.")
        print(msg)
        if strict:
            raise RuntimeError(msg)
    else:
        print("\n  All cells complete. Safe to release the allocation.")

    write_json(out_dir / "_completeness_report.json", rows)
    return df


# =============================================================================
# Notebook dependency guards
# =============================================================================
# Notebooks form a DAG: 04 needs 00's QCNN predictions for the paired McNemar
# tests, 13 needs everything. On a cluster the failure mode is a KeyError or a
# FileNotFoundError hundreds of lines into a job that has already burned an
# hour of walltime. These guards fail in the FIRST cell with a message that
# says which notebook to run first.

NOTEBOOK_DEPENDENCIES = {
    "nb00_verify":     [],
    "nb01_data":       [],
    "nb02_binary":     [],
    "nb03_multiclass": [],
    "nb04_baselines":  ["nb00_verify"],   # McNemar needs the QCNN predictions
    "nb05_ablation":   [],
    "nb06_sweep":      [],
    "nb07_robustness": [],
    "nb08_shots":      [],
    "nb09_xai":        [],
    "nb10_audit":      [],
    "nb11_cost":       [],
    "nb12_cross":      [],
    "nb14_quantum":    ["nb00_verify"],   # McNemar needs the QCNN predictions
    "nb15_gap":        [],
    "nb16_splits":     [],
    "nb13_aggregate":  ["nb00_verify"],   # everything else is optional here
}

NOTEBOOK_FILES = {
    "nb00_verify": "00_verify_archive.ipynb",
    "nb01_data": "01_data_preparation.ipynb",
    "nb02_binary": "02_qcnn_binary.ipynb",
    "nb03_multiclass": "03_qcnn_multiclass.ipynb",
    "nb04_baselines": "04_baselines_multiseed.ipynb",
    "nb05_ablation": "05_ablation.ipynb",
    "nb06_sweep": "06_qubit_depth_sweep.ipynb",
    "nb07_robustness": "07_robustness_adversarial.ipynb",
    "nb08_shots": "08_shots_and_noise.ipynb",
    "nb09_xai": "09_shap_explainability.ipynb",
    "nb10_audit": "10_audit_log.ipynb",
    "nb11_cost": "11_grid5000_timing_energy.ipynb",
    "nb12_cross": "12_cross_dataset_wustl.ipynb",
    "nb14_quantum": "14_quantum_related_baselines.ipynb",
    "nb15_gap": "15_multiclass_gap.ipynb",
    "nb16_splits": "16_split_protocols.ipynb",
    "nb13_aggregate": "13_results_aggregation.ipynb",
}


def require_experiment(out_dir, experiment: str, hint: str = "") -> None:
    """Fail immediately if a prerequisite experiment has produced nothing."""
    root = Path(out_dir) / experiment
    runs = root / "runs"
    ok = runs.exists() and any(
        (d / "manifest.json").exists() for d in runs.iterdir() if d.is_dir())
    if not ok:
        nb = NOTEBOOK_FILES.get(experiment, experiment)
        raise FileNotFoundError(
            f"\nMISSING PREREQUISITE: '{experiment}' has no completed runs under "
            f"{root}.\n"
            f"  Run  {nb}  first.\n"
            + (f"  {hint}\n" if hint else "")
        )


def require_archive(archive_dir) -> None:
    """Fail immediately if the archived Grid5000 artefacts are not reachable.

    On the cluster the package is usually staged into $HOME, so the default
    relative path can miss. Setting ARCHIVE_DIR fixes it.
    """
    p = Path(archive_dir)
    needed = ["prepared_data.npz", "scaler.joblib",
              "hybrid_qcnn_binary (1).pt", "hybrid_qcnn_multiclass (1).pt"]
    missing = [f for f in needed if not (p / f).exists()]
    if missing:
        raise FileNotFoundError(
            f"\nARCHIVE NOT FOUND at {p}\n"
            f"  missing: {missing}\n"
            f"  Set ARCHIVE_DIR to the folder holding salvaged_g5k_artifacts, e.g.\n"
            f"    export ARCHIVE_DIR=$HOME/qmedshield_g5k/salvaged_g5k_artifacts\n"
        )
    print(f"[deps] archive OK: {p}")


def require_packages(*names: str) -> None:
    """Fail immediately if an optional dependency is absent."""
    missing = []
    for n in names:
        try:
            __import__(n)
        except ImportError:
            missing.append(n)
    if missing:
        raise ImportError(
            f"\nMISSING PACKAGES: {missing}\n  pip install {' '.join(missing)}\n")
    print(f"[deps] packages OK: {list(names)}")


def preflight(out_dir, archive_dir, experiment: str,
              packages: Sequence[str] = (), needs_data: bool = False,
              data_dir=None) -> None:
    """One call at the top of every notebook: verify everything it needs.

    Cheap, and it converts a mid-job crash on the cluster into an immediate,
    actionable message.
    """
    print(f"[preflight] {experiment}")
    for dep in NOTEBOOK_DEPENDENCIES.get(experiment, []):
        require_experiment(out_dir, dep)
        print(f"[deps] prerequisite OK: {dep}")
    if archive_dir is not None:
        require_archive(archive_dir)
    if packages:
        require_packages(*packages)
    if needs_data and data_dir is not None:
        csvs = list(Path(data_dir).glob("*.csv"))
        print(f"[deps] raw CSVs in {data_dir}: {len(csvs)}"
              + ("" if csvs else "  (split protocols will be skipped)"))
    print("[preflight] all checks passed\n")

# ==================================================================
# === Inlined from src/circuits.py ===
# ==================================================================
"""
qmed.circuits — parameterized quantum circuits with composable ablation flags.

Every circuit variant used in the package is produced by ONE builder, so the
ablation variants cannot drift away from the headline architecture.

Entanglement scales (section 3.3 of the paper):
    l = 1  ring CNOT      -- all layers
    l = 2  skip-one CZ    -- even layers
    l = 4  cross-half CNOT-- odd layers

The ablation varies the ACTIVE SCALE SET to give a per-scale attribution of
the entanglement structure.
"""


from typing import Sequence, Tuple

import numpy as np
import pennylane as qml

# from config import CFG   # inlined


# ---------------------------------------------------------------------------
# Backend selection
# ---------------------------------------------------------------------------
def select_backend(n_qubits: int, shots: int | None = None) -> Tuple[str, str]:
    """Return (backend_name, diff_method).

    Backend choice is MEASURED, not assumed -- see bench_backend.py (P0.7) and
    the note in qmed.config.QuantumConfig. default.qubit + backprop is 8.3x
    faster than lightning.qubit + adjoint at n_q = 8, because the batch stays
    in one broadcast torch tensor instead of being executed sample by sample.

    Whatever this resolves to is recorded in every result JSON and must match
    what section 4.1 of the paper reports.

    Finite shots are inference-only; see SHOTS_TRAINING_SUPPORTED.
    """
    if shots is not None:
        # Sampling device for the finite-shot experiment. Gradients are not required here.
        for backend in CFG.quantum.BACKEND_PREFERENCE + (CFG.quantum.BACKEND_FALLBACK,):
            try:
                qml.device(backend, wires=n_qubits, shots=shots)
                return backend, "parameter-shift"
            except Exception:
                continue
        raise RuntimeError(f"no shot-capable backend for n_qubits={n_qubits}")

    for backend in CFG.quantum.BACKEND_PREFERENCE:
        try:
            qml.device(backend, wires=n_qubits)
            return backend, CFG.quantum.DIFF_METHOD_PREFERRED
        except Exception:
            continue
    return CFG.quantum.BACKEND_FALLBACK, CFG.quantum.DIFF_METHOD_FALLBACK


def make_device(n_qubits: int, shots: int | None = None):
    backend, diff = select_backend(n_qubits, shots)
    dev = qml.device(backend, wires=n_qubits, shots=shots)
    return dev, backend, diff


# ---------------------------------------------------------------------------
# Entanglement blocks
# ---------------------------------------------------------------------------
def _apply_entanglement(layer: int, n_qubits: int, scales: Sequence[int]) -> None:
    """Apply the composite multi-scale entangler for the requested scales.

    `scales` is a subset of {1, 2, 4}. The full architecture uses all three;
    The ring_only / ring_skip / ring_cross ablation variants use subsets, which is what
    makes the per-scale contribution measurable.
    """
    if 1 in scales:
        for i in range(n_qubits):
            qml.CNOT(wires=[i, (i + 1) % n_qubits])

    # Scales 2 and 4 alternate by layer parity so that consecutive layers do
    # not repeat the same connectivity (section 3.3 rationale in the paper).
    if 2 in scales and layer % 2 == 0:
        for i in range(0, n_qubits, 2):
            qml.CZ(wires=[i, (i + 2) % n_qubits])

    if 4 in scales and layer % 2 == 1:
        for i in range(n_qubits // 2):
            qml.CNOT(wires=[i, i + n_qubits // 2])


# ---------------------------------------------------------------------------
# The single circuit builder
# ---------------------------------------------------------------------------
def build_qnode(
    dev,
    n_qubits: int,
    n_layers: int,
    diff_method: str,
    use_reuploading: bool = True,
    entangle_scales: Sequence[int] = (1, 2, 4),
):
    """Return a batch-broadcasting PennyLane QNode.

    Parameters
    ----------
    use_reuploading
        True  -> angle embedding at EVERY layer (full architecture)
        False -> angle embedding at layer 0 only (`no_reupload` variant)
    entangle_scales
        ()        -> no entanglement           (`no_entangle`)
        (1,)      -> ring only                 (`ring_only`)
        (1,2)     -> ring + skip-one           (`ring_skip`)
        (1,4)     -> ring + cross-half         (`ring_cross`)
        (1,2,4)   -> full composite entangler  (headline)

    Inputs may be shaped (n_qubits,) or (B, n_qubits); PennyLane parameter
    broadcasting vectorises the batch inside the simulator, which is the
    single largest speedup available on CPU.
    """
    scales = tuple(entangle_scales)

    # --- ENCODING CONVENTION (verified against the archived weights) -------
    # The paper, eq. (3.3), defines  AngleEmbed(z) = (x) R_Y(z_j)  with NO
    # pi factor, and the shared_config.py that actually trained the archived
    # Grid'5000 models uses `qml.AngleEmbedding(inputs, rotation='Y')`, i.e.
    # R_Y(z_j) exactly.
    #
    # `smote_correction_optimized_V2/shared_config.py` silently introduced a
    # pi scaling, `qml.RY(inputs[..., i] * np.pi)`, which is a DIFFERENT model:
    # loading the archived weights into it gives 2.9% binary accuracy against
    # the published 99.63%. Do not reintroduce the pi factor -- ANGLE_SCALE
    # exists only to make the convention explicit and testable.
    ANGLE_SCALE = 1.0

    @qml.qnode(dev, interface="torch", diff_method=diff_method)
    def circuit(inputs, weights):
        for layer in range(n_layers):
            if use_reuploading or layer == 0:
                for i in range(n_qubits):
                    qml.RY(inputs[..., i] * ANGLE_SCALE, wires=i)

            for i in range(n_qubits):
                qml.Rot(
                    weights[layer, i, 0],
                    weights[layer, i, 1],
                    weights[layer, i, 2],
                    wires=i,
                )

            if scales:
                _apply_entanglement(layer, n_qubits, scales)

        return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

    return circuit


def n_quantum_params(n_qubits: int, n_layers: int) -> int:
    """|theta_q| = 3 * L * n_q. Used by the parameter-budget table."""
    return 3 * n_layers * n_qubits


def circuit_evals_per_forward(batch_size: int = 1) -> int:
    """Circuit evaluations per forward pass -- feeds the cost model."""
    return batch_size


def circuit_evals_per_gradient(n_qubits: int, n_layers: int) -> int:
    """Parameter-shift gradient cost: 2 evaluations per trainable quantum
    parameter. This is the number that makes 'parameter count is not
    deployment cost' concrete."""
    return 2 * n_quantum_params(n_qubits, n_layers)


# ---------------------------------------------------------------------------
# Variant registry used by the ablation
# ---------------------------------------------------------------------------
CIRCUIT_VARIANTS = {
    "full":         dict(use_reuploading=True,  entangle_scales=(1, 2, 4)),
    "no_reupload":  dict(use_reuploading=False, entangle_scales=(1, 2, 4)),
    "no_entangle":  dict(use_reuploading=True,  entangle_scales=()),
    "ring_only":    dict(use_reuploading=True,  entangle_scales=(1,)),
    "ring_skip":    dict(use_reuploading=True,  entangle_scales=(1, 2)),
    "ring_cross":   dict(use_reuploading=True,  entangle_scales=(1, 4)),
}

# ==================================================================
# === Inlined from src/models.py ===
# ==================================================================
"""
qmed.models — ONE model factory for the headline models and every ablation.

Every model variant (headline models and ablations) is built by this single
factory, so parameter counts differ only where the bottleneck differs.

EVERY variant is `HybridIDS` with identical encoder and decoder; only
`bottleneck` changes. `assert_scaffolding_identical()` enforces this and is
called before any ablation training (validation gate G4), so a scaffolding
mismatch cannot be introduced silently.
"""


import math
from typing import Dict, List, Optional, Sequence

import numpy as np
import torch
import torch.nn as nn

# from circuits import (   # inlined
#     CIRCUIT_VARIANTS,   # inlined
#     build_qnode,   # inlined
#     make_device,   # inlined
#     n_quantum_params,   # inlined
# )   # inlined
# from config import CFG, n_classes_for   # inlined


# ---------------------------------------------------------------------------
# Bottleneck modules -- all map R^{n_q} -> R^{n_q}
# ---------------------------------------------------------------------------
class QuantumBottleneck(nn.Module):
    """The 8-qubit PQC. 3*L*n_q trainable parameters (96 at L=4, n_q=8)."""

    def __init__(self, n_qubits: int, n_layers: int, use_reuploading: bool = True,
                 entangle_scales: Sequence[int] = (1, 2, 4), shots: Optional[int] = None):
        super().__init__()
        self.n_qubits, self.n_layers, self.shots = n_qubits, n_layers, shots
        self.dev, self.backend, self.diff_method = make_device(n_qubits, shots)
        self.q_weights = nn.Parameter(torch.randn(n_layers, n_qubits, 3) * 0.1)
        self.qnode = build_qnode(
            self.dev, n_qubits, n_layers, self.diff_method,
            use_reuploading=use_reuploading, entangle_scales=entangle_scales,
        )
        self.n_circuit_evals = 0  # incremented per forward; feeds the cost model

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        self.n_circuit_evals += x.shape[0]
        q = self.qnode(x, self.q_weights)
        # PennyLane returns float64 expectation values; cast back so the
        # classical decoder (float32) and torch.compile stay happy.
        return torch.stack(list(q), dim=-1).to(x.dtype)


class LinearBottleneck(nn.Module):
    """Dense n_q -> n_q. 72 params at n_q=8. The tightest classical control
    for the PQC (matched-control Family A)."""

    def __init__(self, n_qubits: int):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_qubits, n_qubits), nn.Tanh())

    def forward(self, x):
        return self.net(x)


class FourierBottleneck(nn.Module):
    """Random Fourier features: matches the PQC's smooth trigonometric map
    without any quantum mechanics. Trainable frequencies + phases give 96
    parameters at n_q=8, D=... (chosen to match |theta_q| exactly).

    This is the control that distinguishes 'the quantum circuit helps' from
    'a smooth periodic bottleneck helps'.
    """

    def __init__(self, n_qubits: int, n_params_target: int = 96):
        super().__init__()
        # freq: (D, n_q), phase: (D,) -> D*(n_q+1) params; pick D to match.
        D = max(1, n_params_target // (n_qubits + 1))
        self.freq = nn.Parameter(torch.randn(D, n_qubits) * 0.1)
        self.phase = nn.Parameter(torch.zeros(D))
        self.proj = nn.Linear(D, n_qubits, bias=False)
        self.proj.weight.requires_grad_(False)  # fixed random readout
        nn.init.normal_(self.proj.weight, std=1.0 / math.sqrt(D))

    def forward(self, x):
        h = torch.cos(x @ self.freq.T * math.pi + self.phase)
        return torch.tanh(self.proj(h))


class MLP96Bottleneck(nn.Module):
    """Tiny MLP tuned to ~96 trainable parameters (matched-control Family A)."""

    def __init__(self, n_qubits: int, n_params_target: int = 96):
        super().__init__()
        # n_q*h + h + h*n_q + n_q = 2*n_q*h + h + n_q  ->  solve for h
        h = max(1, (n_params_target - n_qubits) // (2 * n_qubits + 1))
        self.net = nn.Sequential(
            nn.Linear(n_qubits, h), nn.GELU(), nn.Linear(h, n_qubits), nn.Tanh()
        )

    def forward(self, x):
        return self.net(x)


class IdentityBottleneck(nn.Module):
    """Encoder -> decoder directly. 0 parameters (matched-control Family A)."""

    def forward(self, x):
        return x


def build_bottleneck(kind: str, n_qubits: int, n_layers: int,
                     use_reuploading: bool = True,
                     entangle_scales: Sequence[int] = (1, 2, 4),
                     shots: Optional[int] = None) -> nn.Module:
    if kind == "pqc":
        return QuantumBottleneck(n_qubits, n_layers, use_reuploading,
                                 entangle_scales, shots)
    if kind == "linear":
        return LinearBottleneck(n_qubits)
    if kind == "fourier":
        return FourierBottleneck(n_qubits, n_quantum_params(n_qubits, n_layers))
    if kind == "mlp96":
        return MLP96Bottleneck(n_qubits, n_quantum_params(n_qubits, n_layers))
    if kind == "identity":
        return IdentityBottleneck()
    raise ValueError(f"unknown bottleneck: {kind}")


# ---------------------------------------------------------------------------
# Encoder / decoder builders
# ---------------------------------------------------------------------------
# Extracted verbatim from HybridIDS.__init__ so that the hierarchical and
# feature-path models of NB15 reuse the SAME scaffolding instead of
# copying it. The module names and layer order are unchanged, so archived
# state_dicts still load bit-exactly (NB00 verifies this on every run).
def build_encoder(task: str, n_features: int, n_qubits: int) -> nn.Sequential:
    """Classical encoder. Determined ONLY by `task` -- never by a variant."""
    a = CFG.arch
    if task == "binary":
        (h1,) = a.ENC_BINARY
        return nn.Sequential(
            nn.Linear(n_features, h1), nn.LayerNorm(h1), nn.GELU(),
            nn.Dropout(a.DROPOUT_ENC), nn.Linear(h1, n_qubits), nn.Tanh(),
        )
    h1, h2 = a.ENC_MULTICLASS
    return nn.Sequential(
        nn.Linear(n_features, h1), nn.LayerNorm(h1), nn.GELU(),
        nn.Dropout(a.DROPOUT_ENC),
        nn.Linear(h1, h2), nn.LayerNorm(h2), nn.GELU(),
        nn.Linear(h2, n_qubits), nn.Tanh(),
    )


def build_decoder(task: str, n_in: int, n_classes: int) -> nn.Sequential:
    """Classical decoder. `n_in` is the bottleneck width; it is the only thing
    the feature-path model of NB15 changes, and it changes it explicitly."""
    a = CFG.arch
    if task == "binary":
        (d1,) = a.DEC_BINARY
        return nn.Sequential(
            nn.Linear(n_in, d1), nn.LayerNorm(d1), nn.GELU(),
            nn.Dropout(a.DROPOUT_DEC_BINARY), nn.Linear(d1, n_classes),
        )
    d1, d2 = a.DEC_MULTICLASS
    p1, p2 = a.DROPOUT_DEC_MULTI
    return nn.Sequential(
        nn.Linear(n_in, d1), nn.LayerNorm(d1), nn.GELU(),
        nn.Dropout(p1), nn.Linear(d1, d2), nn.GELU(),
        nn.Dropout(p2), nn.Linear(d2, n_classes),
    )


# ---------------------------------------------------------------------------
# The single hybrid model
# ---------------------------------------------------------------------------
class HybridIDS(nn.Module):
    """Encoder -> bottleneck -> decoder.

    The encoder and decoder are determined ONLY by `task`. No variant, no
    ablation, and no experiment may change them -- that invariant is enforced
    by `assert_scaffolding_identical`.
    """

    def __init__(self, task: str, n_features: int = 16, n_qubits: int = 8,
                 n_layers: int = 4, bottleneck: str = "pqc",
                 use_reuploading: bool = True,
                 entangle_scales: Sequence[int] = (1, 2, 4),
                 shots: Optional[int] = None,
                 n_classes: Optional[int] = None):
        super().__init__()
        self.task = task
        self.n_qubits = n_qubits
        self.n_layers = n_layers
        self.bottleneck_kind = bottleneck
        n_classes = n_classes if n_classes is not None else n_classes_for(task)
        self.n_classes = n_classes

        # ---- encoder (task-determined, variant-invariant) -----------------
        self.classical_encoder = build_encoder(task, n_features, n_qubits)

        # ---- bottleneck (the ONLY thing a variant may change) -------------
        self.bottleneck = build_bottleneck(
            bottleneck, n_qubits, n_layers, use_reuploading, entangle_scales, shots
        )

        # ---- decoder (task-determined, variant-invariant) -----------------
        self.classical_decoder = build_decoder(task, n_qubits, n_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        z = self.classical_encoder(x)
        q = self.bottleneck(z)
        return self.classical_decoder(q)

    # -- introspection used by the result registry --------------------------
    def param_breakdown(self) -> Dict[str, int]:
        """Per-component trainable parameter counts, computed from the LIVE
        model. Table 8's parameter columns are generated from this, never
        transcribed."""
        def n(m):
            return sum(p.numel() for p in m.parameters() if p.requires_grad)

        enc, bot, dec = (n(self.classical_encoder), n(self.bottleneck),
                         n(self.classical_decoder))
        return {"encoder": enc, "bottleneck": bot, "decoder": dec,
                "total": enc + bot + dec}

    def footprint_kb(self, bits: int = 32) -> float:
        """Deployable size. Reported alongside -- never instead of -- the
        operational cost measured by the cost model."""
        return self.param_breakdown()["total"] * (bits / 8) / 1024.0


# ---------------------------------------------------------------------------
# Variant construction for the matched controls and the ablation
# ---------------------------------------------------------------------------
def build_model(task: str, variant: str = "full", n_features: int = 16,
                n_qubits: int = 8, n_layers: int = 4,
                shots: Optional[int] = None,
                n_classes: Optional[int] = None) -> HybridIDS:
    """Build any ablation variant or matched-control bottleneck variant by name.

    `n_classes` overrides the head width only. NB15's stage-1 model needs a
    7-way head (DoS and DDoS merged) on otherwise identical scaffolding; every
    other caller leaves it at None and gets the task default, so no existing
    result changes.
    """
    if variant in CIRCUIT_VARIANTS:
        return HybridIDS(task, n_features, n_qubits, n_layers,
                         bottleneck="pqc", shots=shots, n_classes=n_classes,
                         **CIRCUIT_VARIANTS[variant])

    # Matched-control Family A: same scaffolding, non-quantum bottleneck.
    if variant in ("linear", "fourier", "mlp96", "identity"):
        return HybridIDS(task, n_features, n_qubits, n_layers,
                         bottleneck=variant, n_classes=n_classes)

    if variant == "classical_matched":
        return HybridIDS(task, n_features, n_qubits, n_layers,
                         bottleneck="mlp96", n_classes=n_classes)

    raise ValueError(f"unknown variant: {variant}")


class SimpleMLP(nn.Module):
    """Minimal 3-layer reference network (`simple_mlp`). Deliberately does
    NOT share the scaffolding -- it is a different architecture by design, and
    it must be described as such."""

    def __init__(self, task: str, n_features: int = 16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, n_classes_for(task)),
        )

    def forward(self, x):
        return self.net(x)

    def param_breakdown(self) -> Dict[str, int]:
        t = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return {"encoder": 0, "bottleneck": 0, "decoder": 0, "total": t}

    def footprint_kb(self, bits: int = 32) -> float:
        return self.param_breakdown()["total"] * (bits / 8) / 1024.0


# ---------------------------------------------------------------------------
# Validation gate G4 -- called before any ablation training
# ---------------------------------------------------------------------------
def assert_scaffolding_identical(models: Dict[str, nn.Module],
                                 exempt: Sequence[str] = ("simple_mlp",)) -> None:
    """Raise if any two ablation variants differ in encoder or decoder size.

    It runs
    BEFORE training, so a scaffolding mismatch costs seconds, not a full
    ablation sweep.
    """
    ref_name, ref = None, None
    for name, m in models.items():
        if name in exempt:
            continue
        b = m.param_breakdown()
        if ref is None:
            ref_name, ref = name, b
            continue
        for part in ("encoder", "decoder"):
            if b[part] != ref[part]:
                raise AssertionError(
                    f"Scaffolding mismatch: "
                    f"variant '{name}' has {part}={b[part]} but '{ref_name}' has "
                    f"{part}={ref[part]}. Every ablation variant must share the "
                    f"identical classical encoder and decoder; only the "
                    f"bottleneck may differ."
                )

# ==================================================================
# === Inlined from src/train.py ===
# ==================================================================
"""
qmed.train — training and evaluation primitives.

Model selection is done on validation data only, never on the test set.

`train_model` accepts a validation loader only, and `assert_no_leakage`
(validation gate G3) refuses to run if test indices appear in the training or
validation folds. Test evaluation happens exactly once, in the experiment
script, after training returns.
"""


import os
import time
from typing import Dict, Optional, Sequence

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
    roc_auc_score,
)

# from config import CFG, epochs_for, lr_for   # inlined

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ---------------------------------------------------------------------------
# Validation gate G3 -- leakage assertion
# ---------------------------------------------------------------------------
def assert_no_leakage(train_idx: np.ndarray, val_idx: np.ndarray,
                      test_idx: np.ndarray) -> None:
    """Raise if the test indices intersect training or validation indices.

    Assert it; don't trust it.
    """
    tr, va, te = set(map(int, train_idx)), set(map(int, val_idx)), set(map(int, test_idx))
    if tr & te:
        raise AssertionError(
            f"LEAKAGE (gate G3): {len(tr & te)} test indices appear in the "
            f"training split."
        )
    if va & te:
        raise AssertionError(
            f"LEAKAGE (gate G3): {len(va & te)} test indices appear in the "
            f"validation split."
        )
    if tr & va:
        raise AssertionError(
            f"LEAKAGE (gate G3): {len(tr & va)} indices are shared between the "
            f"training and validation splits."
        )


# ---------------------------------------------------------------------------
# Data plumbing
# ---------------------------------------------------------------------------
def make_loader(X: np.ndarray, y: np.ndarray, batch_size: int = None,
                shuffle: bool = True) -> DataLoader:
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32),
                       torch.tensor(y, dtype=torch.long))
    return DataLoader(ds, batch_size=batch_size or CFG.train.BATCH_SIZE,
                      shuffle=shuffle, num_workers=0)


def compute_class_weights(y: np.ndarray, n_classes: int) -> torch.Tensor:
    """sqrt-dampened inverse frequency, as in the original work."""
    counts = np.bincount(y, minlength=n_classes)
    w = np.array([1.0 / np.sqrt(max(c, 1)) for c in counts], dtype=np.float32)
    w = w / w.sum() * n_classes
    return torch.tensor(w, dtype=torch.float32)


# ---------------------------------------------------------------------------
# Evaluation
# ---------------------------------------------------------------------------
@torch.no_grad()
def evaluate(model, loader, n_classes: int = 2, device=DEVICE) -> Dict:
    model.eval()
    preds_all, labels_all, probs_all = [], [], []
    for xb, yb in loader:
        logits = model(xb.to(device))
        probs_all.append(F.softmax(logits, dim=1).cpu().numpy())
        preds_all.append(logits.argmax(1).cpu().numpy())
        labels_all.append(yb.numpy())

    preds = np.concatenate(preds_all)
    labels = np.concatenate(labels_all)
    probs = np.concatenate(probs_all)

    average = "binary" if n_classes == 2 else "weighted"
    p, r, f1, _ = precision_recall_fscore_support(
        labels, preds, average=average, zero_division=0
    )
    try:
        auc = (roc_auc_score(labels, probs[:, 1]) if n_classes == 2
               else roc_auc_score(labels, probs, multi_class="ovr", average="weighted"))
    except Exception:
        auc = float("nan")

    # Per-class metrics -- with the enlarged test set these are
    # finally reportable for every class, including the rare ones.
    pc_p, pc_r, pc_f1, pc_sup = precision_recall_fscore_support(
        labels, preds, average=None, labels=list(range(n_classes)), zero_division=0
    )

    return {
        "accuracy": float(accuracy_score(labels, preds)),
        "precision": float(p), "recall": float(r), "f1": float(f1), "auc": float(auc),
        "per_class": {
            "precision": pc_p.tolist(), "recall": pc_r.tolist(),
            "f1": pc_f1.tolist(), "support": pc_sup.tolist(),
        },
        "confusion_matrix": confusion_matrix(
            labels, preds, labels=list(range(n_classes))
        ).tolist(),
        "_predictions": preds, "_labels": labels, "_probabilities": probs,
    }


# ---------------------------------------------------------------------------
# Training
# ---------------------------------------------------------------------------
def train_epoch(model, loader, optimizer, criterion, device=DEVICE):
    model.train()
    total_loss = correct = total = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), CFG.train.GRAD_CLIP)
        optimizer.step()
        total_loss += loss.item() * yb.size(0)
        correct += (logits.argmax(1) == yb).sum().item()
        total += yb.size(0)
    return total_loss / total, correct / total


def train_model(model: nn.Module, train_loader: DataLoader,
                val_loader: DataLoader, task: str,
                class_weights: Optional[torch.Tensor] = None,
                smoke: bool = False, checkpoint_path: Optional[str] = None,
                device=DEVICE, verbose: bool = True) -> Dict:
    """Train with early stopping and val-only checkpoint selection.

    NOTE the signature: there is no `test_loader` parameter, and there is no
    way to pass one. That is deliberate -- it makes test-set selection unexpressible.

    Epochs and learning rates come from qmed.config via `epochs_for(task)` and
    `lr_for(task)`; no caller may override them.
    """
    epochs = epochs_for(task, smoke)
    lr, lr_min = lr_for(task)
    n_classes = 2 if task == "binary" else 8

    model = model.to(device)
    criterion = nn.CrossEntropyLoss(
        weight=class_weights.to(device) if class_weights is not None else None
    )
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=CFG.train.WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=lr_min)

    history = {"train_loss": [], "train_acc": [], "val_acc": [], "val_f1": [], "val_auc": []}
    best_f1, best_state, since_best = -1.0, None, 0
    t_start = time.time()

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        tl, ta = train_epoch(model, train_loader, optimizer, criterion, device)
        vm = evaluate(model, val_loader, n_classes, device)   # VALIDATION only
        scheduler.step()

        history["train_loss"].append(tl)
        history["train_acc"].append(ta)
        history["val_acc"].append(vm["accuracy"])
        history["val_f1"].append(vm["f1"])
        history["val_auc"].append(vm["auc"])

        if vm["f1"] > best_f1:
            best_f1 = vm["f1"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            since_best = 0
        else:
            since_best += 1

        if verbose and (epoch == 1 or epoch % 10 == 0 or epoch == epochs):
            print(f"  epoch {epoch:3d}/{epochs}  train_acc={ta:.4f}  "
                  f"val_f1={vm['f1']:.4f}  val_auc={vm['auc']:.4f}  "
                  f"({time.time()-t0:.1f}s)", flush=True)

        if checkpoint_path and CFG.train.CHECKPOINT_EVERY_N_EPOCHS > 0 \
                and epoch % CFG.train.CHECKPOINT_EVERY_N_EPOCHS == 0:
            torch.save({"epoch": epoch, "model": model.state_dict(),
                        "best_f1": best_f1, "best_state": best_state},
                       checkpoint_path)

        if not smoke and epoch >= CFG.train.EARLY_STOP_MIN_EPOCHS \
                and since_best >= CFG.train.EARLY_STOP_PATIENCE:
            if verbose:
                print(f"  early stop at epoch {epoch} (best val_f1={best_f1:.4f})")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    history["epochs_run"] = epoch
    history["best_val_f1"] = best_f1
    history["train_wall_s"] = time.time() - t_start
    return history

# ==================================================================
# === Inlined from src/load_g5k.py ===
# ==================================================================
"""
load_g5k.py -- reuse the trained Grid'5000 models instead of retraining.

The headline binary and multiclass models from the Grid'5000 run were saved as
state_dicts. Their tensor shapes are IDENTICAL to the rebuilt `HybridIDS`
factory in `qmed/models.py`; only the module NAMES differ:

    Grid'5000                    rebuilt factory
    -----------------------      -----------------------------
    encoder.*                    classical_encoder.*
    quantum_layer.weights        bottleneck.q_weights
    decoder.*                    classical_decoder.*

So a key remap is all that stands between the archived weights and a working
model. This matters more than it looks: every INFERENCE-ONLY experiment
can then be produced on a laptop, with no cluster time:

    realistic + adversarial robustness
    finite-shot / noisy simulation
    inference latency + energy
    SHAP/LIME fidelity + stability
    PCA inverse-mapping validation
    per-class tables at any granularity

Only experiments that require NEW TRAINING (multi-seed, matched controls,
ablation, qubit/depth sweep, quantum baselines, split protocols, hierarchical
multiclass) actually need Grid'5000.
"""


import os
import re
from typing import Dict, Optional, Tuple

import numpy as np
import torch

# Key rewrites, applied in order.
_KEY_MAP = (
    (r"^encoder\.",            "classical_encoder."),
    (r"^decoder\.",            "classical_decoder."),
    (r"^quantum_layer\.weights$", "bottleneck.q_weights"),
)


def remap_state_dict(sd: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
    """Rewrite Grid'5000 module names to the rebuilt factory's names."""
    out = {}
    for k, v in sd.items():
        nk = k
        for pat, rep in _KEY_MAP:
            nk = re.sub(pat, rep, nk)
        out[nk] = v
    return out


def load_hybrid(task: str, weights_path: str, model=None, strict: bool = True):
    """Load an archived Grid'5000 hybrid model into the rebuilt factory.

    Parameters
    ----------
    task : 'binary' | 'multiclass'
    weights_path : path to `hybrid_qcnn_{binary,multiclass}*.pt`
    model : optional pre-built HybridIDS; built from the factory if omitted.
    """
    if model is None:
        from qmed.models import build_model
        model = build_model(task, "full")

    raw = torch.load(weights_path, map_location="cpu", weights_only=False)
    if isinstance(raw, dict) and "state_dict" in raw:
        raw = raw["state_dict"]
    if isinstance(raw, dict) and "model" in raw and isinstance(raw["model"], dict):
        raw = raw["model"]

    mapped = remap_state_dict(raw)
    missing, unexpected = model.load_state_dict(mapped, strict=False)

    if strict and (missing or unexpected):
        raise RuntimeError(
            f"state_dict mismatch loading {weights_path}\n"
            f"  missing:    {list(missing)}\n"
            f"  unexpected: {list(unexpected)}"
        )
    model.eval()
    return model, {"missing": list(missing), "unexpected": list(unexpected)}


def load_preprocessors(artifact_dir: str) -> Dict:
    """Load the fitted scaler / PCA / label encoder from the archive.

    These are what make the archived model usable on new inputs, and what let
    the robustness perturbations be applied in the ORIGINAL 25-feature space: the archive
    stores X_scaled, and `scaler.inverse_transform` recovers raw feature units.
    """
    import joblib
    out = {}
    for key, fname in (("scaler", "scaler.joblib"),
                       ("pca_binary", "pca_binary.joblib"),
                       ("pca_multi", "pca_multi.joblib"),
                       ("label_encoder", "multiclass_encoder.joblib")):
        p = os.path.join(artifact_dir, fname)
        if os.path.exists(p):
            out[key] = joblib.load(p)
    return out


def load_prepared_data(artifact_dir: str) -> Dict[str, np.ndarray]:
    """Load the archived prepared_data.npz as a plain dict."""
    d = np.load(os.path.join(artifact_dir, "prepared_data.npz"), allow_pickle=True)
    return {k: d[k] for k in d.files}


def recover_raw_features(artifact_dir: str) -> Tuple[np.ndarray, list]:
    """Invert the StandardScaler to recover the ORIGINAL 25-feature values.

    Required by the robustness experiment: Gaussian noise in PCA space is not
    realistic, whereas packet loss, timing jitter, missing features and
    protocol variation are only meaningful in raw feature units. The archive makes this possible without the source CSVs.

    Returns (X_raw [50000, 25], feature_names).
    """
    data = load_prepared_data(artifact_dir)
    pre = load_preprocessors(artifact_dir)
    if "scaler" not in pre:
        raise FileNotFoundError("scaler.joblib absent; cannot invert scaling")
    X_raw = pre["scaler"].inverse_transform(data["X_scaled"])
    names = [str(s) for s in data["kept_names"]]
    return X_raw, names


# ---------------------------------------------------------------------------
# What the archive can and cannot answer
# ---------------------------------------------------------------------------
RECOVERABLE_WITHOUT_CLUSTER = {
    "realistic + adversarial robustness": "trained model + raw features",
    "finite-shot / noisy simulation":     "trained model, inference only",
    "inference latency + energy":         "trained model, inference only",
    "SHAP/LIME fidelity + stability":     "trained model + test data",
    "PCA inverse-map validation":         "trained model + PCA + raw names",
    "per-class tables (any granularity)":     "saved predictions + labels",
    "noise robustness, FULL detail":          "re-run inference over sigmas",
    "baseline per-sample predictions":        "refit from saved best_params",
}

NEEDS_CLUSTER = {
    "multi-seed headline":       "new training, 5 seeds",
    "parameter-matched controls": "new training",
    "ablation":             "new training (ablation weights were never saved)",
    "qubit/depth sweep":         "new training",
    "quantum baselines":         "new training",
    "session/device/chrono splits": "new training + source CSVs (not archived)",
    "hierarchical multiclass":   "new training",
}

NOT_RECOVERABLE_AT_ALL = {
    "flow provenance (source capture file, timestamps, device id)":
        "never saved; the split protocols REQUIRE re-downloading the CICIoMT2024 CSVs",
    "ablation model weights / curves / predictions":
        "never saved; these must be retrained",
    "CV per-fold models and predictions": "never saved",
    "SHAP / LIME values from the original run": "only PNGs were kept",
}


def print_recovery_map() -> None:
    for title, d in (("RECOVERABLE ON A LAPTOP", RECOVERABLE_WITHOUT_CLUSTER),
                     ("NEEDS GRID'5000", NEEDS_CLUSTER),
                     ("GONE FOR GOOD", NOT_RECOVERABLE_AT_ALL)):
        print(f"\n{title}\n{'-'*len(title)}")
        for k, v in d.items():
            print(f"  {k:52s} {v}")

# ==================================================================
# === Inlined from src/audit.py ===
# ==================================================================
"""
audit.py -- SHA-256 hash-chained audit log, with a measurement harness.

Section 6.3 of the paper asserts tamper evidence, chronological integrity,
O(T) verification and sub-millisecond overhead. This module turns each
assertion into something measurable, and provides two comparison points: a
plain append-only log (what a conventional IDS gives you) and a Merkle tree
(what you would use if you needed O(log T) inclusion proofs).

Requires no model and no dataset.
"""

import hashlib
import json
import random
import time
from typing import Callable, Dict, List, Optional, Tuple


# =============================================================================
# The logs
# =============================================================================
class HashChainLog:
    """The audit log of section 6.3 of the paper.

    Record i commits to (hash_{i-1}, payload_i), so the chain head
    authenticates the entire history: altering any record invalidates every
    subsequent link.
    """

    GENESIS = "0" * 64

    def __init__(self):
        self.records: List[Dict] = []

    @staticmethod
    def digest(prev_hash: str, payload: Dict) -> str:
        return hashlib.sha256(
            (prev_hash + json.dumps(payload, sort_keys=True)).encode()
        ).hexdigest()

    def append(self, payload: Dict) -> str:
        prev = self.records[-1]["hash"] if self.records else self.GENESIS
        h = self.digest(prev, payload)
        self.records.append({"prev": prev, "payload": payload, "hash": h})
        return h

    @property
    def head(self) -> str:
        return self.records[-1]["hash"] if self.records else self.GENESIS

    def verify(self) -> Tuple[bool, Optional[int]]:
        """Return (is_valid, index_of_first_corruption). O(T) by construction."""
        prev = self.GENESIS
        for i, rec in enumerate(self.records):
            if rec["prev"] != prev:
                return False, i
            if self.digest(rec["prev"], rec["payload"]) != rec["hash"]:
                return False, i
            prev = rec["hash"]
        return True, None


class AppendOnlyLog:
    """Control: a conventional IDS log. No linkage, so a modified record is
    indistinguishable from a genuine one -- detection rate is identically 0."""

    def __init__(self):
        self.records: List[Dict] = []

    def append(self, payload: Dict) -> None:
        self.records.append({"payload": payload})

    def verify(self) -> Tuple[bool, Optional[int]]:
        return True, None


class MerkleLog:
    """Alternative: a Merkle tree over the same payloads.

    Same tamper evidence, but inclusion proofs are O(log T) rather than O(T).
    The trade-off is that the whole tree must be rebuilt (or incrementally
    maintained) rather than appended to in constant time -- which is why the
    paper's single-node, low-latency setting favours the chain.
    """

    def __init__(self):
        self.leaves: List[str] = []

    def append(self, payload: Dict) -> None:
        self.leaves.append(
            hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest())

    def root(self) -> str:
        if not self.leaves:
            return "0" * 64
        level = list(self.leaves)
        while len(level) > 1:
            if len(level) % 2:
                level.append(level[-1])
            level = [hashlib.sha256((level[i] + level[i + 1]).encode()).hexdigest()
                     for i in range(0, len(level), 2)]
        return level[0]

    def proof_length(self) -> int:
        n, d = max(1, len(self.leaves)), 0
        while n > 1:
            n = (n + 1) // 2
            d += 1
        return d


# =============================================================================
# Event construction
# =============================================================================
def make_event(i: int, rng: random.Random, n_classes: int = 8) -> Dict:
    """A realistic IDS decision record.

    Section 6.3 states that the prediction, the SHAP/LIME explanatory features
    and a SHA-256 digest of the input are all committed, so the measured
    overhead must be for a payload of that size -- not for a toy string.
    """
    return {
        "event_id": i,
        "timestamp": 1.7e9 + i * 0.01,
        "predicted_class": rng.randrange(n_classes),
        "confidence": round(rng.random(), 6),
        "top_features": [f"pca_{rng.randrange(16)}" for _ in range(5)],
        "shap_values": [round(rng.uniform(-1, 1), 6) for _ in range(5)],
        "lime_features": [f"feat_{rng.randrange(25)}" for _ in range(5)],
        "input_sha256": hashlib.sha256(str(i).encode()).hexdigest(),
    }


# =============================================================================
# Tamper harness
# =============================================================================
def tamper_inplace(chain: HashChainLog, mode: str,
                   rng: random.Random) -> Tuple[Optional[int], Callable[[], None]]:
    """Corrupt the chain IN PLACE; return (corrupted_index, undo).

    Deep-copying a 10^5-record chain per trial would dominate runtime and
    measure the copy, not the verification. Mutate and restore instead.
    """
    recs = chain.records
    n = len(recs)

    if mode == "modify":
        i = rng.randrange(n)
        old = recs[i]["payload"]["predicted_class"]
        recs[i]["payload"]["predicted_class"] = old ^ 1        # flip a verdict
        return i, lambda: recs[i]["payload"].__setitem__("predicted_class", old)

    if mode == "delete":
        i = rng.randrange(1, n)
        old_rec = recs.pop(i)
        return i, lambda: recs.insert(i, old_rec)

    if mode == "reorder":
        i = rng.randrange(n - 1)

        def swap():
            recs[i], recs[i + 1] = recs[i + 1], recs[i]
        swap()
        return i, swap

    if mode == "truncate":
        i = rng.randrange(n // 2, n)
        tail = recs[i:]
        del recs[i:]
        return None, lambda: recs.extend(tail)

    raise ValueError(f"unknown tamper mode: {mode}")


def detect_once(chain: HashChainLog, mode: str, rng: random.Random,
                head_before: str) -> Tuple[bool, bool]:
    """One tamper trial. Returns (detected, localised)."""
    i_corrupt, undo = tamper_inplace(chain, mode, rng)
    valid, idx = chain.verify()

    if mode == "truncate":
        # An intact prefix still verifies; truncation is caught by comparing
        # the retained head against the independently published one.
        detected = (not valid) or (chain.head != head_before)
    else:
        detected = not valid

    localised = bool(detected and i_corrupt is not None and idx is not None
                     and idx <= i_corrupt + 1)
    undo()
    return detected, localised


# =============================================================================
# Measurements
# =============================================================================
def measure_overhead(n_records: int, rng: random.Random) -> Dict:
    """Append cost, verification cost and storage for one chain length."""
    events = [make_event(i, rng) for i in range(n_records)]

    t0 = time.perf_counter()
    chain = HashChainLog()
    for e in events:
        chain.append(e)
    t_chain = time.perf_counter() - t0

    t0 = time.perf_counter()
    plain = AppendOnlyLog()
    for e in events:
        plain.append(e)
    t_plain = time.perf_counter() - t0

    t0 = time.perf_counter()
    merkle = MerkleLog()
    for e in events:
        merkle.append(e)
    _ = merkle.root()
    t_merkle = time.perf_counter() - t0

    t0 = time.perf_counter()
    chain.verify()
    t_verify = time.perf_counter() - t0

    size_mb = len(json.dumps(chain.records).encode()) / 1e6

    return {
        "n_records": n_records,
        "us_per_event_chain": t_chain / n_records * 1e6,
        "us_per_event_plain": t_plain / n_records * 1e6,
        "us_per_event_merkle": t_merkle / n_records * 1e6,
        "us_overhead_vs_plain": (t_chain - t_plain) / n_records * 1e6,
        "throughput_events_per_s": n_records / t_chain,
        "verify_total_s": t_verify,
        "us_per_record_verify": t_verify / n_records * 1e6,
        "size_mb": size_mb,
        "mb_per_million_events": size_mb / n_records * 1e6,
        "merkle_proof_length": merkle.proof_length(),
    }


def measure_detection(n_records: int, trials: int, attacks: List[str],
                      rng: random.Random) -> List[Dict]:
    chain = HashChainLog()
    for i in range(n_records):
        chain.append(make_event(i, rng))
    head = chain.head

    rows = []
    for mode in attacks:
        det = loc = 0
        for _ in range(trials):
            d, l = detect_once(chain, mode, rng, head)
            det += d
            loc += l
        # Truncation leaves an intact PREFIX, so verify() returns valid and
        # yields no corruption index: the attack is caught by comparing the
        # retained head, not by walking the chain. Localisation is therefore
        # not applicable rather than 0% -- reporting 0% would read as a failure.
        rows.append({
            "n_records": n_records, "attack": mode, "trials": trials,
            "detection_rate_pct": det / trials * 100,
            "localisation_rate_pct": (None if mode == "truncate"
                                      else loc / trials * 100),
            "localisation_note": ("n/a -- detected by head comparison"
                                  if mode == "truncate" else ""),
            "append_only_detection_pct": 0.0,   # the control, by construction
        })
    return rows


def project_storage(mb_per_million: float, flows_per_s: List[int]) -> List[Dict]:
    """Storage projection at realistic hospital flow rates."""
    rows = []
    for r in flows_per_s:
        per_day = r * 86_400
        rows.append({
            "flows_per_s": r,
            "events_per_day": per_day,
            "gb_per_day": per_day / 1e6 * mb_per_million / 1000,
            "gb_per_year": per_day * 365 / 1e6 * mb_per_million / 1000,
        })
    return rows


# =============================================================================
# Threat model -- what the mechanism does and does not provide
# =============================================================================
THREAT_MODEL = {
    "detects": [
        "silent modification of a past detection verdict or its explanation",
        "deletion of an inconvenient alert",
        "reordering of records to misrepresent the timeline",
        "truncation of the log tail (given an independently retained head)",
    ],
    "does_not_detect": [
        "an adversary who rewrites the whole chain AND replaces the retained "
        "head -- tamper evidence requires the head to be published or held "
        "outside the compromised node",
        "suppression of an event before it is ever logged",
        "a compromised model producing wrong but faithfully logged verdicts",
    ],
    "regulatory_mapping": {
        "HIPAA 164.312(b)": "audit controls -- record and examine activity",
        "HIPAA 164.312(c)(1)": "integrity -- protect from improper alteration",
        "EU MDR Annex I 17.2": "repeatability, reliability and traceability "
                               "of software decisions",
    },
}

# ==================================================================
# === Inlined from src/perturb.py ===
# ==================================================================
"""
perturb.py -- realistic perturbations and adversarial attacks.

Gaussian perturbations in PCA space do not necessarily correspond to realistic
variations encountered in IoMT network traffic. This module therefore
evaluates robustness under more representative network perturbations (packet
loss, timing jitter, missing features, protocol variation) and under
adversarial attacks.

What matters is WHERE the perturbation is applied. Everything in this module
operates on the ORIGINAL 25-feature space, before the scaler and before PCA;
the pipeline is then re-applied end to end. The PCA-space Gaussian sweep of the
paper is retained as `pca_gaussian` and should be labelled as a sanity
baseline, not as the robustness result.

Feature names come from the archived `kept_names`:
    Header_Length, Duration, Rate, fin_flag_number, syn_flag_number,
    rst_flag_number, ...
so the operators below select columns by semantic role rather than by index.
"""

from typing import Dict, List, Optional, Sequence

import numpy as np


# =============================================================================
# Semantic column groups
# =============================================================================
def column_groups(feature_names: Sequence[str]) -> Dict[str, List[int]]:
    """Map each perturbation to the feature columns it may legitimately touch.

    Getting this mapping right is what separates a realistic perturbation from
    noise: packet loss must deflate counts and rates, not flag bits.
    """
    lower = [n.lower() for n in feature_names]

    def match(*keys):
        return [i for i, n in enumerate(lower) if any(k in n for k in keys)]

    return {
        # counters and rates shrink when packets are dropped
        "volume": match("rate", "number", "count", "tot", "sum", "length", "size"),
        # timing-derived features distort under jitter
        "timing": match("duration", "rate", "iat", "time", "interval"),
        # header / payload shape shifts with protocol variation
        "protocol": match("header", "length", "size", "protocol", "proto"),
        # TCP/IP flags an attacker can set freely
        "flags": match("flag", "fin", "syn", "rst", "psh", "ack", "urg", "ece", "cwr"),
        # features an attacker can plausibly control end-to-end
        "attacker_controllable": sorted(set(
            match("rate", "length", "size", "duration", "iat", "interval",
                  "flag", "fin", "syn", "rst", "psh", "ack"))),
    }


# =============================================================================
# Realistic perturbations, in RAW feature units
# =============================================================================
def apply_perturbation(X_raw: np.ndarray, kind: str, severity: float,
                       feature_names: Sequence[str],
                       rng: Optional[np.random.Generator] = None) -> np.ndarray:
    """Return a perturbed copy of the raw 25-feature matrix.

    severity is a fraction in [0, 1] whose meaning is per-perturbation and is
    documented below, so a report can state exactly what was simulated.
    """
    rng = rng or np.random.default_rng(42)
    if severity == 0:
        return X_raw.copy()

    X = X_raw.copy()
    g = column_groups(feature_names)

    if kind == "missing":
        # MCAR sensor/feature dropout at rate `severity`, mean-imputed --
        # the standard failure mode when an IoMT gateway loses a telemetry field.
        mask = rng.random(X.shape) < severity
        col_mean = X.mean(axis=0)
        rows, cols = np.where(mask)
        X[rows, cols] = col_mean[cols]

    elif kind == "jitter":
        # Timing jitter: multiplicative Gaussian noise of relative scale
        # `severity` on duration / rate / inter-arrival features.
        c = g["timing"]
        if c:
            X[:, c] *= (1.0 + rng.normal(0.0, severity, size=(len(X), len(c))))

    elif kind == "packet_loss":
        # A fraction `severity` of packets is lost, so counters and rates
        # deflate proportionally; the loss is itself slightly variable per flow.
        c = g["volume"]
        if c:
            keep = 1.0 - severity
            per_flow = np.clip(
                rng.normal(keep, severity * 0.25, size=(len(X), 1)), 0.0, 1.0)
            X[:, c] *= per_flow

    elif kind == "protocol_var":
        # Protocol variation: header and payload size distributions shift
        # (different client stack, MQTT version, or MTU).
        c = g["protocol"]
        if c:
            X[:, c] *= (1.0 + rng.normal(0.0, severity, size=(len(X), len(c))))

    elif kind == "pca_gaussian":
        raise ValueError("pca_gaussian operates in PCA space -- "
                         "use apply_pca_gaussian(), and label it a sanity baseline")
    else:
        raise ValueError(f"unknown perturbation: {kind}")

    return X


def apply_pca_gaussian(X_pca: np.ndarray, sigma: float,
                       rng: Optional[np.random.Generator] = None) -> np.ndarray:
    """The perturbation used in Table 11 of the paper.

    Retained ONLY as a labelled sanity baseline: additive Gaussian noise in PCA
    space does not correspond to any physical process in IoMT traffic.
    """
    rng = rng or np.random.default_rng(42)
    if sigma == 0:
        return X_pca.copy()
    return X_pca + rng.normal(0.0, sigma, size=X_pca.shape)


# =============================================================================
# Adversarial attacks
# =============================================================================
def fgsm(model, X_pca: np.ndarray, y: np.ndarray, eps: float) -> np.ndarray:
    """Single-step white-box attack on the differentiable hybrid model."""
    return pgd(model, X_pca, y, eps, steps=1)


def pgd(model, X_pca: np.ndarray, y: np.ndarray, eps: float,
        steps: int = 10, alpha: Optional[float] = None) -> np.ndarray:
    """L-infinity PGD in PCA space, where the model is differentiable.

    Operating in PCA space is a stated limitation: it measures the model's
    local sensitivity, not an attack a network adversary could physically
    mount. `constrained_attack` is the operationally meaningful variant.
    """
    import torch

    alpha = alpha or (eps / max(1, steps)) * 2.5
    lossf = torch.nn.CrossEntropyLoss()
    x0 = torch.tensor(X_pca, dtype=torch.float32)
    x = x0.clone()
    yt = torch.tensor(y, dtype=torch.long)

    for _ in range(steps):
        x = x.detach().requires_grad_(True)
        lossf(model(x), yt).backward()
        x = x + alpha * x.grad.sign()
        x = x0 + (x - x0).clamp(-eps, eps)

    return x.detach().numpy()


def constrained_attack(model, X_raw: np.ndarray, y: np.ndarray,
                       feature_names: Sequence[str], scaler, pca,
                       eps_rel: float, steps: int = 10) -> np.ndarray:
    """Adversarial perturbation restricted to attacker-controllable features.

    This is the only adversarial variant that is operationally meaningful for a
    network IDS: an attacker can change packet sizes, rates, timings and TCP
    flags, but cannot rewrite features the defender derives independently.
    Perturbations are bounded RELATIVE to each feature's scale, and applied in
    raw units before the scaler, so the result stays a realisable flow.

    Uses a gradient-free coordinate search on the restricted feature set,
    because the raw-space path (scaler + PCA) is not differentiated end to end.
    """
    import torch

    g = column_groups(feature_names)
    cols = g["attacker_controllable"]
    if not cols:
        return X_raw.copy()

    X = X_raw.copy()
    col_scale = np.abs(X[:, cols]).mean(axis=0) + 1e-12

    def loss_of(Xr):
        Xp = pca.transform(scaler.transform(Xr))
        with torch.no_grad():
            logits = model(torch.tensor(Xp, dtype=torch.float32))
            return torch.nn.functional.cross_entropy(
                logits, torch.tensor(y, dtype=torch.long), reduction="none").numpy()

    best = loss_of(X)
    for _ in range(steps):
        for j, c in enumerate(cols):
            for sign in (+1.0, -1.0):
                trial = X.copy()
                trial[:, c] += sign * (eps_rel / steps) * col_scale[j]
                l = loss_of(trial)
                better = l > best
                if better.any():
                    X[better] = trial[better]
                    best[better] = l[better]
    return X


def transfer_attack(surrogate, target_model, X_pca: np.ndarray, y: np.ndarray,
                    eps: float, steps: int = 10) -> np.ndarray:
    """Black-box transfer: craft on a surrogate, evaluate on the target.

    The realistic threat model for a deployed IDS, where the attacker cannot
    query gradients of the fielded detector.
    """
    return pgd(surrogate, X_pca, y, eps, steps=steps)

# ==================================================================
# === Inlined from src/stats.py ===
# ==================================================================
"""
stats.py -- multi-seed aggregation and paired significance testing.

Repeating experiments over multiple random seeds and reporting mean
performance together with standard deviations or confidence intervals
strengthens the empirical evidence over single runs. The reported spread of a 2-fold CV
(+/- 0.0002) is not a variance estimate. This module provides
multi-seed aggregation, and the paired tests that the saved per-sample
predictions make possible.

Test selection:
  * McNemar          -- paired, per-sample, for comparing two classifiers on the
                        SAME test set. The correct test for binary agreement;
                        used with the exact binomial when discordant pairs are few.
  * Wilcoxon         -- paired across seeds, when comparing two methods over a
                        common seed set (n = 5 here, so exact and low-powered:
                        report the effect size alongside).
  * Friedman         -- omnibus across >2 methods over a common seed set.
  * Bootstrap CI     -- for any metric, from the saved per-sample predictions.
  * Holm-Bonferroni  -- family-wise correction across the baseline family.

A note worth reporting alongside the results: with 5 seeds, Wilcoxon's minimum
attainable p-value is 0.0625, so no pairwise seed-level comparison can reach
p < 0.05. Paired McNemar on per-sample predictions is therefore the test that
carries the statistical weight; the seed-level tests describe stability.
"""

from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np


# =============================================================================
# Aggregation
# =============================================================================
def summarise(values: Sequence[float], confidence: float = 0.95) -> Dict:
    """mean, std, and a t-based CI appropriate for a small seed count."""
    from scipy import stats as sps

    v = np.asarray(list(values), dtype=float)
    v = v[~np.isnan(v)]
    n = len(v)
    if n == 0:
        return {"n": 0, "mean": np.nan, "std": np.nan, "ci_low": np.nan,
                "ci_high": np.nan}
    mean, std = float(v.mean()), float(v.std(ddof=1)) if n > 1 else 0.0
    if n > 1:
        half = sps.t.ppf(0.5 + confidence / 2, n - 1) * std / np.sqrt(n)
    else:
        half = np.nan
    return {"n": n, "mean": mean, "std": std,
            "ci_low": mean - half, "ci_high": mean + half}


def summarise_runs(records: Iterable[Dict], metric: str,
                   group_keys: Sequence[str] = ("variant", "task")) -> List[Dict]:
    """Group saved run manifests and summarise one metric across seeds."""
    groups: Dict[Tuple, List[float]] = {}
    for r in records:
        key = tuple(r.get(k) for k in group_keys)
        val = (r.get("metrics") or {}).get(metric)
        if val is not None:
            groups.setdefault(key, []).append(float(val))

    out = []
    for key, vals in sorted(groups.items(), key=lambda kv: str(kv[0])):
        row = dict(zip(group_keys, key))
        row["metric"] = metric
        row.update(summarise(vals))
        row["values"] = vals
        out.append(row)
    return out


def format_mean_std(row: Dict, dp: int = 4) -> str:
    """Render 'mean +/- std' for a LaTeX table cell."""
    if row["n"] <= 1:
        return f"{row['mean']:.{dp}f}"
    return f"{row['mean']:.{dp}f} ± {row['std']:.{dp}f}"


# =============================================================================
# Paired tests
# =============================================================================
def mcnemar(y_true, pred_a, pred_b, exact: Optional[bool] = None) -> Dict:
    """Paired McNemar test on per-sample predictions.

    b = A right / B wrong, c = A wrong / B right. Uses the exact binomial when
    b + c < 25, otherwise the continuity-corrected chi-square.
    """
    from scipy import stats as sps

    y = np.asarray(y_true)
    a_ok = np.asarray(pred_a) == y
    b_ok = np.asarray(pred_b) == y
    b = int(np.sum(a_ok & ~b_ok))
    c = int(np.sum(~a_ok & b_ok))
    n_disc = b + c

    if exact is None:
        exact = n_disc < 25

    if n_disc == 0:
        return {"b": b, "c": c, "n_discordant": 0, "statistic": 0.0,
                "p_value": 1.0, "test": "degenerate (identical predictions)"}

    if exact:
        p = float(sps.binomtest(min(b, c), n_disc, 0.5).pvalue)
        stat, test = float(min(b, c)), "exact binomial"
    else:
        stat = (abs(b - c) - 1.0) ** 2 / n_disc
        p = float(sps.chi2.sf(stat, df=1))
        test = "chi-square (continuity-corrected)"

    return {"b": b, "c": c, "n_discordant": n_disc, "statistic": float(stat),
            "p_value": p, "test": test,
            "acc_a": float(a_ok.mean()), "acc_b": float(b_ok.mean())}


def wilcoxon_seeds(a: Sequence[float], b: Sequence[float],
                   alternative: str = "two-sided") -> Dict:
    """Paired Wilcoxon across seeds, with the power caveat made explicit."""
    from scipy import stats as sps

    a, b = np.asarray(a, float), np.asarray(b, float)
    n = len(a)
    if n < 3 or np.allclose(a, b):
        return {"n": n, "statistic": np.nan, "p_value": 1.0,
                "note": "too few seeds or identical values"}
    stat, p = sps.wilcoxon(a, b, zero_method="wilcox", alternative=alternative)
    return {"n": n, "statistic": float(stat), "p_value": float(p),
            "mean_a": float(a.mean()), "mean_b": float(b.mean()),
            "min_attainable_p": 2.0 ** -(n - 1) * 2,
            "note": (f"with n={n} seeds the smallest attainable two-sided "
                     f"p-value is {2.0 ** -(n - 1) * 2:.4f}")}


def friedman(*groups: Sequence[float]) -> Dict:
    """Omnibus test across >2 methods measured on a common seed set."""
    from scipy import stats as sps

    if len(groups) < 3:
        return {"statistic": np.nan, "p_value": np.nan,
                "note": "Friedman needs at least 3 groups"}
    stat, p = sps.friedmanchisquare(*groups)
    return {"statistic": float(stat), "p_value": float(p), "k": len(groups),
            "n": len(groups[0])}


def holm_bonferroni(pvalues: Sequence[float],
                    labels: Optional[Sequence[str]] = None,
                    alpha: float = 0.05) -> List[Dict]:
    """Holm-Bonferroni step-down correction across a family of comparisons."""
    p = np.asarray(list(pvalues), float)
    m = len(p)
    labels = list(labels) if labels is not None else [str(i) for i in range(m)]

    order = np.argsort(p)
    adjusted = np.empty(m, float)
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (m - rank) * p[i])
        adjusted[i] = min(1.0, running)

    return [{"label": labels[i], "p_raw": float(p[i]),
             "p_holm": float(adjusted[i]), "reject": bool(adjusted[i] < alpha)}
            for i in range(m)]


# =============================================================================
# Bootstrap
# =============================================================================
def bootstrap_metric(y_true, y_pred, metric_fn, n_boot: int = 2000,
                     seed: int = 42, confidence: float = 0.95) -> Dict:
    """Percentile bootstrap CI for any metric, from saved predictions.

    Because per-sample predictions are persisted for every run, this can be
    added to a report at any time without re-running an experiment.
    """
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    n = len(y_true)

    stats = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, n)
        stats[i] = metric_fn(y_true[idx], y_pred[idx])

    lo = float(np.percentile(stats, (1 - confidence) / 2 * 100))
    hi = float(np.percentile(stats, (1 + confidence) / 2 * 100))
    return {"point": float(metric_fn(y_true, y_pred)),
            "ci_low": lo, "ci_high": hi, "n_boot": n_boot}


# =============================================================================
# Convenience: full pairwise comparison of a method against a baseline family
# =============================================================================
def compare_against_family(y_true, pred_ours, family: Dict[str, np.ndarray],
                           alpha: float = 0.05) -> List[Dict]:
    """Paired McNemar of one method against each baseline, Holm-corrected."""
    rows, pvals, labels = [], [], []
    for name, pred in family.items():
        r = mcnemar(y_true, pred_ours, pred)
        r["baseline"] = name
        rows.append(r)
        pvals.append(r["p_value"])
        labels.append(name)

    for row, adj in zip(rows, holm_bonferroni(pvals, labels, alpha)):
        row["p_holm"] = adj["p_holm"]
        row["significant_holm"] = adj["reject"]
    return rows

# ==================================================================
# === Inlined from src/plots.py ===
# ==================================================================
"""
plots.py -- every curve the paper needs, quantitative and qualitative.

Design rule, and the reason this module exists:

    A FIGURE IS NOT DATA. Every plotting function here ALSO returns the arrays
    it drew, and every caller passes them to `save_curve()`, so changing an
    axis, adding a confidence band, or re-plotting at a different DPI never
    requires recomputing the experiment.

Coverage:

  Quantitative (model performance)
    training_curves          loss / accuracy / F1 / AUC per epoch, +/- seed band
    roc_curves               one-vs-rest ROC, per class and micro/macro
    pr_curves                precision-recall (more informative under 66.8% DDoS)
    calibration_curve        reliability diagram + Brier score
    confusion_matrix         counts and row-normalised
    per_class_bars           precision/recall/F1 per class with support
    metric_vs_sweep          any metric against any swept parameter, +/- CI
    seed_boxplot             distribution across seeds
    pareto_scatter           accuracy vs footprint vs energy

  Qualitative (data and explanation)
    class_distribution       class balance, train vs test
    pca_variance             explained variance and cumulative curve
    tsne_projection          2-D embedding coloured by class
    correlation_heatmap      feature correlation structure
    shap_beeswarm_data       instance-level SHAP distributions
    feature_importance_bars  SHAP vs LIME side by side
"""

from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np

# Matplotlib is configured headless by the notebook bootstrap (Agg on OAR).
import matplotlib.pyplot as plt

DPI = 300
FIGSIZE = (7.0, 4.2)


def _finish(fig, path: Optional[str], show: bool = False):
    fig.tight_layout()
    if path:
        fig.savefig(path, dpi=DPI, bbox_inches="tight")
        print(f"[plot] {path}")
    if show:
        plt.show()
    else:
        plt.close(fig)
    return fig


# =============================================================================
# QUANTITATIVE
# =============================================================================
def training_curves(histories: Dict[str, Dict], path: Optional[str] = None,
                    metrics: Sequence[str] = ("train_loss", "val_f1"),
                    ) -> Dict[str, np.ndarray]:
    """Learning curves, optionally averaged over seeds with a +/-1 std band.

    `histories` maps a label -> history dict (or label -> list of history dicts
    for a multi-seed run). Returns every array drawn, for `save_curve`.
    """
    fig, axes = plt.subplots(1, len(metrics), figsize=(6.2 * len(metrics), 4.0))
    axes = np.atleast_1d(axes)
    out: Dict[str, np.ndarray] = {}

    for ax, metric in zip(axes, metrics):
        for label, hist in histories.items():
            runs = hist if isinstance(hist, list) else [hist]
            series = [np.asarray(h[metric], float) for h in runs if metric in h]
            if not series:
                continue
            n = min(len(s) for s in series)
            arr = np.stack([s[:n] for s in series])
            mean = arr.mean(0)
            epochs = np.arange(1, n + 1)

            ax.plot(epochs, mean, label=label, linewidth=1.8)
            if arr.shape[0] > 1:
                sd = arr.std(0, ddof=1)
                ax.fill_between(epochs, mean - sd, mean + sd, alpha=0.18)
                out[f"{label}__{metric}__std"] = sd
            out[f"{label}__{metric}__mean"] = mean
            out[f"{label}__{metric}__epochs"] = epochs

        ax.set_xlabel("epoch")
        ax.set_ylabel(metric.replace("_", " "))
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)

    _finish(fig, path)
    return out


def roc_curves(y_true, probs, class_names: Optional[Sequence[str]] = None,
               path: Optional[str] = None) -> Dict[str, np.ndarray]:
    """One-vs-rest ROC with per-class AUC, plus macro average."""
    from sklearn.metrics import roc_curve, auc
    from sklearn.preprocessing import label_binarize

    probs = np.asarray(probs)
    n_cls = probs.shape[1]
    classes = list(range(n_cls))
    names = list(class_names) if class_names else [str(c) for c in classes]

    Y = label_binarize(y_true, classes=classes) if n_cls > 2 else \
        np.c_[1 - np.asarray(y_true), np.asarray(y_true)]

    fig, ax = plt.subplots(figsize=FIGSIZE)
    out, aucs = {}, []
    for i in range(n_cls):
        if Y[:, i].sum() == 0:
            continue
        fpr, tpr, thr = roc_curve(Y[:, i], probs[:, i])
        a = auc(fpr, tpr)
        aucs.append(a)
        ax.plot(fpr, tpr, linewidth=1.4, label=f"{names[i]} (AUC={a:.3f})")
        out[f"fpr__{names[i]}"] = fpr
        out[f"tpr__{names[i]}"] = tpr
        out[f"thresholds__{names[i]}"] = thr

    ax.plot([0, 1], [0, 1], "k--", linewidth=0.8)
    ax.set_xlabel("false positive rate")
    ax.set_ylabel("true positive rate")
    ax.set_title(f"one-vs-rest ROC (macro AUC = {np.mean(aucs):.3f})")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=7, loc="lower right")
    _finish(fig, path)
    out["macro_auc"] = np.array([np.mean(aucs)])
    return out


def pr_curves(y_true, probs, class_names: Optional[Sequence[str]] = None,
              path: Optional[str] = None) -> Dict[str, np.ndarray]:
    """Precision-recall curves.

    Under the 66.8% DDoS prevalence of this test set, PR is more discriminating
    between competing models than ROC -- which is why the paper reports it.
    """
    from sklearn.metrics import precision_recall_curve, average_precision_score
    from sklearn.preprocessing import label_binarize

    probs = np.asarray(probs)
    n_cls = probs.shape[1]
    classes = list(range(n_cls))
    names = list(class_names) if class_names else [str(c) for c in classes]
    Y = label_binarize(y_true, classes=classes) if n_cls > 2 else \
        np.c_[1 - np.asarray(y_true), np.asarray(y_true)]

    fig, ax = plt.subplots(figsize=FIGSIZE)
    out = {}
    for i in range(n_cls):
        if Y[:, i].sum() == 0:
            continue
        p, r, thr = precision_recall_curve(Y[:, i], probs[:, i])
        ap = average_precision_score(Y[:, i], probs[:, i])
        ax.plot(r, p, linewidth=1.4, label=f"{names[i]} (AP={ap:.3f})")
        out[f"precision__{names[i]}"] = p
        out[f"recall__{names[i]}"] = r

    ax.set_xlabel("recall")
    ax.set_ylabel("precision")
    ax.set_title("precision-recall (one-vs-rest)")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=7, loc="lower left")
    _finish(fig, path)
    return out


def calibration_curve(y_true, probs, n_bins: int = 10,
                      path: Optional[str] = None) -> Dict[str, np.ndarray]:
    """Reliability diagram + Brier score.

    Not part of the original paper's figures, but cheap and directly relevant to the
    deployment story: an IDS that is over-confident on rare attack classes is
    operationally worse than its accuracy suggests.
    """
    probs = np.asarray(probs)
    y = np.asarray(y_true)
    conf = probs.max(1)
    pred = probs.argmax(1)
    correct = (pred == y).astype(float)

    edges = np.linspace(0, 1, n_bins + 1)
    idx = np.clip(np.digitize(conf, edges) - 1, 0, n_bins - 1)
    acc_bin = np.full(n_bins, np.nan)
    conf_bin = np.full(n_bins, np.nan)
    cnt = np.zeros(n_bins)
    for b in range(n_bins):
        m = idx == b
        cnt[b] = m.sum()
        if m.any():
            acc_bin[b] = correct[m].mean()
            conf_bin[b] = conf[m].mean()

    valid = ~np.isnan(acc_bin)
    ece = float(np.sum(cnt[valid] / cnt.sum() * np.abs(acc_bin[valid] - conf_bin[valid])))
    onehot = np.zeros_like(probs)
    onehot[np.arange(len(y)), y] = 1.0
    brier = float(np.mean(np.sum((probs - onehot) ** 2, axis=1)))

    fig, ax = plt.subplots(figsize=(5.0, 4.6))
    ax.plot([0, 1], [0, 1], "k--", linewidth=0.8, label="perfect")
    ax.plot(conf_bin[valid], acc_bin[valid], "o-", linewidth=1.6, label="model")
    ax.set_xlabel("mean predicted confidence")
    ax.set_ylabel("empirical accuracy")
    ax.set_title(f"calibration (ECE={ece:.4f}, Brier={brier:.4f})")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    _finish(fig, path)

    return {"bin_confidence": conf_bin, "bin_accuracy": acc_bin,
            "bin_count": cnt, "ece": np.array([ece]), "brier": np.array([brier])}


def confusion_matrix_plot(cm, class_names: Optional[Sequence[str]] = None,
                          normalise: bool = True,
                          path: Optional[str] = None) -> Dict[str, np.ndarray]:
    cm = np.asarray(cm, float)
    names = list(class_names) if class_names else [str(i) for i in range(len(cm))]
    show = cm / np.maximum(cm.sum(1, keepdims=True), 1) if normalise else cm

    fig, ax = plt.subplots(figsize=(6.4, 5.4))
    im = ax.imshow(show, cmap="Blues", vmin=0, vmax=show.max())
    ax.set_xticks(range(len(names)), names, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(names)), names, fontsize=8)
    ax.set_xlabel("predicted")
    ax.set_ylabel("true")
    for i in range(len(names)):
        for j in range(len(names)):
            v = show[i, j]
            ax.text(j, i, f"{v:.2f}" if normalise else f"{int(v)}",
                    ha="center", va="center", fontsize=7,
                    color="white" if v > show.max() * 0.55 else "black")
    fig.colorbar(im, ax=ax, shrink=0.8)
    ax.set_title("confusion matrix" + (" (row-normalised)" if normalise else ""))
    _finish(fig, path)
    return {"confusion_matrix": np.asarray(cm),
            "confusion_matrix_normalised": show}


def per_class_bars(report: Dict, path: Optional[str] = None) -> Dict[str, np.ndarray]:
    """Per-class precision/recall/F1 with support annotated.

    Support matters: the paper reports F1 on classes with n=4 and n=15,
    which is why the enlarged test set was introduced.
    """
    names = report["class_names"]
    p = np.asarray(report["precision"], float)
    r = np.asarray(report["recall"], float)
    f = np.asarray(report["f1"], float)
    s = np.asarray(report["support"], int)

    x = np.arange(len(names))
    w = 0.26
    fig, ax = plt.subplots(figsize=(max(7.0, len(names) * 1.05), 4.4))
    ax.bar(x - w, p, w, label="precision")
    ax.bar(x, r, w, label="recall")
    ax.bar(x + w, f, w, label="F1")
    for i, n in enumerate(s):
        ax.text(x[i], 1.02, f"n={n}", ha="center", fontsize=7,
                color="crimson" if n < 30 else "dimgray")
    ax.set_xticks(x, names, rotation=45, ha="right", fontsize=8)
    ax.set_ylim(0, 1.12)
    ax.set_ylabel("score")
    ax.grid(alpha=0.3, axis="y")
    ax.legend(fontsize=8)
    ax.set_title("per-class performance (red support = statistically thin)")
    _finish(fig, path)
    return {"precision": p, "recall": r, "f1": f, "support": s}


def metric_vs_sweep(records: Sequence[Dict], x_key: str, y_key: str,
                    group_key: Optional[str] = None,
                    path: Optional[str] = None,
                    xlabel: Optional[str] = None,
                    ylabel: Optional[str] = None) -> Dict[str, np.ndarray]:
    """Any metric against any swept parameter, with a +/-1 std band over seeds."""
    import collections

    groups = collections.defaultdict(lambda: collections.defaultdict(list))
    for rec in records:
        g = rec.get(group_key, "all") if group_key else "all"
        groups[g][rec[x_key]].append(rec[y_key])

    fig, ax = plt.subplots(figsize=FIGSIZE)
    out = {}
    for g, by_x in groups.items():
        xs = sorted(by_x, key=lambda v: (v is None, v))
        mean = np.array([np.mean(by_x[x]) for x in xs], float)
        sd = np.array([np.std(by_x[x], ddof=1) if len(by_x[x]) > 1 else 0.0
                       for x in xs], float)
        pos = np.arange(len(xs))
        ax.plot(pos, mean, "o-", linewidth=1.7, label=str(g))
        if (sd > 0).any():
            ax.fill_between(pos, mean - sd, mean + sd, alpha=0.18)
        ax.set_xticks(pos, [str(x) for x in xs])
        out[f"{g}__x"] = np.array([str(x) for x in xs])
        out[f"{g}__mean"] = mean
        out[f"{g}__std"] = sd

    ax.set_xlabel(xlabel or x_key)
    ax.set_ylabel(ylabel or y_key)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    _finish(fig, path)
    return out


def seed_boxplot(values_by_group: Dict[str, Sequence[float]],
                 ylabel: str = "score",
                 path: Optional[str] = None) -> Dict[str, np.ndarray]:
    """Across-seed distribution of a metric."""
    labels = list(values_by_group)
    data = [np.asarray(values_by_group[k], float) for k in labels]

    fig, ax = plt.subplots(figsize=(max(6.0, len(labels) * 1.2), 4.2))
    ax.boxplot(data, labels=labels, showmeans=True)
    for i, d in enumerate(data, start=1):
        ax.plot(np.random.default_rng(0).normal(i, 0.04, len(d)), d,
                "o", alpha=0.6, markersize=4)
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.3, axis="y")
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right", fontsize=8)
    _finish(fig, path)
    return {k: np.asarray(v, float) for k, v in values_by_group.items()}


def pareto_scatter(records: Sequence[Dict], x_key: str = "footprint_kb",
                   y_key: str = "accuracy", label_key: str = "model",
                   path: Optional[str] = None) -> Dict[str, np.ndarray]:
    """Accuracy against deployment cost.

    The paper's compactness claim is a Pareto claim, so it should be drawn
    as one rather than asserted in prose.
    """
    xs = np.array([r[x_key] for r in records], float)
    ys = np.array([r[y_key] for r in records], float)
    labels = [str(r[label_key]) for r in records]

    fig, ax = plt.subplots(figsize=(6.4, 4.6))
    ax.scatter(xs, ys, s=60)
    for x, y, l in zip(xs, ys, labels):
        ax.annotate(l, (x, y), fontsize=7,
                    xytext=(4, 4), textcoords="offset points")

    order = np.argsort(xs)
    best, fx, fy = -np.inf, [], []
    for i in order:
        if ys[i] > best:
            best = ys[i]
            fx.append(xs[i])
            fy.append(ys[i])
    ax.step(fx, fy, where="post", linestyle="--", alpha=0.6,
            label="Pareto frontier")

    ax.set_xscale("log")
    ax.set_xlabel(f"{x_key} (log)")
    ax.set_ylabel(y_key)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    _finish(fig, path)
    return {"x": xs, "y": ys, "labels": np.array(labels),
            "frontier_x": np.array(fx), "frontier_y": np.array(fy)}


# =============================================================================
# QUALITATIVE
# =============================================================================
def class_distribution(y_by_split: Dict[str, np.ndarray],
                       class_names: Optional[Sequence[str]] = None,
                       path: Optional[str] = None) -> Dict[str, np.ndarray]:
    names = list(class_names) if class_names else None
    n_cls = int(max(arr.max() for arr in y_by_split.values())) + 1
    names = names or [str(i) for i in range(n_cls)]

    x = np.arange(n_cls)
    w = 0.8 / len(y_by_split)
    fig, ax = plt.subplots(figsize=(max(7.0, n_cls * 1.05), 4.2))
    out = {}
    for k, (split, y) in enumerate(y_by_split.items()):
        counts = np.bincount(np.asarray(y), minlength=n_cls)
        ax.bar(x + k * w - 0.4 + w / 2, counts, w, label=split)
        out[f"counts__{split}"] = counts

    ax.set_yscale("log")
    ax.set_xticks(x, names, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("count (log)")
    ax.set_title("class distribution by split")
    ax.grid(alpha=0.3, axis="y")
    ax.legend(fontsize=8)
    _finish(fig, path)
    return out


def pca_variance(explained_ratio, n_keep: Optional[int] = None,
                 path: Optional[str] = None) -> Dict[str, np.ndarray]:
    ev = np.asarray(explained_ratio, float)
    cum = np.cumsum(ev)

    fig, ax = plt.subplots(1, 2, figsize=(11.0, 4.0))
    ax[0].bar(np.arange(1, len(ev) + 1), ev)
    ax[0].set_xlabel("component")
    ax[0].set_ylabel("explained variance ratio")
    ax[0].grid(alpha=0.3, axis="y")

    ax[1].plot(np.arange(1, len(cum) + 1), cum, "o-")
    if n_keep:
        ax[1].axvline(n_keep, linestyle="--", color="crimson")
        ax[1].annotate(f"{n_keep} comps = {cum[n_keep-1]*100:.1f}%",
                       (n_keep, cum[n_keep - 1]), fontsize=8,
                       xytext=(6, -14), textcoords="offset points")
    ax[1].set_xlabel("components")
    ax[1].set_ylabel("cumulative explained variance")
    ax[1].grid(alpha=0.3)
    _finish(fig, path)
    return {"explained_ratio": ev, "cumulative": cum}


def tsne_projection(X, y, class_names: Optional[Sequence[str]] = None,
                    n_sample: int = 3000, seed: int = 42,
                    path: Optional[str] = None) -> Dict[str, np.ndarray]:
    from sklearn.manifold import TSNE

    rng = np.random.default_rng(seed)
    X, y = np.asarray(X), np.asarray(y)
    if len(X) > n_sample:
        idx = rng.choice(len(X), n_sample, replace=False)
        X, y = X[idx], y[idx]

    emb = TSNE(n_components=2, random_state=seed, init="pca",
               perplexity=min(30, max(5, len(X) // 100))).fit_transform(X)

    names = list(class_names) if class_names else \
        [str(c) for c in range(int(y.max()) + 1)]
    fig, ax = plt.subplots(figsize=(6.6, 5.4))
    for c in np.unique(y):
        m = y == c
        ax.scatter(emb[m, 0], emb[m, 1], s=6, alpha=0.6,
                   label=f"{names[int(c)]} (n={m.sum()})")
    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")
    ax.legend(fontsize=7, markerscale=2)
    ax.set_title("t-SNE of the PCA representation")
    _finish(fig, path)
    return {"embedding": emb, "labels": y}


def correlation_heatmap(X, feature_names: Optional[Sequence[str]] = None,
                        path: Optional[str] = None) -> Dict[str, np.ndarray]:
    C = np.corrcoef(np.asarray(X, float).T)
    names = list(feature_names) if feature_names else \
        [str(i) for i in range(C.shape[0])]

    fig, ax = plt.subplots(figsize=(7.6, 6.6))
    im = ax.imshow(C, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_xticks(range(len(names)), names, rotation=90, fontsize=6)
    ax.set_yticks(range(len(names)), names, fontsize=6)
    fig.colorbar(im, ax=ax, shrink=0.8)
    ax.set_title("feature correlation")
    _finish(fig, path)
    return {"correlation": C, "feature_names": np.array(names)}


def feature_importance_bars(importances: Dict[str, Sequence[float]],
                            feature_names: Sequence[str], top_k: int = 15,
                            path: Optional[str] = None) -> Dict[str, np.ndarray]:
    """SHAP vs LIME side by side -- whether the pair adds anything over
    either alone, so they must be drawn comparably."""
    ref = np.asarray(list(importances.values())[0], float)
    order = np.argsort(-np.abs(ref))[:top_k]
    names = [feature_names[i] for i in order]

    x = np.arange(len(order))
    w = 0.8 / len(importances)
    fig, ax = plt.subplots(figsize=(7.6, max(4.0, len(order) * 0.32)))
    out = {}
    for k, (label, vals) in enumerate(importances.items()):
        v = np.asarray(vals, float)[order]
        ax.barh(x + k * w - 0.4 + w / 2, v, w, label=label)
        out[f"importance__{label}"] = v

    ax.set_yticks(x, names, fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel("importance")
    ax.grid(alpha=0.3, axis="x")
    ax.legend(fontsize=8)
    _finish(fig, path)
    out["feature_names"] = np.array(names)
    return out


def shap_beeswarm_data(shap_values, X, feature_names: Sequence[str],
                       top_k: int = 12,
                       path: Optional[str] = None) -> Dict[str, np.ndarray]:
    """Beeswarm drawn from raw arrays, so the values are saved alongside."""
    S = np.asarray(shap_values, float)
    Xv = np.asarray(X, float)
    order = np.argsort(-np.abs(S).mean(0))[:top_k]

    fig, ax = plt.subplots(figsize=(7.2, max(4.0, len(order) * 0.34)))
    rng = np.random.default_rng(0)
    for row, j in enumerate(order):
        v = S[:, j]
        c = Xv[:, j]
        c = (c - c.min()) / (np.ptp(c) + 1e-12)
        ax.scatter(v, row + rng.normal(0, 0.07, len(v)), c=c, cmap="coolwarm",
                   s=7, alpha=0.7)
    ax.set_yticks(range(len(order)), [feature_names[j] for j in order], fontsize=8)
    ax.invert_yaxis()
    ax.axvline(0, color="k", linewidth=0.7)
    ax.set_xlabel("SHAP value")
    _finish(fig, path)
    return {"shap_values": S[:, order],
            "feature_values": Xv[:, order],
            "feature_names": np.array([feature_names[j] for j in order])}

# ==================================================================
# === Inlined from src/baselines.py ===
# ==================================================================
"""
baselines.py -- classical comparators.

Three families of comparators:

  A. PAPER BASELINES (Tables 6/7) -- RF, GB, MLP, KNN, SVM(RBF).
     This module SAVES the fitted models and the per-sample predictions, so
     paired McNemar tests can be computed directly from the saved artefacts.

  B. PARAMETER-MATCHED CONTROLS --
     These isolate whether the observed behaviour is attributable to the
     quantum component rather than to differences in network architecture.
     Implemented in models.py: the SAME encoder/decoder scaffolding with the
     PQC swapped for linear / Fourier / tiny-MLP / identity bottlenecks.

  C. LIGHTWEIGHT DEPLOYABLES --
     Highly optimised lightweight classical models, for comparison with the
     quantum approach. Quantised, pruned and depth-limited models at a
     footprint comparable to the 1,178-parameter binary detector, reported on
     the same accuracy / footprint / energy axes.

Refitting family A from the archived `best_params` takes minutes on CPU and
needs no cluster.
"""

from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np


# =============================================================================
# A. Paper baselines
# =============================================================================
# Grids exactly as reported in Table 8 of the paper (baseline configuration).
PARAM_GRIDS = {
    "Random Forest":     {"n_estimators": [100, 200, 500], "max_depth": [10, 20, None]},
    "Gradient Boosting": {"n_estimators": [100, 200], "max_depth": [3, 5, 7],
                          "learning_rate": [0.01, 0.1]},
    "MLP":               {"hidden_layer_sizes": [(64, 32), (128, 64)],
                          "alpha": [1e-3, 1e-2]},
    "KNN":               {"n_neighbors": [3, 5, 7, 11]},
    "SVM (RBF)":         {"C": [0.1, 1, 10], "gamma": ["scale", "auto"]},
}


def make_estimator(name: str, seed: int = 42, **params):
    from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.neural_network import MLPClassifier
    from sklearn.svm import SVC

    if name == "Random Forest":
        return RandomForestClassifier(random_state=seed, n_jobs=-1, **params)
    if name == "Gradient Boosting":
        return GradientBoostingClassifier(random_state=seed, **params)
    if name == "MLP":
        return MLPClassifier(random_state=seed, max_iter=500, **params)
    if name == "KNN":
        return KNeighborsClassifier(n_jobs=-1, **params)
    if name == "SVM (RBF)":
        # probability=True is REQUIRED: without it there is no predict_proba,
        # hence no AUC, no PR curve and no calibration analysis.
        return SVC(random_state=seed, probability=True, **params)
    raise ValueError(f"unknown baseline: {name}")


def fit_baseline(name: str, X_train, y_train, seed: int = 42,
                 best_params: Optional[Dict] = None, tune: bool = False,
                 cv: int = 3) -> Tuple[object, Dict]:
    """Fit one baseline.

    Pass `best_params` (archived from the original run) to refit directly and
    skip the grid search entirely -- minutes instead of hours, and it reproduces
    the original configuration exactly.
    """
    import time
    t0 = time.time()

    if best_params and not tune:
        est = make_estimator(name, seed, **best_params)
        est.fit(X_train, y_train)
        return est, {"best_params": best_params, "tuned": False,
                     "fit_time_s": time.time() - t0}

    if tune:
        from sklearn.model_selection import GridSearchCV
        gs = GridSearchCV(make_estimator(name, seed), PARAM_GRIDS[name],
                          cv=cv, scoring="f1_weighted", n_jobs=-1)
        gs.fit(X_train, y_train)
        return gs.best_estimator_, {"best_params": gs.best_params_, "tuned": True,
                                    "fit_time_s": time.time() - t0}

    est = make_estimator(name, seed)
    est.fit(X_train, y_train)
    return est, {"best_params": {}, "tuned": False,
                 "fit_time_s": time.time() - t0}


def evaluate_sklearn(est, X_test, y_test, n_classes: int) -> Dict:
    """Metrics + the per-sample arrays that make paired tests possible."""
    from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                                 roc_auc_score, confusion_matrix)

    pred = est.predict(X_test)
    try:
        probs = est.predict_proba(X_test)
    except Exception:
        probs = None

    avg = "binary" if n_classes == 2 else "weighted"
    p, r, f1, _ = precision_recall_fscore_support(y_test, pred, average=avg,
                                                  zero_division=0)
    auc = float("nan")
    if probs is not None:
        try:
            auc = (roc_auc_score(y_test, probs[:, 1]) if n_classes == 2
                   else roc_auc_score(y_test, probs, multi_class="ovr",
                                      average="weighted"))
        except Exception:
            pass

    return {"accuracy": float(accuracy_score(y_test, pred)),
            "precision": float(p), "recall": float(r), "f1": float(f1),
            "auc": float(auc),
            "confusion_matrix": confusion_matrix(y_test, pred).tolist(),
            "_predictions": pred, "_labels": np.asarray(y_test),
            "_probabilities": probs}


def model_footprint_kb(est) -> float:
    """Serialised size -- the deployment-cost axis for the Pareto plot.

    For tree ensembles this is the meaningful number: the quantum model is
    compared against highly optimised lightweight classical models, and a
    500-tree forest is not lightweight however fast it predicts.
    """
    import io
    import joblib
    buf = io.BytesIO()
    joblib.dump(est, buf)
    return buf.getbuffer().nbytes / 1024.0


# =============================================================================
# C. Lightweight classical deployables
# =============================================================================
def make_lightweight(name: str, seed: int = 42, n_classes: int = 8):
    """Classical models chosen to be genuinely small, not merely accurate."""
    from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
    from sklearn.neural_network import MLPClassifier

    if name == "rf20":
        # 20 shallow trees instead of 500 deep ones
        return RandomForestClassifier(n_estimators=20, max_depth=8,
                                      random_state=seed, n_jobs=-1)
    if name == "gbdt_shallow":
        return GradientBoostingClassifier(n_estimators=50, max_depth=3,
                                          random_state=seed)
    if name == "pruned_mlp":
        # Sized to land near the 1,178-parameter budget of the binary detector
        return MLPClassifier(hidden_layer_sizes=(48,), alpha=1e-2,
                             max_iter=500, random_state=seed)
    if name == "int8_mlp":
        # Same topology as pruned_mlp; quantisation is applied post-fit by
        # quantise_mlp_int8() so the footprint, not the fit, is what changes.
        return MLPClassifier(hidden_layer_sizes=(48,), alpha=1e-2,
                             max_iter=500, random_state=seed)
    raise ValueError(f"unknown lightweight baseline: {name}")


def quantise_mlp_int8(est) -> Dict:
    """Post-training int8 quantisation of a fitted sklearn MLP.

    Reports the accuracy cost and the footprint saving. This is a fair
    comparator: a classical model optimised for deployment, not just for
    accuracy.
    """
    import copy

    q = copy.deepcopy(est)
    scales = []
    for i, W in enumerate(q.coefs_):
        s = np.abs(W).max() / 127.0 if np.abs(W).max() > 0 else 1.0
        q.coefs_[i] = np.round(W / s) * s          # simulated int8 round-trip
        scales.append(float(s))

    n_w = int(sum(W.size for W in est.coefs_) +
              sum(b.size for b in est.intercepts_))
    return {"model": q, "n_params": n_w,
            "footprint_kb_fp32": n_w * 4 / 1024.0,
            "footprint_kb_int8": n_w * 1 / 1024.0,
            "scales": scales}


def count_sklearn_params(est) -> int:
    """Parameter count on the same axis as the hybrid model's 1,178."""
    if hasattr(est, "coefs_"):        # MLP
        return int(sum(W.size for W in est.coefs_) +
                   sum(b.size for b in est.intercepts_))
    if hasattr(est, "estimators_"):   # tree ensemble: total node count
        total = 0
        for t in np.ravel(est.estimators_):
            tree = getattr(t, "tree_", None)
            if tree is not None:
                total += int(tree.node_count)
        return total
    if hasattr(est, "support_vectors_"):   # SVM
        return int(est.support_vectors_.size + est.dual_coef_.size)
    if hasattr(est, "_fit_X"):             # KNN stores the training set
        return int(est._fit_X.size)
    return -1

# ==================================================================
# === Inlined from src/xai.py ===
# ==================================================================
"""
xai.py -- explainability, with quantitative validation.

The benefit of combining SHAP and LIME needs to be demonstrated against using a
single explainability method, and the transformation of SHAP and LIME
attributions from PCA space back to the original feature space is a heuristic
that should be presented as such, with experiments on explanation stability
and fidelity.

So this module does three things:

  1. FIDELITY -- deletion and insertion curves, comprehensiveness and
     sufficiency, computed identically for SHAP-only, LIME-only and a fused
     ranking. That is what turns "we used both" into a testable claim.
  2. STABILITY -- repeat-run variance (LIME is stochastic, so this is where a
     combination could genuinely pay off) and a local Lipschitz estimate.
  3. INVERSE-MAP VALIDATION -- the PCA back-projection is treated as a
     heuristic and checked against two references computed directly in the
     original feature space: permutation importance on the full pipeline, and
     KernelSHAP over the whole scaler+PCA+model chain as one black box.

If the fused ranking turns out to be no better than the better single method,
that is a publishable answer and should be reported as such -- under the
a negative result here costs the paper nothing.
"""

from typing import Callable, Dict, List, Optional, Sequence, Tuple

import numpy as np


# =============================================================================
# Prediction wrappers
# =============================================================================
def torch_predict_proba(model) -> Callable[[np.ndarray], np.ndarray]:
    """PCA-space probability function for SHAP/LIME."""
    import torch

    def f(X):
        with torch.no_grad():
            logits = model(torch.tensor(np.asarray(X), dtype=torch.float32))
            return torch.softmax(logits, dim=1).numpy()
    return f


def pipeline_predict_proba(model, scaler, pca) -> Callable[[np.ndarray], np.ndarray]:
    """RAW-feature-space probability function.

    Treating scaler + PCA + model as ONE black box is what lets SHAP attribute
    directly to the original 25 features, with no inverse mapping and therefore
    no heuristic -- the reference against which the heuristic is validated.
    """
    import torch

    def f(X_raw):
        Xp = pca.transform(scaler.transform(np.asarray(X_raw)))
        with torch.no_grad():
            logits = model(torch.tensor(Xp, dtype=torch.float32))
            return torch.softmax(logits, dim=1).numpy()
    return f


# =============================================================================
# Attributions
# =============================================================================
def kernel_shap(predict_fn, X_explain: np.ndarray, X_background: np.ndarray,
                nsamples: int = 100, seed: int = 42) -> np.ndarray:
    """KernelSHAP. Returns (n_samples, n_features) for the predicted class."""
    import shap

    bg = shap.kmeans(X_background, min(50, len(X_background)))
    expl = shap.KernelExplainer(predict_fn, bg)
    vals = expl.shap_values(X_explain, nsamples=nsamples, silent=True)

    if isinstance(vals, list):                    # one array per class
        pred = predict_fn(X_explain).argmax(1)
        return np.stack([vals[c][i] for i, c in enumerate(pred)])
    if np.asarray(vals).ndim == 3:                # (n, f, classes)
        vals = np.asarray(vals)
        pred = predict_fn(X_explain).argmax(1)
        return np.stack([vals[i, :, c] for i, c in enumerate(pred)])
    return np.asarray(vals)


def lime_attributions(predict_fn, X_explain: np.ndarray, X_train: np.ndarray,
                      feature_names: Sequence[str], num_samples: int = 1000,
                      seed: int = 42) -> np.ndarray:
    """LIME local attributions, aligned to the same (n_samples, n_features)."""
    from lime.lime_tabular import LimeTabularExplainer

    expl = LimeTabularExplainer(X_train, feature_names=list(feature_names),
                                discretize_continuous=False,
                                random_state=seed, mode="classification")
    n_f = X_explain.shape[1]
    out = np.zeros((len(X_explain), n_f))
    for i, x in enumerate(X_explain):
        label = int(predict_fn(x.reshape(1, -1)).argmax())
        e = expl.explain_instance(x, predict_fn, num_features=n_f,
                                  labels=(label,), num_samples=num_samples)
        for j, w in e.as_map()[label]:
            out[i, j] = w
    return out


def global_importance(attr: np.ndarray) -> np.ndarray:
    """Mean |attribution| per feature -- the global ranking."""
    return np.abs(np.asarray(attr)).mean(axis=0)


def fuse_rankings(*importances: np.ndarray, method: str = "rank_mean") -> np.ndarray:
    """Combine SHAP and LIME into one ranking.

    `rank_mean` averages the per-method ranks, which is scale-free -- SHAP and
    LIME attributions are not on a common scale, so averaging raw magnitudes
    would silently let whichever method has larger values dominate.
    """
    if method == "rank_mean":
        ranks = [np.argsort(np.argsort(-np.abs(v))) for v in importances]
        return -np.mean(ranks, axis=0)
    if method == "zscore_mean":
        zs = [(np.abs(v) - np.abs(v).mean()) / (np.abs(v).std() + 1e-12)
              for v in importances]
        return np.mean(zs, axis=0)
    raise ValueError(method)


# =============================================================================
# 1. FIDELITY
# =============================================================================
def deletion_curve(predict_fn, X: np.ndarray, importance: np.ndarray,
                   baseline: Optional[np.ndarray] = None,
                   steps: Optional[int] = None) -> Dict[str, np.ndarray]:
    """Remove features most-important-first; a faithful ranking drops fastest.

    Returns the full curve, not just its AUC, so it can be plotted.
    """
    X = np.asarray(X, float)
    base = baseline if baseline is not None else X.mean(0)
    order = np.argsort(-np.abs(importance))
    steps = steps or len(order)

    p0 = predict_fn(X)
    cls = p0.argmax(1)
    conf = [p0[np.arange(len(X)), cls].mean()]

    Xw = X.copy()
    for k in range(steps):
        Xw[:, order[k]] = base[order[k]]
        p = predict_fn(Xw)
        conf.append(p[np.arange(len(X)), cls].mean())

    conf = np.asarray(conf)
    return {"n_removed": np.arange(len(conf)), "confidence": conf,
            "auc": np.array([float(np.trapezoid(conf) / (len(conf) - 1))])}


def insertion_curve(predict_fn, X: np.ndarray, importance: np.ndarray,
                    baseline: Optional[np.ndarray] = None,
                    steps: Optional[int] = None) -> Dict[str, np.ndarray]:
    """Add features most-important-first, starting from the baseline."""
    X = np.asarray(X, float)
    base = baseline if baseline is not None else X.mean(0)
    order = np.argsort(-np.abs(importance))
    steps = steps or len(order)

    cls = predict_fn(X).argmax(1)
    Xw = np.tile(base, (len(X), 1))
    conf = [predict_fn(Xw)[np.arange(len(X)), cls].mean()]
    for k in range(steps):
        Xw[:, order[k]] = X[:, order[k]]
        conf.append(predict_fn(Xw)[np.arange(len(X)), cls].mean())

    conf = np.asarray(conf)
    return {"n_inserted": np.arange(len(conf)), "confidence": conf,
            "auc": np.array([float(np.trapezoid(conf) / (len(conf) - 1))])}


def comprehensiveness_sufficiency(predict_fn, X: np.ndarray,
                                  importance: np.ndarray,
                                  k: int = 5) -> Dict[str, float]:
    """Comprehensiveness: confidence drop when the top-k are removed.
    Sufficiency: confidence retained when ONLY the top-k are kept.
    Higher comprehensiveness and lower sufficiency-gap mean a more faithful
    explanation."""
    X = np.asarray(X, float)
    base = X.mean(0)
    order = np.argsort(-np.abs(importance))[:k]

    p0 = predict_fn(X)
    cls = p0.argmax(1)
    c0 = p0[np.arange(len(X)), cls].mean()

    Xr = X.copy()
    Xr[:, order] = base[order]
    c_removed = predict_fn(Xr)[np.arange(len(X)), cls].mean()

    Xk = np.tile(base, (len(X), 1))
    Xk[:, order] = X[:, order]
    c_kept = predict_fn(Xk)[np.arange(len(X)), cls].mean()

    return {"base_confidence": float(c0),
            "comprehensiveness": float(c0 - c_removed),
            "sufficiency": float(c0 - c_kept), "k": k}


# =============================================================================
# 2. STABILITY
# =============================================================================
def stability_repeat(attr_fn: Callable[[int], np.ndarray],
                     repeats: int = 10) -> Dict[str, float]:
    """Variance of the explanation across repeated runs at fixed input.

    LIME samples its neighbourhood stochastically, so this is non-zero for LIME
    and ~zero for KernelSHAP at fixed background -- the most plausible concrete
    benefit of reporting both.
    """
    from scipy.stats import spearmanr

    runs = [global_importance(attr_fn(s)) for s in range(repeats)]
    R = np.stack(runs)

    rhos = [spearmanr(R[i], R[j]).statistic
            for i in range(len(R)) for j in range(i + 1, len(R))]
    top5 = [set(np.argsort(-r)[:5]) for r in R]
    jac = [len(top5[i] & top5[j]) / len(top5[i] | top5[j])
           for i in range(len(top5)) for j in range(i + 1, len(top5))]

    return {"mean_pairwise_spearman": float(np.mean(rhos)),
            "std_pairwise_spearman": float(np.std(rhos)),
            "mean_top5_jaccard": float(np.mean(jac)),
            "mean_cv": float(np.mean(R.std(0) / (np.abs(R).mean(0) + 1e-12))),
            "repeats": repeats}


def local_lipschitz(attr_fn: Callable[[np.ndarray], np.ndarray],
                    X: np.ndarray, eps: float = 0.01,
                    n_probe: int = 5, seed: int = 42) -> Dict[str, float]:
    """Sensitivity of the explanation to a small input perturbation.

    An explanation that swings under a perturbation the MODEL is insensitive to
    is not describing the model. This is the explanation-side analogue of the
    robustness analysis in NB07.
    """
    rng = np.random.default_rng(seed)
    X = np.asarray(X, float)
    a0 = attr_fn(X)

    ratios = []
    for _ in range(n_probe):
        d = rng.normal(0, eps, X.shape)
        a1 = attr_fn(X + d)
        num = np.linalg.norm(a1 - a0, axis=1)
        den = np.linalg.norm(d, axis=1) + 1e-12
        ratios.append(num / den)

    r = np.concatenate(ratios)
    return {"mean_lipschitz": float(r.mean()), "max_lipschitz": float(r.max()),
            "p95_lipschitz": float(np.percentile(r, 95)), "eps": eps}


# =============================================================================
# 3. AGREEMENT and INVERSE-MAP VALIDATION
# =============================================================================
def agreement(imp_a: np.ndarray, imp_b: np.ndarray, k: int = 5) -> Dict[str, float]:
    from scipy.stats import spearmanr, kendalltau

    a, b = np.abs(imp_a), np.abs(imp_b)
    ta, tb = set(np.argsort(-a)[:k]), set(np.argsort(-b)[:k])
    return {"spearman": float(spearmanr(a, b).statistic),
            "kendall_tau": float(kendalltau(a, b).statistic),
            f"jaccard_top{k}": len(ta & tb) / len(ta | tb),
            f"overlap_top{k}": len(ta & tb)}


def pca_inverse_map(importance_pca: np.ndarray, pca) -> np.ndarray:
    """The paper's heuristic: project PCA-space importance back through
    the loading matrix onto the original features.

    It is a heuristic because |importance| is not a signed quantity that the
    linear map is entitled to carry, and because component importances are not
    independent. `validate_inverse_map` measures how much that costs.
    """
    W = np.abs(np.asarray(pca.components_))          # (n_comp, n_features)
    return np.abs(np.asarray(importance_pca)) @ W


def permutation_importance_pipeline(predict_fn, X_raw: np.ndarray, y: np.ndarray,
                                    n_repeats: int = 5, seed: int = 42
                                    ) -> np.ndarray:
    """Model-agnostic importance in the ORIGINAL feature space.

    Requires no PCA algebra at all, so it is a clean reference for validating
    the inverse map.
    """
    rng = np.random.default_rng(seed)
    X_raw = np.asarray(X_raw, float)
    base = (predict_fn(X_raw).argmax(1) == y).mean()

    imp = np.zeros(X_raw.shape[1])
    for j in range(X_raw.shape[1]):
        drops = []
        for _ in range(n_repeats):
            Xp = X_raw.copy()
            rng.shuffle(Xp[:, j])
            drops.append(base - (predict_fn(Xp).argmax(1) == y).mean())
        imp[j] = float(np.mean(drops))
    return imp


def validate_inverse_map(mapped: np.ndarray, reference: np.ndarray,
                         feature_names: Sequence[str], k: int = 5) -> Dict:
    """Compare the heuristic inverse map against a directly-computed reference.

    High agreement validates the heuristic; low agreement means the direct
    rankings should be reported instead and the inverse map dropped. Either
    outcome is informative -- what is not acceptable is leaving it unchecked.
    """
    a = agreement(mapped, reference, k=k)
    top_m = [feature_names[i] for i in np.argsort(-np.abs(mapped))[:k]]
    top_r = [feature_names[i] for i in np.argsort(-np.abs(reference))[:k]]
    verdict = ("validated" if a["spearman"] >= 0.7 else
               "weak -- prefer the direct ranking" if a["spearman"] >= 0.4 else
               "not supported -- report direct rankings and drop the inverse map")
    return {**a, "top_mapped": top_m, "top_reference": top_r, "verdict": verdict}

# ==================================================================
# === Inlined from src/cost.py ===
# ==================================================================
"""
cost.py -- operational cost, latency and energy.

Parameter count alone does not reflect deployment cost for quantum machine
learning systems: practical execution also depends on quantum circuit
evaluation, shot requirements, preprocessing overhead, classical control and
hardware constraints. This module measures those quantities.

The key point: PARAMETER COUNT IS NOT DEPLOYMENT COST. A
1,178-parameter model that needs one circuit execution per inference, at some
shot budget, on hardware with a queue, is not cheaper than a 500-tree forest
just because it serialises to 4.6 KB.

Energy: on Grid5000 the node wattmeters (Kwollect) give measured wall power,
which is a real measurement rather than an estimate. Off-cluster this module
falls back to a CPU-time-based estimate and LABELS it as such -- an estimate
presented as a measurement would be misleading.
"""

import os
import platform
import time
from typing import Dict, List, Optional, Sequence

import numpy as np


# =============================================================================
# Latency and throughput
# =============================================================================
def measure_latency(predict_fn, X: np.ndarray, batch_sizes: Sequence[int] = (1,),
                    repeats: int = 200, warmup: int = 10) -> List[Dict]:
    """Per-sample latency at several batch sizes.

    Median plus p95/p99, because a real-time IDS is sized by its tail, not its
    mean.
    """
    rows = []
    for bs in batch_sizes:
        Xb = X[:bs] if len(X) >= bs else np.repeat(X, bs // len(X) + 1, 0)[:bs]

        for _ in range(warmup):
            predict_fn(Xb)

        times = []
        for _ in range(repeats):
            t0 = time.perf_counter()
            predict_fn(Xb)
            times.append(time.perf_counter() - t0)
        t = np.asarray(times)

        rows.append({
            "batch_size": bs,
            "median_ms_per_batch": float(np.median(t) * 1e3),
            "median_ms_per_sample": float(np.median(t) * 1e3 / bs),
            "p95_ms_per_sample": float(np.percentile(t, 95) * 1e3 / bs),
            "p99_ms_per_sample": float(np.percentile(t, 99) * 1e3 / bs),
            "throughput_samples_per_s": float(bs / np.median(t)),
            "repeats": repeats,
        })
    return rows


def measure_preprocessing_overhead(scaler, pca, X_raw: np.ndarray,
                                   repeats: int = 100) -> Dict:
    """Scaler + PCA cost, timed separately.

    This is excluded from the parameter-count story. It is
    paid on every inference in deployment, so it belongs in the cost table.
    """
    def timeit(fn, n):
        ts = []
        for _ in range(n):
            t0 = time.perf_counter()
            fn()
            ts.append(time.perf_counter() - t0)
        return float(np.median(ts))

    Xs = scaler.transform(X_raw)
    t_scale = timeit(lambda: scaler.transform(X_raw), repeats)
    t_pca = timeit(lambda: pca.transform(Xs), repeats)
    n = len(X_raw)
    return {"n_samples": n,
            "scaler_ms_per_sample": t_scale * 1e3 / n,
            "pca_ms_per_sample": t_pca * 1e3 / n,
            "total_preproc_ms_per_sample": (t_scale + t_pca) * 1e3 / n}


# =============================================================================
# Compute accounting
# =============================================================================
def quantum_cost_model(n_qubits: int, n_layers: int, n_samples: int,
                       shots: Optional[int] = None,
                       epochs: int = 0, batch_size: int = 32) -> Dict:
    """Circuit evaluations, gate counts and shot budget.

    Gate count per layer: n_q RY (re-upload) + n_q Rot + the entangler
    (n_q ring CNOT, plus n_q/2 skip CZ on even layers or n_q/2 cross CNOT on
    odd layers) -- so about 3.5*n_q per layer on average.

    Training with the hardware-compatible parameter-shift rule costs
    2*|theta_q| circuit evaluations per gradient, which is the number that makes
    "compact" and "cheap to train on hardware" two different claims.
    """
    n_params_q = 3 * n_layers * n_qubits
    gates_per_layer = n_qubits * 2 + n_qubits + n_qubits // 2
    total_gates = gates_per_layer * n_layers

    inference_evals = n_samples
    inference_shots = n_samples * shots * n_qubits if shots else None

    steps = (n_samples // batch_size) * epochs if epochs else 0
    train_evals_ps = steps * batch_size * 2 * n_params_q if steps else 0

    return {
        "n_qubits": n_qubits, "n_layers": n_layers,
        "n_quantum_params": n_params_q,
        "gates_per_layer": gates_per_layer, "total_gates": total_gates,
        "circuit_depth_estimate": total_gates,
        "statevector_dim": 2 ** n_qubits,
        "sim_cost_per_sample_ops": n_layers * n_qubits * (2 ** n_qubits),
        "inference_circuit_evals": inference_evals,
        "inference_total_shots": inference_shots,
        "training_steps": steps,
        "training_circuit_evals_parameter_shift": train_evals_ps,
        "note": ("parameter-shift training cost = 2*|theta_q| circuit "
                 "evaluations per gradient step; adjoint/backprop is a "
                 "simulator-only shortcut unavailable on hardware"),
    }


def qpu_cost_projection(circuit_evals: int, shots: int,
                        seconds_per_shot: float = 1e-4,
                        price_per_second_eur: float = 0.0,
                        price_per_task_eur: float = 0.0,
                        tasks: int = 1) -> Dict:
    """What the workload would cost on real NISQ hardware, today.

    Defaults are zero so nothing is invented: supply the numbers from whichever
    provider is being cited, and the report states the arithmetic and its
    assumptions rather than a guess. Queueing and calibration are excluded and
    must be stated as excluded.
    """
    total_shots = circuit_evals * shots
    qpu_seconds = total_shots * seconds_per_shot
    return {
        "circuit_evals": circuit_evals, "shots_per_eval": shots,
        "total_shots": total_shots,
        "qpu_seconds": qpu_seconds, "qpu_hours": qpu_seconds / 3600.0,
        "cost_eur": qpu_seconds * price_per_second_eur +
                    tasks * price_per_task_eur,
        "assumptions": {"seconds_per_shot": seconds_per_shot,
                        "price_per_second_eur": price_per_second_eur,
                        "price_per_task_eur": price_per_task_eur,
                        "excluded": ["queue wait", "calibration",
                                     "transpilation", "error mitigation"]},
    }


def classical_flops(model) -> Dict:
    """Forward-pass multiply-accumulates for the classical layers.

    The comparable axis to the quantum circuit-evaluation count: both are
    per-inference work, and neither is the parameter count.
    """
    import torch.nn as nn

    total, per_layer = 0, {}
    for name, mod in model.named_modules():
        if isinstance(mod, nn.Linear):
            f = mod.in_features * mod.out_features
            per_layer[name] = f
            total += f
    return {"total_macs": total, "per_layer": per_layer}


# =============================================================================
# Energy
# =============================================================================
def kwollect_available() -> bool:
    return os.environ.get("KWOLLECT", "0") == "1" and "grid5000" in platform.node()


def fetch_kwollect_energy(node: str, t_start: float, t_end: float,
                          site: Optional[str] = None) -> Dict:
    """Measured wall power from the Grid5000 wattmeters.

    Requires an EXCLUSIVE node reservation: with co-located jobs the reading
    describes the machine, not the experiment.
    """
    import requests

    site = site or os.environ.get("OAR_SITE", "lyon")
    url = (f"https://api.grid5000.fr/stable/sites/{site}/metrics"
           f"?nodes={node}&metrics=wattmetre_power_watt"
           f"&start_time={int(t_start)}&end_time={int(t_end)}")
    try:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        pts = r.json()
        watts = np.array([p["value"] for p in pts], float)
        if len(watts) == 0:
            return {"available": False, "reason": "no samples returned"}
        dur = t_end - t_start
        return {"available": True, "source": "kwollect_wattmeter",
                "mean_watts": float(watts.mean()),
                "peak_watts": float(watts.max()),
                "duration_s": dur,
                "energy_joules": float(watts.mean() * dur),
                "energy_wh": float(watts.mean() * dur / 3600.0),
                "n_samples": len(watts)}
    except Exception as e:
        return {"available": False, "reason": f"{type(e).__name__}: {e}"}


def estimate_energy_from_cpu(cpu_seconds: float, tdp_watts: float = 105.0,
                             n_cores_used: int = 1,
                             n_cores_total: Optional[int] = None) -> Dict:
    """Fallback ESTIMATE when no wattmeter is available.

    Explicitly labelled an estimate. It must not be presented as a
    measurement; reports must say which of the two they contain.
    """
    total = n_cores_total or os.cpu_count() or 1
    watts = tdp_watts * (n_cores_used / total)
    j = watts * cpu_seconds
    return {"available": True, "source": "cpu_time_estimate",
            "is_estimate": True,
            "assumed_tdp_watts": tdp_watts, "cores_used": n_cores_used,
            "cores_total": total, "effective_watts": watts,
            "cpu_seconds": cpu_seconds,
            "energy_joules": j, "energy_wh": j / 3600.0,
            "caveat": "estimate from CPU time and assumed TDP, NOT a measurement"}


class EnergyMeter:
    """Context manager: wattmeter if available, CPU-time estimate otherwise."""

    def __init__(self, label: str = "run", n_cores_used: int = 1):
        self.label = label
        self.n_cores_used = n_cores_used
        self.result: Dict = {}

    def __enter__(self):
        self.t0 = time.time()
        self.c0 = time.process_time()
        return self

    def __exit__(self, *exc):
        wall = time.time() - self.t0
        cpu = time.process_time() - self.c0
        if kwollect_available():
            e = fetch_kwollect_energy(platform.node(), self.t0, time.time())
            if not e.get("available"):
                e = estimate_energy_from_cpu(cpu, n_cores_used=self.n_cores_used)
        else:
            e = estimate_energy_from_cpu(cpu, n_cores_used=self.n_cores_used)
        e.update({"label": self.label, "wall_seconds": wall, "cpu_seconds": cpu})
        self.result = e
        return False


def co2_equivalent(energy_wh: float, g_per_kwh: float = 56.0) -> Dict:
    """CO2e. Default is the French grid intensity (~56 gCO2e/kWh), which is
    where Grid5000 runs; state the factor in the report, since a US or
    German grid would give a very different number for identical compute."""
    return {"energy_wh": energy_wh, "grid_g_per_kwh": g_per_kwh,
            "co2e_grams": energy_wh / 1000.0 * g_per_kwh}

# ==================================================================
# === Inlined from src/qbaselines.py ===
# ==================================================================
"""
qbaselines.py -- quantum and related-work comparators.

Compares the hybrid model against quantum baselines and against a related
method (Al-Hawawreh & Hossain (2023), Information Fusion 99:101889), in
addition to the classical machine learning baselines.

Four comparator families, all trained and evaluated on the SAME splits, the
same preprocessing and the same schedule as the headline model, so the rows are
directly comparable:

    qsvm_angle      quantum kernel SVM, product-state angle embedding
    qsvm_iqp        quantum kernel SVM, IQP embedding (genuinely entangling)
    vqc             variational classifier: no re-uploading, NO classical
                    decoder -- the measurement is the head
    amp_qnn         amplitude encoding of all 16 PCA components into 4 qubits
    alhawawreh_qdl  re-implementation of the fusion + quantum-deep-learning
                    framework of Al-Hawawreh & Hossain (2023), under the
                    documented deviations in ALHAWAWREH_DEVIATIONS

Why the kernels are computed from state vectors
-----------------------------------------------
For an embedding |phi(x)>, the fidelity kernel is k(x,y) = |<phi(x)|phi(y)>|^2.
Evaluating that pair by pair on a simulator costs N^2 circuit executions and is
what makes quantum-kernel baselines infeasible at realistic N. But the states
themselves only need N executions: simulate once per sample, stack into
S (N x 2^n), and the whole Gram matrix is |S S^H|^2 -- one matmul.

This is exact, not an approximation, and `verify_kernel_matrix` checks it
against pair-by-pair circuit evaluation on a random sample of pairs before any
result is reported. It is what allows a 3,000-sample QSVM here instead of the
few hundred a pairwise implementation would afford.
"""


import time
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pennylane as qml
import torch
import torch.nn as nn

# from config import (   # inlined
#     ALHAWAWREH_FUSION_DIM,   # inlined
#     ALHAWAWREH_GROUPS,   # inlined
#     ALHAWAWREH_GROUP_DIM,   # inlined
#     ALHAWAWREH_MODALITIES,   # inlined
#     AMP_QNN_LAYERS,   # inlined
#     AMP_QNN_QUBITS,   # inlined
#     CFG,   # inlined
#     LITERATURE_ROWS,   # inlined
#     N_LAYERS,   # inlined
#     N_QUBITS,   # inlined
#     QSVM_C_GRID,   # inlined
#     QSVM_TRAIN_SUBSAMPLE,   # inlined
#     VQC_LAYERS,   # inlined
#     n_classes_for,   # inlined
# )   # inlined


# =============================================================================
# Feature preparation shared by every quantum comparator
# =============================================================================
def to_angles(X: np.ndarray, n_qubits: int = N_QUBITS,
              bounds: Optional[Tuple[np.ndarray, np.ndarray]] = None):
    """Map PCA components onto rotation angles in [-pi, pi].

    Angle embedding is only meaningful on a bounded input, and PCA components
    are not bounded. The min/max are fitted on the TRAINING rows and reused
    verbatim for validation and test -- fitting them on the test set would leak,
    which is exactly the class of mistake that evaluation protocols guard against.

    Returns (angles, bounds) so the caller can thread the training bounds
    through to the test split.
    """
    Z = np.asarray(X, dtype=np.float64)[:, :n_qubits]
    if bounds is None:
        lo, hi = Z.min(axis=0), Z.max(axis=0)
        bounds = (lo, hi)
    lo, hi = bounds
    span = np.where(hi - lo < 1e-12, 1.0, hi - lo)
    return (2.0 * (np.clip(Z, lo, hi) - lo) / span - 1.0) * np.pi, bounds


def stratified_subsample(X: np.ndarray, y: np.ndarray, n: int, seed: int = 42):
    """Seeded stratified subsample; returns (X, y, idx). Used by the QSVM,
    whose Gram matrix is quadratic in the training-set size."""
    if n is None or len(X) <= n:
        return X, y, np.arange(len(X))
    rng = np.random.default_rng(seed)
    idx: List[int] = []
    classes, counts = np.unique(y, return_counts=True)
    quota = np.maximum(1, np.round(n * counts / counts.sum()).astype(int))
    for c, q in zip(classes, quota):
        pool = np.flatnonzero(y == c)
        idx.extend(rng.choice(pool, size=min(q, len(pool)), replace=False).tolist())
    idx = np.array(sorted(idx))[:n]
    return X[idx], y[idx], idx


# =============================================================================
# Quantum kernels
# =============================================================================
def _state_qnode(embedding: str, n_qubits: int):
    dev = qml.device("default.qubit", wires=n_qubits)

    @qml.qnode(dev, interface="numpy")
    def circuit(x):
        if embedding == "angle":
            # Product state: sum_j R_Y(x_j). Same encoding family as the
            # paper's eq. (3), so the kernel row is the kernelised
            # counterpart of our own embedding.
            qml.AngleEmbedding(x, wires=range(n_qubits), rotation="Y")
        elif embedding == "iqp":
            # IQP embedding entangles: its Gram matrix is not a product of
            # single-qubit overlaps, so this row is the one that cannot be
            # written down in closed form.
            qml.IQPEmbedding(x, wires=range(n_qubits), n_repeats=2)
        else:
            raise ValueError(f"unknown embedding: {embedding}")
        return qml.state()

    return circuit


def embed_states(X: np.ndarray, embedding: str = "angle",
                 n_qubits: int = N_QUBITS, batch: int = 512) -> np.ndarray:
    """Simulate |phi(x)> for every row. N executions, not N^2."""
    circuit = _state_qnode(embedding, n_qubits)
    out = np.empty((len(X), 2 ** n_qubits), dtype=np.complex128)
    for s in range(0, len(X), batch):
        chunk = np.asarray(X[s:s + batch], dtype=np.float64)
        out[s:s + batch] = np.asarray(circuit(chunk)).reshape(len(chunk), -1)
    return out


def fidelity_kernel(S1: np.ndarray, S2: np.ndarray) -> np.ndarray:
    """k(x,y) = |<phi(x)|phi(y)>|^2 for pre-computed state matrices."""
    return np.abs(S1 @ S2.conj().T) ** 2


def verify_kernel_matrix(X: np.ndarray, K: np.ndarray, embedding: str,
                         n_qubits: int = N_QUBITS, n_pairs: int = 100,
                         seed: int = 42, tol: float = 1e-9) -> Dict:
    """Check the state-vector Gram matrix against pair-by-pair evaluation.

    The fast path is a mathematical identity, but an identity that is asserted
    and never tested can hide a bug. This tests it.
    """
    dev = qml.device("default.qubit", wires=n_qubits)

    def _embed(x):
        if embedding == "angle":
            qml.AngleEmbedding(x, wires=range(n_qubits), rotation="Y")
        else:
            qml.IQPEmbedding(x, wires=range(n_qubits), n_repeats=2)

    @qml.qnode(dev, interface="numpy")
    def overlap(x1, x2):
        _embed(x1)
        qml.adjoint(_embed)(x2)
        return qml.probs(wires=range(n_qubits))

    rng = np.random.default_rng(seed)
    ii = rng.integers(0, len(X), n_pairs)
    jj = rng.integers(0, len(X), n_pairs)
    err = [abs(float(overlap(X[i], X[j])[0]) - float(K[i, j]))
           for i, j in zip(ii, jj)]
    max_err = float(np.max(err))
    return {"embedding": embedding, "n_pairs": int(n_pairs),
            "max_abs_error": max_err, "tolerance": tol, "passed": max_err < tol}


def fit_qsvm(X_tr: np.ndarray, y_tr: np.ndarray, X_te: np.ndarray,
             embedding: str = "angle", n_qubits: int = N_QUBITS,
             C_grid: Sequence[float] = tuple(QSVM_C_GRID),
             X_val: Optional[np.ndarray] = None,
             y_val: Optional[np.ndarray] = None,
             seed: int = 42) -> Tuple[object, np.ndarray, np.ndarray, Dict]:
    """Quantum-kernel SVM with a precomputed Gram matrix.

    C is selected on the VALIDATION split when one is supplied -- never on the
    test split (gate G3 in spirit: no test data touches model selection).

    Returns (estimator, K_test, K_train, meta).
    """
    from sklearn.svm import SVC

    t0 = time.time()
    A_tr, bounds = to_angles(X_tr, n_qubits)
    A_te, _ = to_angles(X_te, n_qubits, bounds)
    S_tr = embed_states(A_tr, embedding, n_qubits)
    S_te = embed_states(A_te, embedding, n_qubits)
    K_tr = fidelity_kernel(S_tr, S_tr)
    K_te = fidelity_kernel(S_te, S_tr)
    t_kernel = time.time() - t0

    best_C, best_score = C_grid[0], -1.0
    if X_val is not None and y_val is not None and len(C_grid) > 1:
        A_va, _ = to_angles(X_val, n_qubits, bounds)
        K_va = fidelity_kernel(embed_states(A_va, embedding, n_qubits), S_tr)
        for C in C_grid:
            est = SVC(C=C, kernel="precomputed", probability=False,
                      random_state=seed).fit(K_tr, y_tr)
            score = float((est.predict(K_va) == y_val).mean())
            if score > best_score:
                best_C, best_score = C, score

    t1 = time.time()
    est = SVC(C=best_C, kernel="precomputed", probability=True,
              random_state=seed).fit(K_tr, y_tr)
    meta = {"embedding": embedding, "n_qubits": n_qubits, "C": float(best_C),
            "val_accuracy": best_score if best_score >= 0 else None,
            "n_train": int(len(X_tr)), "n_test": int(len(X_te)),
            "kernel_wall_s": t_kernel, "fit_wall_s": time.time() - t1,
            "circuit_evals": int(len(X_tr) + len(X_te)),
            "circuit_evals_pairwise_equivalent":
                int(len(X_tr) ** 2 + len(X_te) * len(X_tr)),
            "n_support_vectors": int(est.support_.size),
            "angle_bounds_fitted_on": "train"}
    return est, K_te, K_tr, meta


# =============================================================================
# Variational classifier -- no classical decoder
# =============================================================================
class VQC(nn.Module):
    """Angle-encoded variational classifier, no re-uploading, no decoder.

    This is the comparator that answers "is the classical encoder/decoder doing
    the work?" from the opposite direction to NB04's parameter-matched controls:
    there we kept the scaffolding and removed the quantum layer, here we keep
    the quantum layer and remove the scaffolding.

    Head: one measured qubit per class. A per-class affine readout (2*C
    parameters) is kept because cross-entropy over raw expectation values in
    [-1, 1] is badly scaled; it is counted and disclosed, not hidden.
    """

    def __init__(self, task: str, n_qubits: int = N_QUBITS,
                 n_layers: int = VQC_LAYERS, n_classes: Optional[int] = None):
        super().__init__()
        self.task = task
        self.n_qubits, self.n_layers = n_qubits, n_layers
        self.n_classes = n_classes or n_classes_for(task)
        if self.n_classes > n_qubits:
            raise ValueError("VQC head needs one qubit per class")
        self.q_weights = nn.Parameter(torch.randn(n_layers, n_qubits, 3) * 0.1)
        self.readout_scale = nn.Parameter(torch.ones(self.n_classes))
        self.readout_bias = nn.Parameter(torch.zeros(self.n_classes))
        self.n_circuit_evals = 0

        dev = qml.device("default.qubit", wires=n_qubits)

        @qml.qnode(dev, interface="torch", diff_method="backprop")
        def circuit(inputs, weights):
            qml.AngleEmbedding(inputs, wires=range(n_qubits), rotation="Y")
            for layer in range(n_layers):
                for i in range(n_qubits):
                    qml.Rot(weights[layer, i, 0], weights[layer, i, 1],
                            weights[layer, i, 2], wires=i)
                for i in range(n_qubits):
                    qml.CNOT(wires=[i, (i + 1) % n_qubits])
            return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

        self.qnode = circuit

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        self.n_circuit_evals += x.shape[0]
        z = torch.tanh(x[:, :self.n_qubits]) * np.pi   # bounded angle map
        q = torch.stack(list(self.qnode(z, self.q_weights)), dim=-1).to(x.dtype)
        return q[:, :self.n_classes] * self.readout_scale + self.readout_bias

    def param_breakdown(self) -> Dict[str, int]:
        n = lambda t: int(sum(p.numel() for p in t))
        return {"encoder": 0,
                "bottleneck": n([self.q_weights]),
                "decoder": n([self.readout_scale, self.readout_bias]),
                "total": n(list(self.parameters()))}

    def footprint_kb(self, bits: int = 32) -> float:
        return self.param_breakdown()["total"] * (bits / 8) / 1024.0


class AmplitudeQNN(nn.Module):
    """Amplitude encoding: all 16 PCA components into log2(16) = 4 qubits.

    Angle encoding needs one qubit per feature and therefore discards 8 of the
    16 components; amplitude encoding keeps all of them at a quarter of the
    width. The head is `qml.probs` over the first ceil(log2 C) qubits, so this
    comparator too has no classical decoder.
    """

    def __init__(self, task: str, n_features: int = 16,
                 n_qubits: int = AMP_QNN_QUBITS, n_layers: int = AMP_QNN_LAYERS,
                 n_classes: Optional[int] = None):
        super().__init__()
        self.n_classes = n_classes or n_classes_for(task)
        self.n_qubits, self.n_layers = n_qubits, n_layers
        self.n_features = n_features
        self.head_wires = int(np.ceil(np.log2(self.n_classes)))
        self.q_weights = nn.Parameter(torch.randn(n_layers, n_qubits, 3) * 0.1)
        self.readout_scale = nn.Parameter(torch.ones(self.n_classes))
        self.n_circuit_evals = 0

        dev = qml.device("default.qubit", wires=n_qubits)

        @qml.qnode(dev, interface="torch", diff_method="backprop")
        def circuit(inputs, weights):
            qml.AmplitudeEmbedding(inputs, wires=range(n_qubits),
                                   normalize=True, pad_with=0.0)
            for layer in range(n_layers):
                for i in range(n_qubits):
                    qml.Rot(weights[layer, i, 0], weights[layer, i, 1],
                            weights[layer, i, 2], wires=i)
                for i in range(n_qubits):
                    qml.CNOT(wires=[i, (i + 1) % n_qubits])
            return qml.probs(wires=range(self.head_wires))

        self.qnode = circuit

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        self.n_circuit_evals += x.shape[0]
        probs = self.qnode(x, self.q_weights).to(x.dtype)
        # log-probabilities are the natural logits for a probability head
        return torch.log(probs[:, :self.n_classes] + 1e-12) * self.readout_scale

    def param_breakdown(self) -> Dict[str, int]:
        n = lambda t: int(sum(p.numel() for p in t))
        return {"encoder": 0, "bottleneck": n([self.q_weights]),
                "decoder": n([self.readout_scale]),
                "total": n(list(self.parameters()))}

    def footprint_kb(self, bits: int = 32) -> float:
        return self.param_breakdown()["total"] * (bits / 8) / 1024.0


# =============================================================================
# Al-Hawawreh & Hossain (2023) -- re-implementation under stated deviations
# =============================================================================
ALHAWAWREH_DEVIATIONS = {
    "reference": "Al-Hawawreh, M., & Hossain, M. S. (2023). A privacy-aware "
                 "framework for detecting cyber attacks on internet of medical "
                 "things systems using data fusion and quantum deep learning. "
                 "Information Fusion, 99, 101889.",
    "fusion_modalities": (
        "The source framework fuses heterogeneous modalities (patient "
        "biometric telemetry alongside network telemetry). The CICIoMT2024 "
        "WiFi/MQTT partition carries no biometric channel, so the "
        "re-implementation fuses the three FEATURE GROUPS present in the flow "
        "records (volume, timing, protocol) through per-group encoders. This "
        "preserves the architectural pattern -- per-modality encoders into a "
        "shared fused representation -- on the data that exists."),
    "privacy_mechanism": (
        "The privacy-preserving component of the source framework is "
        "orthogonal to detection accuracy and is NOT re-implemented. The row "
        "therefore measures the detection architecture only, and the "
        "results must be reported with this caveat."),
    "hyperparameters": (
        "Layer widths not fixed by the source text are set to this paper's "
        "budget (per-group width 8, fused width 16) so the comparison is at "
        "matched capacity rather than matched-to-an-unknown."),
    "training_protocol": (
        "Identical to every other row: same splits, same optimiser, same "
        "schedule, validation-only checkpoint selection."),
    "status": "faithful re-implementation under the deviations above; NOT the "
              "authors' own reported numbers",
}


class AlHawawrehQDL(nn.Module):
    """Per-group encoders -> fusion -> variational quantum layer -> classifier.

    Deliberately NOT built from `HybridIDS`: it is a different architecture from
    a different paper, and reusing our scaffolding would quietly turn it into a
    variant of our own model. `ALHAWAWREH_DEVIATIONS` travels with every result
    this class produces.
    """

    def __init__(self, task: str, group_sizes: Sequence[int],
                 n_qubits: int = N_QUBITS, n_layers: int = N_LAYERS,
                 n_classes: Optional[int] = None):
        super().__init__()
        self.task = task
        self.group_sizes = list(group_sizes)
        self.n_classes = n_classes or n_classes_for(task)
        self.n_qubits, self.n_layers = n_qubits, n_layers
        self.n_circuit_evals = 0

        self.group_encoders = nn.ModuleList([
            nn.Sequential(nn.Linear(g, ALHAWAWREH_GROUP_DIM), nn.ReLU(),
                          nn.Dropout(0.2))
            for g in self.group_sizes
        ])
        self.fusion = nn.Sequential(
            nn.Linear(ALHAWAWREH_GROUP_DIM * len(self.group_sizes),
                      ALHAWAWREH_FUSION_DIM), nn.ReLU(),
            nn.Linear(ALHAWAWREH_FUSION_DIM, n_qubits), nn.Tanh(),
        )
        self.q_weights = nn.Parameter(torch.randn(n_layers, n_qubits, 3) * 0.1)
        self.head = nn.Linear(n_qubits, self.n_classes)

        dev = qml.device("default.qubit", wires=n_qubits)

        @qml.qnode(dev, interface="torch", diff_method="backprop")
        def circuit(inputs, weights):
            qml.AngleEmbedding(inputs, wires=range(n_qubits), rotation="Y")
            for layer in range(n_layers):
                for i in range(n_qubits):
                    qml.Rot(weights[layer, i, 0], weights[layer, i, 1],
                            weights[layer, i, 2], wires=i)
                for i in range(n_qubits - 1):
                    qml.CNOT(wires=[i, i + 1])
            return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

        self.qnode = circuit

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        parts, s = [], 0
        for enc, g in zip(self.group_encoders, self.group_sizes):
            parts.append(enc(x[:, s:s + g]))
            s += g
        z = self.fusion(torch.cat(parts, dim=1))
        self.n_circuit_evals += x.shape[0]
        q = torch.stack(list(self.qnode(z, self.q_weights)), dim=-1).to(x.dtype)
        return self.head(q)

    def param_breakdown(self) -> Dict[str, int]:
        n = lambda m: int(sum(p.numel() for p in m.parameters()
                              if p.requires_grad))
        enc = n(self.group_encoders) + n(self.fusion)
        return {"encoder": enc, "bottleneck": int(self.q_weights.numel()),
                "decoder": n(self.head),
                "total": enc + int(self.q_weights.numel()) + n(self.head)}

    def footprint_kb(self, bits: int = 32) -> float:
        return self.param_breakdown()["total"] * (bits / 8) / 1024.0


def group_feature_matrix(X_raw: np.ndarray, feature_names: Sequence[str],
                         modalities: Optional[Dict[str, Sequence[str]]] = None):
    """Re-order raw features into contiguous per-modality blocks.

    Returns (X_grouped, group_sizes, group_columns, names_by_group).

    The fusion architecture needs a PARTITION: every feature in exactly one
    modality, no modality empty. Both conditions are asserted here rather than
    assumed -- an empty modality would silently create a `Linear(0, 8)` encoder
    that trains to a constant, which is the kind of bug that only shows up
    as an inexplicably weak comparator row.
    """
    mods = modalities or ALHAWAWREH_MODALITIES
    lower = [str(n).lower() for n in feature_names]
    assigned: Dict[int, str] = {}
    cols: Dict[str, List[int]] = {g: [] for g in mods}

    for i, name in enumerate(lower):
        for g, keys in mods.items():          # priority = declaration order
            if any(k in name for k in keys):
                cols[g].append(i)
                assigned[i] = g
                break

    unassigned = [i for i in range(len(feature_names)) if i not in assigned]
    if unassigned:                            # nothing may be silently dropped
        cols.setdefault("other", []).extend(unassigned)

    empty = [g for g, c in cols.items() if not c]
    for g in empty:
        cols.pop(g)
    if empty:
        print(f"[qbaselines] modalities with no matching feature, dropped: {empty}")

    flat = [i for g in cols for i in cols[g]]
    assert sorted(flat) == list(range(len(feature_names))),         "modality assignment is not a partition of the feature set"

    sizes = [len(cols[g]) for g in cols]
    names_by_group = {g: [str(feature_names[i]) for i in c]
                      for g, c in cols.items()}
    return np.asarray(X_raw)[:, flat], sizes, cols, names_by_group


# =============================================================================
# Literature rows
# =============================================================================
def literature_table(rows: Sequence[Dict] = tuple(LITERATURE_ROWS)) -> Dict:
    """Split the related-work rows into transcribed and outstanding.

    A reported number that has not been copied from the cited article is
    reported as OUTSTANDING. It is never guessed, never interpolated from an
    abstract, and never left as a blank table cell.
    """
    done = [r for r in rows if r.get("reported") is not None]
    todo = [r for r in rows if r.get("reported") is None]
    return {"transcribed": done, "outstanding": todo,
            "note": "Reported numbers are on the authors' own datasets and "
                    "splits; they are NOT directly comparable with the "
                    "CICIoMT2024 rows measured here. The comparable row is the "
                    "re-implementation, trained on the same split as every "
                    "other model in this paper."}


def build_quantum_baseline(name: str, task: str, n_features: int = 16,
                           group_sizes: Optional[Sequence[int]] = None):
    """Factory for the trainable quantum comparators (the QSVMs are fitted by
    `fit_qsvm`, which is not a torch module)."""
    if name == "vqc":
        return VQC(task)
    if name == "amp_qnn":
        return AmplitudeQNN(task, n_features=n_features)
    if name == "alhawawreh_qdl":
        if group_sizes is None:
            raise ValueError("alhawawreh_qdl needs group_sizes from "
                             "group_feature_matrix()")
        return AlHawawrehQDL(task, group_sizes)
    raise ValueError(f"unknown quantum baseline: {name}")

# ==================================================================
# === Inlined from src/hierarchical.py ===
# ==================================================================
"""
hierarchical.py -- diagnosing and attacking the multiclass gap.

The binary detector achieves excellent performance, whereas multiclass
accuracy is relatively lower (79.19%).

The first thing to establish is WHERE the 20.8 points go. On the archived
predictions the answer is unusually clean:

    1,163 DoS  -> DDoS   74.5% of all error
      351 DDoS -> DoS    22.5% of all error
    ---------------------------------------
                         97.0% of all error is one confusion between two
                         volumetric flooding families

Everything else -- all six remaining classes, all other off-diagonal cells --
accounts for 47 of 7,500 test samples. Merge DoS and DDoS into one volumetric
superclass and the same, untouched model scores 99.37%.

That reframes the gap: the multiclass task is not broadly harder, one binary
decision inside it is. This module provides the three things needed to say that
with evidence, and to test whether the decision can be recovered:

    error_decomposition()   where the error actually is          (no training)
    separability_probe()    can ANY classifier separate DoS from DDoS on these
                            features? -- the representation ceiling (minutes)
    two-stage / feature-path models                              (training)

`separability_probe` is the control experiment. If tuned classical specialists,
given only the volumetric rows and the raw 25 features, also plateau, then the
ceiling belongs to the CICIoMT2024 WiFi/MQTT feature set -- which records no
source-address cardinality or dispersion, the very thing that distinguishes a
distributed flood from a single-source one -- and not to the quantum model. The
paper can then state a measured limit.
"""


from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import torch
import torch.nn as nn

# from config import (   # inlined
#     CLASS_NAMES,   # inlined
#     COARSE_NAMES,   # inlined
#     FEATURE_PATH_K,   # inlined
#     N_CLASSES,   # inlined
#     VOLUMETRIC_CLASSES,   # inlined
# )   # inlined
# from models import HybridIDS, build_decoder, build_encoder, build_model   # inlined


# =============================================================================
# Label taxonomy
# =============================================================================
VOL_LO, VOL_HI = min(VOLUMETRIC_CLASSES), max(VOLUMETRIC_CLASSES)
COARSE_VOLUMETRIC = len(COARSE_NAMES) - 1        # index of Volumetric_Flood


def to_coarse(y: np.ndarray) -> np.ndarray:
    """8 classes -> 7, with DoS and DDoS merged into one volumetric class."""
    y = np.asarray(y)
    out = y.copy()
    out[np.isin(y, VOLUMETRIC_CLASSES)] = COARSE_VOLUMETRIC
    return out


def to_fine(y_coarse: np.ndarray, y_specialist: np.ndarray) -> np.ndarray:
    """Compose stage-1 (7-way) and stage-2 (DoS vs DDoS) into 8-class labels.

    `y_specialist` is 0 for the lower volumetric class (DoS) and 1 for the
    higher (DDoS); it is only read where stage 1 predicted volumetric.
    """
    out = np.asarray(y_coarse).copy()
    vol = out == COARSE_VOLUMETRIC
    out[vol] = np.where(np.asarray(y_specialist)[vol] == 1, VOL_HI, VOL_LO)
    return out


def volumetric_mask(y: np.ndarray) -> np.ndarray:
    return np.isin(np.asarray(y), VOLUMETRIC_CLASSES)


def specialist_labels(y: np.ndarray) -> np.ndarray:
    """DoS -> 0, DDoS -> 1 (defined only on volumetric rows)."""
    return (np.asarray(y) == VOL_HI).astype(np.int64)


# =============================================================================
# Where does the error go? -- no training required
# =============================================================================
def error_decomposition(y_true, y_pred,
                        class_names: Sequence[str] = tuple(CLASS_NAMES),
                        top: int = 10) -> Dict:
    """Rank every off-diagonal confusion by its share of total error.

    This is the table that turns "multiclass accuracy is lower" into a located,
    quantified statement, and it costs nothing: it reads the predictions the
    paper already reports.
    """
    from sklearn.metrics import confusion_matrix

    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    n = len(y_true)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
    n_err = int((y_true != y_pred).sum())

    pairs = [{"true": class_names[i], "pred": class_names[j],
              "true_idx": i, "pred_idx": j, "count": int(cm[i, j]),
              "share_of_error": float(cm[i, j] / n_err) if n_err else 0.0,
              "share_of_total": float(cm[i, j] / n)}
             for i in range(len(class_names)) for j in range(len(class_names))
             if i != j and cm[i, j] > 0]
    pairs.sort(key=lambda r: -r["count"])

    vol_conf = int(cm[VOL_LO, VOL_HI] + cm[VOL_HI, VOL_LO])
    coarse_acc = float((to_coarse(y_true) == to_coarse(y_pred)).mean())

    return {
        "n_test": n, "n_errors": n_err,
        "accuracy": float((y_true == y_pred).mean()),
        "top_confusions": pairs[:top],
        "volumetric_confusion_count": vol_conf,
        "volumetric_share_of_error": float(vol_conf / n_err) if n_err else 0.0,
        "accuracy_merging_volumetric": coarse_acc,
        "accuracy_headroom_if_volumetric_perfect": coarse_acc - float(
            (y_true == y_pred).mean()),
        "confusion_matrix": cm.tolist(),
        "class_names": list(class_names),
    }


def ceiling_analysis(y_true, y_pred, specialist_accuracies: Sequence[float]
                     ) -> List[Dict]:
    """End-to-end 8-class accuracy implied by a given DoS-vs-DDoS accuracy.

    Ties the specialist's subproblem accuracy to the headline number, so a
    2-point gain on the subproblem is not quietly reported as a 2-point gain
    overall. Assumes stage 1 keeps its measured coarse accuracy.
    """
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    coarse_acc = float((to_coarse(y_true) == to_coarse(y_pred)).mean())
    vol_frac = float(volumetric_mask(y_true).mean())
    return [{"specialist_accuracy": float(a),
             "implied_overall_accuracy": coarse_acc - vol_frac * (1.0 - float(a)),
             "volumetric_fraction_of_test": vol_frac,
             "coarse_accuracy": coarse_acc}
            for a in specialist_accuracies]


# =============================================================================
# Representation ceiling -- is the information even present?
# =============================================================================
def separability_probe(X_tr, y_tr, X_te, y_te, space: str = "pca16",
                       models: Sequence[str] = ("Random Forest",
                                                "Gradient Boosting", "MLP"),
                       seed: int = 42) -> List[Dict]:
    """Fit classical specialists on the volumetric rows only.

    A specialist that sees nothing but DoS and DDoS, with the full training
    budget of that subproblem, is the strongest evidence available about what
    the features can support. Run it in both the PCA-16 and the raw-25 space:
    the difference is what PCA costs, the level is what the feature set allows.
    """
#     from baselines import evaluate_sklearn, fit_baseline   # inlined

    m_tr, m_te = volumetric_mask(y_tr), volumetric_mask(y_te)
    Xa, ya = np.asarray(X_tr)[m_tr], specialist_labels(y_tr)[m_tr]
    Xb, yb = np.asarray(X_te)[m_te], specialist_labels(y_te)[m_te]

    rows = []
    for name in models:
        est, meta = fit_baseline(name, Xa, ya, seed=seed)
        r = evaluate_sklearn(est, Xb, yb, 2)
        rows.append({"probe": name, "space": space, "n_train": int(m_tr.sum()),
                     "n_test": int(m_te.sum()), "accuracy": r["accuracy"],
                     "f1": r["f1"], "auc": r["auc"],
                     "fit_wall_s": meta.get("fit_time_s")})
    # Majority-class floor: the number every row above must beat to mean anything
    floor = float(max(np.bincount(yb, minlength=2)) / len(yb))
    rows.append({"probe": "majority-class floor", "space": space,
                 "n_train": int(m_tr.sum()), "n_test": int(m_te.sum()),
                 "accuracy": floor, "f1": None, "auc": None, "fit_wall_s": 0.0})
    return rows


def mutual_information_ranking(X, y, feature_names: Sequence[str],
                               seed: int = 42) -> List[Dict]:
    """Rank raw features by MI with the DoS/DDoS label, on TRAIN rows only.

    Feeds the feature-path variant: which raw features are worth routing around
    PCA. Selecting them on the training split is the whole point -- selecting on
    the test split would manufacture the improvement it claims to measure.
    """
    from sklearn.feature_selection import mutual_info_classif

    m = volumetric_mask(y)
    mi = mutual_info_classif(np.asarray(X)[m], specialist_labels(y)[m],
                             random_state=seed)
    order = np.argsort(mi)[::-1]
    return [{"rank": int(r + 1), "feature": str(feature_names[i]),
             "column": int(i), "mutual_information": float(mi[i])}
            for r, i in enumerate(order)]


# =============================================================================
# Approach 1 -- two-stage hierarchical detector
# =============================================================================
class TwoStageIDS(nn.Module):
    """Stage 1: 7-way coarse detector. Stage 2: DoS-vs-DDoS specialist.

    Both stages are `HybridIDS` from the same factory as the headline model --
    same encoder, same 8-qubit PQC, same decoder shape -- so any difference is
    attributable to the decomposition, not to a new architecture. Stage 2 is
    trained on volumetric rows only, which is what gives it a class balance the
    flat 8-way model never sees (32,195 rows split 76/24 instead of 5,012
    against 1,887 inside a 35,000-row 8-class problem).

    Cost is disclosed, not hidden: two models, two forward passes for the
    volumetric fraction of traffic, and roughly twice the parameters of the flat
    model. `param_breakdown` reports the sum.
    """

    def __init__(self, n_features: int = 16, n_qubits: int = 8,
                 n_layers: int = 4):
        super().__init__()
        self.stage1 = build_model("multiclass", "full", n_features, n_qubits,
                                  n_layers, n_classes=len(COARSE_NAMES))
        self.stage2 = build_model("binary", "full", n_features, n_qubits,
                                  n_layers)

    @torch.no_grad()
    def predict(self, X: np.ndarray, batch: int = 4096) -> Dict[str, np.ndarray]:
        """Compose the two stages into 8-class predictions."""
        self.eval()
        xs = torch.tensor(np.asarray(X), dtype=torch.float32)
        coarse, spec = [], []
        for s in range(0, len(xs), batch):
            xb = xs[s:s + batch]
            coarse.append(self.stage1(xb).argmax(1).numpy())
            spec.append(self.stage2(xb).argmax(1).numpy())
        coarse = np.concatenate(coarse)
        spec = np.concatenate(spec)
        return {"y_pred": to_fine(coarse, spec), "coarse": coarse,
                "specialist": spec}

    def param_breakdown(self) -> Dict[str, int]:
        a, b = self.stage1.param_breakdown(), self.stage2.param_breakdown()
        return {"stage1_total": a["total"], "stage2_total": b["total"],
                "encoder": a["encoder"] + b["encoder"],
                "bottleneck": a["bottleneck"] + b["bottleneck"],
                "decoder": a["decoder"] + b["decoder"],
                "total": a["total"] + b["total"]}

    def footprint_kb(self, bits: int = 32) -> float:
        return self.param_breakdown()["total"] * (bits / 8) / 1024.0


# =============================================================================
# Approach 2 -- raw features routed around PCA
# =============================================================================
class FeaturePathIDS(nn.Module):
    """Flat 8-class model with k raw features bypassing PCA into the decoder.

    Hypothesis under test: PCA, fitted to maximise variance over all eight
    classes, discards the directions that separate DoS from DDoS. If so,
    delivering a handful of raw features straight to the decoder should recover
    part of the gap at a cost of k*(d1) extra weights.

    The encoder and the quantum bottleneck are untouched; only the decoder is
    wider by k inputs, and `param_breakdown` reports exactly how much wider.
    """

    def __init__(self, task: str = "multiclass", n_features: int = 16,
                 n_qubits: int = 8, n_layers: int = 4,
                 k_raw: int = FEATURE_PATH_K, raw_columns: Optional[Sequence[int]] = None,
                 n_classes: Optional[int] = None):
        super().__init__()
        self.task = task
        self.k_raw = k_raw
        self.raw_columns = list(raw_columns) if raw_columns is not None else []
        n_classes = n_classes or (2 if task == "binary" else N_CLASSES)
        self.classical_encoder = build_encoder(task, n_features, n_qubits)
        base = HybridIDS(task, n_features, n_qubits, n_layers, bottleneck="pqc")
        self.bottleneck = base.bottleneck
        self.classical_decoder = build_decoder(task, n_qubits + k_raw, n_classes)

    def forward(self, x_pca: torch.Tensor, x_raw: torch.Tensor) -> torch.Tensor:
        q = self.bottleneck(self.classical_encoder(x_pca))
        return self.classical_decoder(torch.cat([q, x_raw], dim=1))

    def param_breakdown(self) -> Dict[str, int]:
        def n(m):
            return int(sum(p.numel() for p in m.parameters() if p.requires_grad))
        enc, bot, dec = (n(self.classical_encoder), n(self.bottleneck),
                         n(self.classical_decoder))
        flat = HybridIDS(self.task).param_breakdown()["total"]
        return {"encoder": enc, "bottleneck": bot, "decoder": dec,
                "total": enc + bot + dec, "k_raw": self.k_raw,
                "extra_vs_flat": enc + bot + dec - flat}

    def footprint_kb(self, bits: int = 32) -> float:
        return self.param_breakdown()["total"] * (bits / 8) / 1024.0


class PairedDataset(torch.utils.data.Dataset):
    """(PCA features, routed raw features, label) for FeaturePathIDS."""

    def __init__(self, X_pca, X_raw, y):
        self.a = torch.tensor(np.asarray(X_pca), dtype=torch.float32)
        self.b = torch.tensor(np.asarray(X_raw), dtype=torch.float32)
        self.y = torch.tensor(np.asarray(y), dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.a[i], self.b[i], self.y[i]


def train_feature_path(model: FeaturePathIDS, tr: Tuple, va: Tuple,
                       task: str = "multiclass",
                       class_weights: Optional[torch.Tensor] = None,
                       batch_size: int = None, verbose: bool = True) -> Dict:
    """Train FeaturePathIDS. Validation-only checkpoint selection, epochs and
    learning rates taken from config exactly as `train.train_model` does -- the
    two-input signature is the only reason this cannot call it directly."""
    from torch.optim import AdamW
    from torch.optim.lr_scheduler import CosineAnnealingLR
    from torch.utils.data import DataLoader

#     from config import CFG, epochs_for, lr_for   # inlined
    from sklearn.metrics import f1_score

    epochs = epochs_for(task)
    lr, lr_min = lr_for(task)
    bs = batch_size or CFG.train.BATCH_SIZE

    dl_tr = DataLoader(PairedDataset(*tr), batch_size=bs, shuffle=True)
    dl_va = DataLoader(PairedDataset(*va), batch_size=bs, shuffle=False)

    crit = nn.CrossEntropyLoss(weight=class_weights)
    opt = AdamW(model.parameters(), lr=lr, weight_decay=CFG.train.WEIGHT_DECAY)
    sched = CosineAnnealingLR(opt, T_max=epochs, eta_min=lr_min)

    hist = {"train_loss": [], "train_acc": [], "val_f1": []}
    best_f1, best_state, since = -1.0, None, 0

    for epoch in range(1, epochs + 1):
        model.train()
        tot = corr = 0
        loss_sum = 0.0
        for xa, xb, yb in dl_tr:
            opt.zero_grad()
            out = model(xa, xb)
            loss = crit(out, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CFG.train.GRAD_CLIP)
            opt.step()
            loss_sum += float(loss) * len(yb)
            corr += int((out.argmax(1) == yb).sum())
            tot += len(yb)
        sched.step()

        model.eval()
        preds, labels = [], []
        with torch.no_grad():
            for xa, xb, yb in dl_va:
                preds.append(model(xa, xb).argmax(1).numpy())
                labels.append(yb.numpy())
        vf1 = float(f1_score(np.concatenate(labels), np.concatenate(preds),
                             average="weighted", zero_division=0))

        hist["train_loss"].append(loss_sum / tot)
        hist["train_acc"].append(corr / tot)
        hist["val_f1"].append(vf1)

        if vf1 > best_f1:
            best_f1, since = vf1, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            since += 1
        if verbose and (epoch == 1 or epoch % 10 == 0 or epoch == epochs):
            print(f"  epoch {epoch:3d}/{epochs} train_acc={corr/tot:.4f} "
                  f"val_f1={vf1:.4f}", flush=True)
        if epoch >= CFG.train.EARLY_STOP_MIN_EPOCHS \
                and since >= CFG.train.EARLY_STOP_PATIENCE:
            if verbose:
                print(f"  early stop at epoch {epoch} (best val_f1={best_f1:.4f})")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    hist["epochs_run"] = epoch
    hist["best_val_f1"] = best_f1
    return hist


@torch.no_grad()
def evaluate_feature_path(model: FeaturePathIDS, X_pca, X_raw, y,
                          n_classes: int = N_CLASSES,
                          batch: int = 4096) -> Dict:
    """Same metric dictionary shape as `train.evaluate`, so the persistence
    layer and every downstream aggregation treat this run like any other."""
    from sklearn.metrics import (accuracy_score, confusion_matrix,
                                 precision_recall_fscore_support, roc_auc_score)

    model.eval()
    xa = torch.tensor(np.asarray(X_pca), dtype=torch.float32)
    xb = torch.tensor(np.asarray(X_raw), dtype=torch.float32)
    probs = []
    for s in range(0, len(xa), batch):
        probs.append(torch.softmax(model(xa[s:s + batch], xb[s:s + batch]),
                                   dim=1).numpy())
    probs = np.concatenate(probs)
    pred = probs.argmax(1)
    y = np.asarray(y)

    avg = "binary" if n_classes == 2 else "weighted"
    p, r, f1, _ = precision_recall_fscore_support(y, pred, average=avg,
                                                  zero_division=0)
    try:
        auc = (roc_auc_score(y, probs[:, 1]) if n_classes == 2
               else roc_auc_score(y, probs, multi_class="ovr", average="weighted"))
    except Exception:
        auc = float("nan")
    pc_p, pc_r, pc_f1, pc_sup = precision_recall_fscore_support(
        y, pred, average=None, labels=list(range(n_classes)), zero_division=0)
    return {"accuracy": float(accuracy_score(y, pred)), "precision": float(p),
            "recall": float(r), "f1": float(f1), "auc": float(auc),
            "per_class": {"precision": pc_p.tolist(), "recall": pc_r.tolist(),
                          "f1": pc_f1.tolist(), "support": pc_sup.tolist()},
            "confusion_matrix": confusion_matrix(
                y, pred, labels=list(range(n_classes))).tolist(),
            "_predictions": pred, "_labels": y, "_probabilities": probs}


# =============================================================================
# Locating pool rows -- shared with NB07
# =============================================================================
def locate_pool_rows(X_pool_pca: np.ndarray, X_split_pca: np.ndarray,
                     tol: float = 1e-9) -> np.ndarray:
    """Index of each split row inside the 50k pool, by exact PCA match.

    The archive stores the splits in PCA space and the pool in both spaces, so
    this is how a raw-space feature is attached to a training row without the
    source CSVs. NB07 does the same thing for its raw-space perturbations; the
    assertion below is what makes it safe.
    """
    from scipy.spatial import cKDTree

    d, idx = cKDTree(np.asarray(X_pool_pca)).query(np.asarray(X_split_pca), k=1)
    if not (d < tol).all():
        raise AssertionError(
            f"{int((d >= tol).sum())} split rows could not be located in the "
            f"pool (max distance {d.max():.3e}); the archive and the split do "
            f"not come from the same preprocessing run.")
    return idx

# ==================================================================
# === Inlined from src/splits.py ===
# ==================================================================
"""
splits.py -- evaluation protocols and the enlarged test set.

A random train-test partition may allow correlated network flows originating
from the same capture sessions to appear in both training and testing subsets,
potentially leading to optimistic performance estimates. Session-level,
device-level and chronological splits provide stronger evidence regarding the
generalization ability of the framework in realistic deployment scenarios.

The pipeline of the paper is reproduced exactly -- constant-column removal,
|r| > 0.95 correlation filter, ANOVA top-25, 50,000-sample stratified pool,
StandardScaler, PCA-16 -- with ONE correction:

    every fitting step runs on the TRAINING PORTION of each protocol only.

That includes the ANOVA selector and the PCA, not merely the classifier. A
session-level split whose PCA was fitted over all sessions leaks through the
preprocessing and reproduces, in a subtler form, exactly the flaw that grouped
splits are meant to expose. `assert_fitted_on_train_only` (validation gate G5) refuses to continue if
any component saw more rows than the training split contains.

Protocols
---------
random   stratified, seeded -- the protocol of the paper, kept for comparison
session  grouped by source capture file; no capture contributes to two splits
device   grouped by device identity; INFEASIBLE unless the CSVs carry a device,
         MAC or IP column -- reported as infeasible with the reason, never
         silently approximated by something else
chrono   time-ordered within each capture: first 70% train, next 15% val, last
         15% test, so the model is always tested on later traffic
"""


from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np

# from config import (   # inlined
#     MIN_TEST_PER_CLASS,   # inlined
#     N_CLASSES,   # inlined
#     N_FEATURES,   # inlined
#     N_PCA,   # inlined
#     POOL_SIZE,   # inlined
#     TEST_SIZE_TARGET,   # inlined
#     TRAIN_FRAC,   # inlined
#     VAL_FRAC,   # inlined
# )   # inlined

# Candidate column names that would make a device-level split possible.
DEVICE_COLUMN_CANDIDATES = ("device", "device_id", "mac", "src_mac", "srcmac",
                            "eth.src", "src_ip", "srcip", "ip.src", "source_ip")


# =============================================================================
# Labels -- byte-for-byte the taxonomy of the paper
# =============================================================================
def parse_label(label_str) -> Tuple[int, int]:
    """Raw CICIoMT2024 label -> (binary, 8-class). Copied unchanged from the
    original `01_Data_Preparation`; changing it would change the task."""
    label = str(label_str).lower().strip()
    binary = 0 if "benign" in label else 1
    if "benign" in label:
        multiclass = 0
    elif "spoof" in label or "arp" in label:
        multiclass = 1
    elif any(k in label for k in ("recon", "scan", "vulscan", "os_detect",
                                  "portscan")):
        multiclass = 2
    elif "malformed" in label:
        multiclass = 3
    elif "mqtt" in label and "ddos" in label:
        multiclass = 5
    elif "mqtt" in label and "dos" in label:
        multiclass = 4
    elif "ddos" in label:
        multiclass = 7
    elif "dos" in label:
        multiclass = 6
    else:
        multiclass = 1
    return binary, multiclass


# =============================================================================
# Raw CSVs -> flows + provenance
# =============================================================================
def load_raw_flows(data_dir) -> Dict:
    """Read every CSV in `data_dir`, attaching capture-session provenance.

    `source_csv` is the capture-session proxy and `row_order` the chronological
    proxy: the archive preserved neither, which is why the split protocols
    cannot be built from it and need the CSVs back.
    """
    from pathlib import Path

    import pandas as pd

    paths = sorted(Path(data_dir).glob("*.csv"))
    if not paths:
        raise FileNotFoundError(
            f"no CSVs in {data_dir}. The CICIoMT2024 WiFi/MQTT partition is "
            f"required for the session/device/chronological protocols: "
            f"https://www.unb.ca/cic/datasets/iomt-dataset-2024.html")

    frames = []
    for p in paths:
        df = pd.read_csv(p, low_memory=False)
        df["__source_csv"] = p.name
        df["__row_order"] = np.arange(len(df))
        if "label" not in df.columns:
            # Several distributions encode the class in the file name only.
            df["label"] = p.stem
        frames.append(df)
    flows = pd.concat(frames, ignore_index=True)

    parsed = flows["label"].apply(parse_label)
    y_binary = np.array([t[0] for t in parsed], dtype=np.int64)
    y_multi = np.array([t[1] for t in parsed], dtype=np.int64)

    device_col = next((c for c in flows.columns
                       if c.lower() in DEVICE_COLUMN_CANDIDATES), None)

    return {"flows": flows, "y_binary": y_binary, "y_multiclass": y_multi,
            "source_csv": flows["__source_csv"].to_numpy(),
            "row_order": flows["__row_order"].to_numpy(),
            "device": (flows[device_col].to_numpy() if device_col else None),
            "device_column": device_col,
            "n_sources": int(flows["__source_csv"].nunique()),
            "paths": [str(p) for p in paths]}


def numeric_matrix(flows) -> Tuple[np.ndarray, List[str]]:
    """All numeric feature columns, NaN/inf-cleaned. No selection yet: feature
    selection is label-dependent and therefore belongs inside the fold."""
    import pandas as pd

    cols = [c for c in flows.columns
            if c != "label" and not str(c).startswith("__")]
    X = flows[cols].apply(pd.to_numeric, errors="coerce")
    X = X.fillna(0.0).replace([np.inf, -np.inf], 0.0)
    return X.to_numpy(dtype=np.float64), list(cols)


# =============================================================================
# Protocols
# =============================================================================
def _stratified_three_way(y: np.ndarray, seed: int,
                          frac=(TRAIN_FRAC, VAL_FRAC)) -> Tuple[np.ndarray, ...]:
    from sklearn.model_selection import train_test_split

    idx = np.arange(len(y))
    tr, rest = train_test_split(idx, train_size=frac[0], stratify=y,
                                random_state=seed)
    rel = frac[1] / (1.0 - frac[0])
    va, te = train_test_split(rest, train_size=rel, stratify=y[rest],
                              random_state=seed)
    return tr, va, te


def _grouped_three_way(groups: np.ndarray, seed: int) -> Tuple[np.ndarray, ...]:
    """Assign whole groups to train/val/test by cumulative size."""
    rng = np.random.default_rng(seed)
    uniq = np.unique(groups)
    rng.shuffle(uniq)
    sizes = {g: int((groups == g).sum()) for g in uniq}
    total = sum(sizes.values())

    buckets: Dict[str, List] = {"train": [], "val": [], "test": []}
    acc = 0
    for g in uniq:
        frac = acc / total
        key = ("train" if frac < TRAIN_FRAC
               else "val" if frac < TRAIN_FRAC + VAL_FRAC else "test")
        buckets[key].append(g)
        acc += sizes[g]

    # A protocol that leaves a split empty is not a protocol.
    for key in ("train", "val", "test"):
        if not buckets[key]:
            donor = max(buckets, key=lambda k: len(buckets[k]))
            if len(buckets[donor]) < 2:
                raise ValueError("too few groups for a grouped split")
            buckets[key].append(buckets[donor].pop())

    sel = lambda gs: np.flatnonzero(np.isin(groups, gs))
    return sel(buckets["train"]), sel(buckets["val"]), sel(buckets["test"])


def _chronological(source: np.ndarray, order: np.ndarray) -> Tuple[np.ndarray, ...]:
    """First 70% / next 15% / last 15% of every capture, in packet order.

    Splitting inside each capture rather than across captures keeps every class
    present in every split -- a global time cut would put whole attack families
    on one side, which measures class coverage rather than temporal
    generalisation.
    """
    tr, va, te = [], [], []
    for s in np.unique(source):
        rows = np.flatnonzero(source == s)
        rows = rows[np.argsort(order[rows])]
        n = len(rows)
        a, b = int(n * TRAIN_FRAC), int(n * (TRAIN_FRAC + VAL_FRAC))
        tr.append(rows[:a]); va.append(rows[a:b]); te.append(rows[b:])
    return (np.sort(np.concatenate(tr)), np.sort(np.concatenate(va)),
            np.sort(np.concatenate(te)))


def protocol_indices(raw: Dict, protocol: str, y: np.ndarray,
                     seed: int = 42) -> Dict:
    """Train/val/test indices for one protocol, plus a feasibility verdict.

    An infeasible protocol returns `{"feasible": False, "reason": ...}` instead
    of an approximation. Reporting "device-level split unavailable because the
    published CSVs carry no device identifier" is a result; silently
    substituting something else is not.
    """
    if protocol == "random":
        tr, va, te = _stratified_three_way(y, seed)
    elif protocol == "session":
        if raw["n_sources"] < 3:
            return {"feasible": False,
                    "reason": f"only {raw['n_sources']} capture file(s); a "
                              f"session-level split needs at least 3"}
        tr, va, te = _grouped_three_way(raw["source_csv"], seed)
    elif protocol == "device":
        if raw["device"] is None:
            return {"feasible": False,
                    "reason": "no device / MAC / IP column in the published "
                              "CICIoMT2024 WiFi-MQTT CSVs, so device identity "
                              "cannot be recovered; searched for "
                              + ", ".join(DEVICE_COLUMN_CANDIDATES)}
        if len(np.unique(raw["device"])) < 3:
            return {"feasible": False,
                    "reason": f"only {len(np.unique(raw['device']))} distinct "
                              f"devices in column '{raw['device_column']}'"}
        tr, va, te = _grouped_three_way(raw["device"], seed)
    elif protocol == "chrono":
        tr, va, te = _chronological(raw["source_csv"], raw["row_order"])
    else:
        raise ValueError(f"unknown protocol: {protocol}")

    assert_disjoint(tr, va, te)
    return {"feasible": True, "protocol": protocol, "seed": seed,
            "train_idx": tr, "val_idx": va, "test_idx": te,
            "n_train": int(len(tr)), "n_val": int(len(va)),
            "n_test": int(len(te)),
            "class_counts": {"train": np.bincount(y[tr], minlength=N_CLASSES).tolist(),
                             "val": np.bincount(y[va], minlength=N_CLASSES).tolist(),
                             "test": np.bincount(y[te], minlength=N_CLASSES).tolist()}}


def assert_disjoint(train_idx, val_idx, test_idx) -> None:
    """Gate G3, at the index level."""
    tr, va, te = set(map(int, train_idx)), set(map(int, val_idx)), set(map(int, test_idx))
    for a, b, na, nb in ((tr, te, "train", "test"), (va, te, "val", "test"),
                         (tr, va, "train", "val")):
        if a & b:
            raise AssertionError(f"LEAKAGE: {len(a & b)} indices shared between "
                                 f"{na} and {nb}")


# =============================================================================
# Preprocessing -- fitted INSIDE the fold (gate G5)
# =============================================================================
class FoldPreprocessor:
    """Constant filter -> correlation filter -> ANOVA top-k -> scaler -> PCA.

    The pipeline of the paper, with every stage fitted on the training rows of the
    current protocol and nothing else. `n_fit_rows` records how many rows each
    stage saw so gate G5 can check it against the training split.
    """

    def __init__(self, n_features: int = N_FEATURES, n_pca: int = N_PCA,
                 corr_threshold: float = 0.95, seed: int = 42):
        self.n_features, self.n_pca = n_features, n_pca
        self.corr_threshold, self.seed = corr_threshold, seed
        self.n_fit_rows: Optional[int] = None
        self.kept_names: List[str] = []

    def fit(self, X: np.ndarray, y: np.ndarray, feature_names: Sequence[str]):
        from sklearn.decomposition import PCA
        from sklearn.feature_selection import SelectKBest, f_classif
        from sklearn.preprocessing import StandardScaler

        self.n_fit_rows = int(len(X))
        names = list(feature_names)

        self.non_const_ = np.std(X, axis=0) > 0
        Xf = X[:, self.non_const_]
        names = [n for n, k in zip(names, self.non_const_) if k]

        corr = np.nan_to_num(np.corrcoef(Xf.T), nan=0.0)
        upper = np.triu(np.abs(corr), k=1)
        drop = {j for i in range(upper.shape[0])
                for j in range(i + 1, upper.shape[1])
                if upper[i, j] > self.corr_threshold}
        self.corr_keep_ = [i for i in range(Xf.shape[1]) if i not in drop]
        Xf = Xf[:, self.corr_keep_]
        names = [names[i] for i in self.corr_keep_]

        k = min(self.n_features, Xf.shape[1])
        self.selector_ = SelectKBest(f_classif, k=k).fit(Xf, y)
        Xf = self.selector_.transform(Xf)
        mask = self.selector_.get_support()
        self.kept_names = [n for n, m in zip(names, mask) if m]

        self.scaler_ = StandardScaler().fit(Xf)
        self.pca_ = PCA(n_components=min(self.n_pca, Xf.shape[1]),
                        random_state=self.seed).fit(self.scaler_.transform(Xf))
        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        Xf = X[:, self.non_const_][:, self.corr_keep_]
        return self.pca_.transform(self.scaler_.transform(
            self.selector_.transform(Xf)))

    def as_dict(self) -> Dict:
        return {"scaler": self.scaler_, "pca": self.pca_,
                "selector": self.selector_}


def assert_fitted_on_train_only(pre: FoldPreprocessor, train_idx) -> None:
    """Validation gate G5.

    The whole point of a session-level split is that nothing crosses the
    session boundary. If the PCA was fitted on more rows than the training
    split holds, it crossed, and every number downstream is optimistic in
    precisely the way grouped-split evaluation is designed to catch.
    """
    if pre.n_fit_rows is None:
        raise AssertionError("gate G5: preprocessor was never fitted")
    if pre.n_fit_rows != len(train_idx):
        raise AssertionError(
            f"gate G5 FAILED: preprocessing was fitted on {pre.n_fit_rows} rows "
            f"but the training split holds {len(train_idx)}. Fit the scaler, "
            f"the ANOVA selector and the PCA inside the fold.")


# =============================================================================
# Pool and enlarged test set
# =============================================================================
def draw_pool(y: np.ndarray, size: int = POOL_SIZE, seed: int = 42) -> np.ndarray:
    """The 50,000-sample stratified pool of the paper."""
    from sklearn.model_selection import train_test_split

    if len(y) <= size:
        return np.arange(len(y))
    idx, _ = train_test_split(np.arange(len(y)), train_size=size, stratify=y,
                              random_state=seed)
    return np.sort(idx)


def identify_archived_pool_rows(X_scaled_all: np.ndarray,
                                X_scaled_pool: np.ndarray,
                                tol: float = 1e-6) -> np.ndarray:
    """Boolean mask of rows that match the archived 50k pool.

    Used to keep the enlarged test set DISJOINT from the data the archived
    model was trained on. Matching is conservative: any row within `tol` of a
    pool row is excluded, so a duplicated flow is dropped rather than risked.
    """
    from scipy.spatial import cKDTree

    d, _ = cKDTree(np.asarray(X_scaled_pool)).query(np.asarray(X_scaled_all), k=1)
    return d < tol


def enlarged_test_indices(y: np.ndarray, excluded: np.ndarray,
                          min_per_class: int = MIN_TEST_PER_CLASS,
                          target: int = TEST_SIZE_TARGET,
                          seed: int = 42) -> Dict:
    """Draw a disjoint test set with at least `min_per_class` rows per class.

    The paper's per-class table reports F1 on n=4 and n=15. Those estimates
    are not defensible at any confidence level, and inference is cheap, so the
    fix is simply to test on more rows -- while recording the TRUE prevalence
    separately, because the enlarged set is deliberately not prevalence-faithful
    and its weighted metrics must be read with that in mind.
    """
    rng = np.random.default_rng(seed)
    avail = np.flatnonzero(~np.asarray(excluded))
    y_av = np.asarray(y)[avail]

    chosen, shortfalls = [], {}
    for c in range(N_CLASSES):
        pool = avail[y_av == c]
        take = min(len(pool), max(min_per_class, int(target / N_CLASSES)))
        if len(pool) < min_per_class:
            shortfalls[int(c)] = int(len(pool))
        if take:
            chosen.append(rng.choice(pool, size=take, replace=False))

    idx = np.sort(np.concatenate(chosen)) if chosen else np.array([], dtype=int)
    counts = np.bincount(np.asarray(y)[idx], minlength=N_CLASSES)
    true_prev = np.bincount(np.asarray(y), minlength=N_CLASSES) / len(y)
    return {"test_idx": idx, "counts": counts.tolist(),
            "true_prevalence": true_prev.tolist(),
            "min_per_class_target": int(min_per_class),
            "classes_below_target": shortfalls,
            "note": "class-balanced by construction; weighted metrics on this "
                    "set do NOT reflect operational prevalence -- report "
                    "per-class metrics here and prevalence-weighted metrics on "
                    "the original test split."}

print(summary())
OUT_DIR.mkdir(parents=True, exist_ok=True)
(ROOT / 'figures').mkdir(parents=True, exist_ok=True)

# --- PREFLIGHT ------------------------------------------------------
# Verify every prerequisite BEFORE any compute. On the cluster this
# turns a crash deep inside an hour-long job into an immediate message
# naming the notebook that has to run first.
preflight(OUT_DIR, ARCHIVE_DIR if True else None,
          'nb03_multiclass', packages=(),
          needs_data=False, data_dir=DATA_DIR)

=== Experiment configuration ===
  MODE          : FULL
  ROOT          : /home/<user>
  DATA_DIR      : /home/<user>/data
  OUT_DIR       : /home/<user>/outputs
  ARCHIVE_DIR   : /home/<user>/salvaged_g5k_artifacts
  SEEDS         : [42, 1, 7, 13, 2024]
  DATASET       : CICIoMT2024-WiFi-MQTT  (8 classes)
  POOL / PCA    : 50000 samples, 25 -> 16
  QCNN          : 8 qubits / 4 layers, scales=(1, 2, 4)
  BACKEND       : default.qubit / backprop
  EPOCHS        : binary 50, multiclass 120
  ENLARGED TEST : True (min 100/class)
[preflight] nb03_multiclass
[deps] archive OK: /home/<user>/salvaged_g5k_artifacts
[preflight] all checks passed



In [3]:
TASK = "multiclass"
data = load_prepared_data(str(ARCHIVE_DIR))
sfx = "b" if TASK == "binary" else "m"
X_tr, y_tr = subsample_for_mode(data[f"X_train_{sfx}"], data[f"y_train_{sfx}"])
X_va, y_va = data[f"X_val_{sfx}"],   data[f"y_val_{sfx}"]
X_te, y_te = data[f"X_test_{sfx}"],  data[f"y_test_{sfx}"]
NC = 2 if TASK == "binary" else N_CLASSES
NAMES = ["Benign", "Attack"] if TASK == "binary" else CLASS_NAMES
FIG = ROOT / "figures"; FIG.mkdir(exist_ok=True)
print(f"{TASK}: train {X_tr.shape}  val {X_va.shape}  test {X_te.shape}  "
      f"epochs={epochs_for(TASK)}")

multiclass: train (35000, 16)  val (7500, 16)  test (7500, 16)  epochs=120


In [ ]:
# --- multi-seed training --------------------------------------
histories, evals = {}, {}
for seed in SEEDS:
    if is_done(OUT_DIR, f"nb0{'2' if TASK=='binary' else '3'}_{TASK}", "qcnn",
               TASK, "random", seed):
        print(f"seed {seed}: already done, skipping")
        continue
    print(f"\n=== seed {seed} ===")
    set_global_seed(seed)
    model = build_model(TASK, "full")
    t0 = time.time()
    hist = train_model(model, make_loader(X_tr, y_tr),
                       make_loader(X_va, y_va, shuffle=False),
                       task=TASK, class_weights=compute_class_weights(y_tr, NC))
    r = evaluate(model, make_loader(X_te, y_te, shuffle=False), NC)
    histories[seed], evals[seed] = hist, r
    print(f"  test acc={r['accuracy']:.4f} f1={r['f1']:.4f} auc={r['auc']:.4f}")

    save_run(OUT_DIR, experiment=f"nb0{'2' if TASK=='binary' else '3'}_{TASK}",
             variant="qcnn", task=TASK, split="random", seed=seed,
             y_true=r['_labels'], y_pred=r['_predictions'], probs=r['_probabilities'],
             metrics={k: v for k, v in r.items()
                      if not k.startswith('_') and np.isscalar(v)},
             history=hist, model=model, params=model.param_breakdown(),
             timings={"train_wall_s": time.time() - t0},
             class_names=NAMES, extra={"notebook": "NB02/NB03", "topic": "multi-seed statistics"})


=== seed 42 ===
  epoch   1/120  train_acc=0.7015  val_f1=0.6330  val_auc=0.7015  (298.3s)
  epoch  10/120  train_acc=0.7584  val_f1=0.7492  val_auc=0.8019  (285.8s)
  epoch  20/120  train_acc=0.7673  val_f1=0.7447  val_auc=0.8233  (288.4s)
  epoch  30/120  train_acc=0.7728  val_f1=0.7615  val_auc=0.8224  (285.4s)
  epoch  40/120  train_acc=0.7792  val_f1=0.7630  val_auc=0.8309  (285.7s)
  epoch  50/120  train_acc=0.7818  val_f1=0.7554  val_auc=0.8281  (288.6s)
  early stop at epoch 50 (best val_f1=0.7653)
  test acc=0.7864 f1=0.7634 auc=0.8257
[persist] nb03_multiclass/qcnn__multiclass__random__seed42 -> ['curves', 'metrics', 'model', 'per_class', 'predictions']

=== seed 1 ===
  epoch   1/120  train_acc=0.7105  val_f1=0.6836  val_auc=0.7184  (287.6s)
  epoch  10/120  train_acc=0.7614  val_f1=0.7511  val_auc=0.8094  (284.3s)
  epoch  20/120  train_acc=0.7701  val_f1=0.7384  val_auc=0.8207  (289.0s)
  epoch  30/120  train_acc=0.7719  val_f1=0.7583  val_auc=0.8209  (286.3s)
  early sto

In [ ]:
# --- aggregate across seeds -----------------------------------------------
EXP = f"nb0{'2' if TASK=='binary' else '3'}_{TASK}"
runs = load_all_runs(OUT_DIR, EXP)
print(f"{len(runs)} seed runs found\n")
for metric in ("accuracy", "precision", "recall", "f1", "auc"):
    for row in summarise_runs(runs, metric, group_keys=("variant", "task")):
        print(f"  {metric:10s} {format_mean_std(row)}  "
              f"95% CI [{row['ci_low']:.4f}, {row['ci_high']:.4f}]  n={row['n']}")

In [ ]:
# --- learning curves with the seed band (all values saved) ----------------
if histories:
    hs = {"QCNN": list(histories.values())}
    c = training_curves(hs, path=str(FIG / f"{TASK}_learning_curves.png"),
                        metrics=("train_loss", "train_acc", "val_f1", "val_auc"))
    save_curve(OUT_DIR, EXP, "learning_curves", **c)
else:
    print("no new histories this run (all seeds already complete);"
          " curves are in outputs/*/runs/*/history.{csv,npz}")

In [ ]:
# --- quantitative curves from the seed-42 run -----------------------------
p = load_predictions(OUT_DIR, EXP, "qcnn", TASK, "random", SEEDS[0])
y_true, y_pred, probs = p["y_true"], p["y_pred"], p["probs"]

save_curve(OUT_DIR, EXP, "roc",
           **roc_curves(y_true, probs, NAMES, path=str(FIG / f"{TASK}_roc.png")))
save_curve(OUT_DIR, EXP, "pr",
           **pr_curves(y_true, probs, NAMES, path=str(FIG / f"{TASK}_pr.png")))
save_curve(OUT_DIR, EXP, "calibration",
           **calibration_curve(y_true, probs, path=str(FIG / f"{TASK}_calibration.png")))

rep = per_class_report(y_true, y_pred, NAMES)
save_curve(OUT_DIR, EXP, "per_class", **per_class_bars(rep,
           path=str(FIG / f"{TASK}_per_class.png")))
save_curve(OUT_DIR, EXP, "confusion", **confusion_matrix_plot(
    rep["confusion_matrix"], NAMES, path=str(FIG / f"{TASK}_confusion.png")))

In [ ]:
check_completeness(OUT_DIR, EXP)
print("\nEvery curve above is saved as CSV+NPZ; new figures need no re-training.")